<a href="https://colab.research.google.com/github/empirical-solutions/Spheres/blob/main/The_Mirror_of_Spheres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
The Mirror of Spheres -- 72-Gate Decan Traversal (Gemini Omni, all-JSON prompts).

84 chained segments: each of the 12 acts opens with a 10s act-transition --
the zodiac glyph materializing as a vast physical sigil while the elemental
shift occurs -- followed by its 6 gates (~12:00+ total at 10s gates).

WHAT'S DIFFERENT IN THIS BUILD
------------------------------
- ALL-JSON PROMPTS. Every request sent to Omni is one JSON document: a `meta`
  block (standing rules + state variables, substituted from the environment at
  run time) and a `segment` block (every variable for that gate: act, element,
  wheel tone, angel {name, choir, form}, demon {name, form}, environment,
  threshold, glyph, beats). No markdown, no mixed prose -- the model gets
  structure, the way it likes it.
- FULL AUDIO. Diegetic audio is rendered: the act's audio bed, the wheel-tone
  handoff (every segment opens continuing the previous tail tone and closes
  settled into its own), and the ONLY permitted spoken words -- the World
  Bridger speaking the two guardian names at the naming beat.
- STILL-FRAME CHAINING ONLY. Each segment takes a still of the TRUE last frame
  of the previous segment and patches it in as the first frame. No video
  references (no prev-segment attach, no hero clip) -- stills only.
- COLD OPEN. An optional pre-step generates the loop-point frame first: pure
  black void with him walking ("a black screen with him on it"). Its last
  frame seeds segment 0, so the film opens deliberately and gate 72's tail
  closes back into it -- the loop plays all day with no visible seam.
- SIMPLIFIED BEHAVIOR. The prompts never mention his face at all. At each
  gate: he raises his staff and nods toward the angel as he speaks its name,
  then nods toward the demon as he speaks its name. That's it.
- DRAFT/PROD TOGGLE. DRAFT_MODE=1 -> 360p into separate dirs at roughly a
  third of the cost. Unset -> full 1080p production.

    pip install -U "google-genai>=2.10.0" imageio-ffmpeg
    export GEMINI_API_KEY="your-key"      # from https://aistudio.google.com/
    python produce_video_mirror_json.py

Continuity: F_k,start = F_{k-1},end -- the previous segment's true last frame
(extracted with a bundled ffmpeg -- the exact final decoded frame, not an
estimate) is declared as the first-frame source of the next segment.

Gate roster: all 72 pairs match Paul's canonical blueprint verbatim.
Guardian iconography: 144 expanded historically-grounded descriptions
(ANGEL_FORMS / DEMON_FORMS, supplied by Paul 2026-10-08) -- every guardian
towering, elementally distinct, bearing its seal. Angel choirs derived from
gate number (1-8 Seraphim ... 65-72 Angels).

Quota handling: on rate-limit / quota errors (or a rejected key) the script
pauses and prompts for another Google AI Studio key (hidden input). Type
'skip' to skip the segment, 'quit' (or Ctrl+C) to stop.

NOTE: guardian names (Bael, Asmoday, Belial, Astaroth, ...) can trip the
content filter. The tiered fallback sheds the first-frame still on refusal
(chained -> fresh) and the run continues; re-run heals gaps via gap-fill.
"""

import base64
import getpass
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

try:
    from google import genai
except ImportError:  # --dry-run works without the SDK installed
    genai = None

# ------------------------------------------------------------------ config

SCHEMA_VERSION = "mirror-json/1.0"

# Paul's finding (2026-10-08): gemini-omni-flash-preview (the legacy preview
# path) encodes the same-quality renders at ~2-4MB vs ~9MB for 1.1-flash --
# same per-second price, far lighter segments and stitches. Default is the
# preview path he actually renders on; override with OMNI_MODEL=....
MODEL_ID = os.environ.get("OMNI_MODEL", "gemini-omni-flash-preview")

API_KEY = os.environ.get("GEMINI_API_KEY")
# --dry-run prints every segment's full JSON prompt without touching the API.
DRY_RUN = "--dry-run" in sys.argv
if not API_KEY and not DRY_RUN:
    raise SystemExit("Set GEMINI_API_KEY first: export GEMINI_API_KEY='your-key'")

client = None
API_KEY_POOL = []


def set_active_key(api_key, source=""):
    global client
    if genai is None:
        raise SystemExit("The google-genai SDK is not installed -- run via "
                         "run_mirror_json.sh or: pip install \"google-genai>=2.10.0\"")
    api_key = api_key.strip()
    os.environ["GEMINI_API_KEY"] = api_key
    if api_key not in API_KEY_POOL:
        API_KEY_POOL.append(api_key)
    client = genai.Client(api_key=api_key)
    print(f"    active API key set ({source or 'startup'}).")


if not DRY_RUN:
    set_active_key(API_KEY, source="environment")

# Draft <-> production toggle: DRAFT_MODE=1 renders 360p into isolated dirs.
DRAFT_MODE = os.environ.get("DRAFT_MODE", "").strip().lower() in ("1", "true", "yes")
OUT_DIR = "segments_mirror_json_draft" if DRAFT_MODE else "segments_mirror_json"
FRAMES_DIR = "frames_mirror_json_draft" if DRAFT_MODE else "frames_mirror_json"
RESOLUTION = "360p" if DRAFT_MODE else "1080p"

# Gate render length in seconds.
try:
    GATE_SECONDS = int(float(os.environ.get("GATE_SECONDS", "10")))
except ValueError:
    raise SystemExit(f"GATE_SECONDS must be a number, got {os.environ.get('GATE_SECONDS')!r}")
if GATE_SECONDS < 8:
    raise SystemExit(f"GATE_SECONDS must be >= 8, got {GATE_SECONDS}")

# Join style: 0 = direct hard cuts (the chained renders land so closely the
# cuts read as seamless). XFADE_S=0.5 for short dissolves, 1 for long ones.
XFADE_S = float(os.environ.get("XFADE_S", "0"))

# Cap a run to the first N segments (e.g. MAX_SEGMENTS=4 = cold open check +
# first gates). Unset or 0 = the full run.
try:
    MAX_SEGMENTS = int(os.environ.get("MAX_SEGMENTS", "0") or 0)
except ValueError:
    raise SystemExit("MAX_SEGMENTS must be an integer")

# Cold-open pre-step: generate the black-void loop-point frame first and seed
# segment 0 from its last frame. COLD_OPEN=0 skips it (segment 0 goes fresh).
COLD_OPEN = os.environ.get("COLD_OPEN", "1").strip().lower() not in ("0", "false", "no")

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # embedded / exec contexts without __file__
    _HERE = os.getcwd()

# Canonical behind-view still of the Bridger's look. Attached to every
# generation as an IMAGE (appearance only). This is a still, not a video --
# video references are out of this build. BRIDGER_REF="" disables it.
_BRIDGER_REF_PATH = os.environ.get("BRIDGER_REF", os.path.join(_HERE, "bridger_ref.jpg"))
BRIDGER_REF_B64 = None
if _BRIDGER_REF_PATH:
    try:
        with open(_BRIDGER_REF_PATH, "rb") as _rf:
            BRIDGER_REF_B64 = base64.b64encode(_rf.read()).decode("ascii")
        print(f"    bridger reference still: {_BRIDGER_REF_PATH}")
    except OSError:
        print(f"    bridger reference not found ({_BRIDGER_REF_PATH}) -- continuing without it")

RESUME_EXISTING = True
GENERATE_TIMEOUT_S = 1800
POLL_S = 15
VERBOSE = os.environ.get("VERBOSE", "1").strip() not in ("0", "false", "no")


def log(msg):
    """Timestamped log line (flush immediately so hangs are visible)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vlog(msg):
    """Verbose-only log line."""
    if VERBOSE:
        log(msg)


# ------------------------------------------------------------------ run log
# Developer-level run statistics: one JSON object per line, per run, under
# logs_mirror_json/. Captures what happened (timings, tiers, retries, sizes)
# -- never prompt text.
RUN_LOG_DIR = os.path.join(_HERE, "logs_mirror_json")
RUN_LOG_PATH = None


def rlog_init(config):
    """Open a new per-run JSONL log; writes the run_start event."""
    global RUN_LOG_PATH
    os.makedirs(RUN_LOG_DIR, exist_ok=True)
    RUN_LOG_PATH = os.path.join(RUN_LOG_DIR,
                                f"run_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
    rlog({"event": "run_start", "ts": time.time(), "config": config})
    print(f"    run log: {RUN_LOG_PATH}")


def rlog(event):
    """Append one event dict to the current run log (best-effort)."""
    if not RUN_LOG_PATH:
        return
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")
    except OSError:
        pass


def find_ffmpeg():
    try:
        from imageio_ffmpeg import get_ffmpeg_exe
        return get_ffmpeg_exe()
    except ImportError:
        return shutil.which("ffmpeg")


FFMPEG = find_ffmpeg()
if not FFMPEG:
    print("WARNING: no ffmpeg found (pip install imageio-ffmpeg). "
          "Segments will generate standalone without last-frame chaining.")

# ------------------------------------------------------------------ data tables
# (Paul's canonical data -- act/gate roster, environments, tones, iconography.)

# act: (name, element, gates lo-hi, audio profile)
ACTS = {
    1:  ("ARIES -- The Fire of Ignition",        "Fire",  (1, 6),   "heavy pulsating 40Hz sub-bass rhythm"),
    2:  ("TAURUS -- The Earth of Materiality",  "Earth", (7, 12),  "deep sustained earth drone with low stone-grinding friction"),
    3:  ("GEMINI -- The Air of Duality",        "Air",   (13, 18), "high-frequency rapidly panning binaural whispers"),
    4:  ("CANCER -- The Water of the Abyss",    "Water", (19, 24), "heavy filtered aqua reverb with sub-aquatic hydrostatic swells"),
    5:  ("LEO -- The Fire of Sovereignty",      "Fire",  (25, 30), "majestic brass swells underscored by visceral lion-roar sub-bass"),
    6:  ("VIRGO -- The Earth of Perfection",    "Earth", (31, 36), "ultra-clean exact sine tones oscillating in pure mathematical intervals"),
    7:  ("LIBRA -- The Air of Equilibrium",     "Air",   (37, 42), "wide-spectrum Perfect Fifth synth harmonics at 1.5 ratio"),
    8:  ("SCORPIO -- The Water of Transformation", "Water", (43, 48), "visceral slow 40Hz sub-bass heartbeat pulse"),
    9:  ("SAGITTARIUS -- The Fire of the Seeker", "Fire", (49, 54), "fast-tempo percussive drive with galloping rhythmic transients"),
    10: ("CAPRICORN -- The Earth of Architecture", "Earth", (55, 60), "massive acoustic stone cavern resonance and subterranean sub-drones"),
    11: ("AQUARIUS -- The Air of the Cosmos",   "Air",   (61, 66), "glitching bitcrushed modular synths and digital circuit feedback"),
    12: ("PISCES -- The Water of Dissolution",  "Water", (67, 72), "lush wordless 432Hz synth pad wash slowly pitching down into 40Hz sub-bass"),
}

# (gate, angel, demon, environment beat)
GATES = [
    # ACT 1 -- ARIES
    (1,  "Vehuiah",   "Bael",      "a suspended perfect circle of floating highly polished obsidian and volcanic glass shards erupting with cymatic fire fractals"),
    (2,  "Jeliel",    "Agares",    "twin ignition arcs spiral off the obsidian ring as the second fire node kindles"),
    (3,  "Sitael",    "Vassago",   "cymatic flames braid into triple-helix columns ringing the Bridger"),
    (4,  "Elemiah",   "Gamigin",   "volcanic glass shards ignite in sequence around the circle like a struck bell of fire"),
    (5,  "Mahasiah",  "Marbas",    "a roaring wall of flame parts down the middle as the guardians take their stations"),
    (6,  "Lelahel",   "Valefor",   "the full obsidian circle blazes white-gold as the sixth node locks the Aries fire circuit"),
    # ACT 2 -- TAURUS
    (7,  "Achaiah",   "Amon",      "colossal granite monoliths covered in glowing moss sigils forming an earthly circuit"),
    (8,  "Cahetel",   "Barbatos",  "glowing subterranean moss and deep stone fissures breathing faint green light"),
    (9,  "Haziel",    "Paimon",    "alabaster light channels cutting through bedrock echo vaults"),
    (10, "Aladiah",   "Buer",      "emerald ley-lines projected across slowly rotating basalt gears"),
    (11, "Lauviah",   "Gusion",    "sapphire starlight woven down into iron-ore foundations"),
    (12, "Hahaiah",   "Sitri",     "a marble sanctuary bound to smoldering red obsidian"),
    # ACT 3 -- GEMINI
    (13, "Iezalel",   "Beleth",    "dual mirrored prisms floating on aggressive high-altitude air currents"),
    (14, "Mebahel",   "Leraje",    "crystalline refractions sliced by whistling wind currents"),
    (15, "Hariel",    "Eligos",    "diamond-bright starlight colliding with sweeping iron dust vortices"),
    (16, "Hakamiah",  "Zepar",     "symmetrical light paths intersecting crimson atmospheric distortion"),
    (17, "Lauviah",   "Botis",     "soothing aura flares bound to razor-sharp shear winds"),
    (18, "Caliel",    "Bathin",    "geometric beam alignment meeting shadow-shrouded downdrafts"),
    # ACT 4 -- CANCER
    (19, "Leuviah",   "Sallos",    "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton forming moving seals"),
    (20, "Pahaliah",  "Purson",    "golden underwater arches veiled in drifting motes of light"),
    (21, "Nelchael",  "Marax",     "fluid geometry cutting through iron-infused silt clouds"),
    (22, "Yeiayel",   "Ipos",      "shimmering starlight piercing dark ocean undertows"),
    (23, "Melahel",   "Aim",       "pure aqua-energy distilling against undersea thermal vents"),
    (24, "Haheuiah",  "Naberius",  "a protective bioluminescent sphere anchored to ocean-floor vaults"),
    # ACT 5 -- LEO
    (25, "Nith-Haiah","Glasya-Labolas", "a blinding solar corona filled with spinning gold geometric projections"),
    (26, "Haaiah",    "Bune",      "gold architectural strictures anchoring into smoldering embers"),
    (27, "Yeratel",   "Ronove",    "corona flares locking into magnetic retention rings"),
    (28, "Seheiah",   "Berith",    "cooling golden aura balancing against molten royal flame"),
    (29, "Reiyel",    "Astaroth",  "pure solar sovereign light intersecting eclipsing copper shadow"),
    (30, "Omael",     "Forneus",   "vitalizing solar warmth grounding against cold solar vacuum"),
    # ACT 6 -- VIRGO
    (31, "Lecabel",   "Foras",     "hyper-precise cold sacred geometry and infinite anatomical spine mappings"),
    (32, "Vasariah",  "Asmoday",   "geometric symmetry carving along vertebrae entwined with chthonic horns"),
    (33, "Yehuiah",   "Gaap",      "starlight vertebrae seals binding to subterranean nerve roots"),
    (34, "Lehahiah",  "Furfur",    "cold spine plates aligning with bio-electric energy"),
    (35, "Chavakiah", "Marchosias","gold analytical geometry stabilizing inside skeletal frameworks"),
    (36, "Menadel",   "Stolas",    "a beam of mathematical order anchoring into starlight observatory roots"),
    # ACT 7 -- LIBRA
    (37, "Aniel",     "Phenex",    "unfolding scales of light and absolute mirrored symmetry on the 180-degree halfway axis"),
    (38, "Haamiah",   "Halphas",   "a symmetrical light sanctuary anchored into iron battlements"),
    (39, "Rehael",    "Malphas",   "silver threads of equilibrium meeting obsidian tower geometry"),
    (40, "Ieiazel",   "Raum",      "balanced air beams held inside absorbing shadow vortices"),
    (41, "Hahahel",   "Focalor",   "solar-winged scales balancing against heavy atmospheric pressure"),
    (42, "Mikael",    "Vepar",     "a central axis of pristine light mirroring deep sea-mist"),
    # ACT 8 -- SCORPIO
    (43, "Veuliah",   "Sabnock",   "massive wet obsidian vaults pulsing with a bioluminescent serpentine nervous system"),
    (44, "Yelahiah",  "Shax",      "bioluminescent aqua filaments pulsing through biological stone walls"),
    (45, "Sealiah",   "Vine",      "invigorating aqua currents anchoring into serpentine root systems"),
    (46, "Ariel",     "Bifrons",   "starlight streams balancing against tomb-stone shadows"),
    (47, "Asaliah",   "Vual",      "turquoise clarity illuminating the deep abyssal undertow"),
    (48, "Mihael",    "Haagenti",  "the 28th spoke rim at 240 degrees, wet obsidian giving way to void"),
    # ACT 9 -- SAGITTARIUS
    (49, "Vehuel",    "Crocell",   "an arrow of pure light shooting endlessly through an expanding cosmic Akashic library of flying parchment codices"),
    (50, "Daniel",    "Furcas",    "starlight illuminating ancient codices bound by iron quill marks"),
    (51, "Hahasiah",  "Balam",     "universal truths revealing in golden light through veils of mystery"),
    (52, "Imamiah",   "Alloces",   "forward trajectory of light accelerating through armor-plated momentum"),
    (53, "Nanael",    "Camio",     "celestial wisdom resonating across endless library halls"),
    (54, "Nithael",   "Murmur",    "a starlight throne bound to deep trumpet resonance"),
    # ACT 10 -- CAPRICORN
    (55, "Mebahiah",  "Orobas",    "an ancient mountain archive of deeply carved glowing seals"),
    (56, "Poiel",     "Gremory",   "gold light filling mountain vaults counterbalanced by velvet shadow chambers"),
    (57, "Nemamiah",  "Ose",       "crystalline spires raising toward mountain peaks"),
    (58, "Yeialel",   "Amy",       "iron beams aligning across stone archives bound by subterranean fire"),
    (59, "Harahel",   "Orias",     "cosmic knowledge vaults opening beside star-charting pillars"),
    (60, "Mitzrael",  "Vapula",    "architectural mountain integrity binding to mechanical craftsmanship"),
    # ACT 11 -- AQUARIUS
    (61, "Umabel",    "Zagan",     "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations"),
    (62, "Iah-Hel",   "Valac",     "high-speed data sanctuaries anchored into dragon-circuit foundations"),
    (63, "Anauel",    "Andras",    "collective thought-currents coordinating through static voltage spikes"),
    (64, "Mehiel",    "Flauros",   "protective light-scripts writing across sky firewalls"),
    (65, "Damabiah",  "Andrealphus","crystal-clear data streams transforming through prism matrices"),
    (66, "Manakel",   "Kimaris",   "a neural storm calming into steady logic above server vaults"),
    # ACT 12 -- PISCES
    (67, "Eyael",     "Amdusias",  "an infinite borderless ocean of consciousness dissolving into starlight mist"),
    (68, "Habuhiah",  "Belial",    "primordial waters of rebirth nurturing above seabed bedrock"),
    (69, "Rochel",    "Decarabia", "cosmic memories restoring in glowing tides and star-shadows"),
    (70, "Jabamiah",  "Seere",     "ocean energy transforming into spirit propelled by swift currents"),
    (71, "Haiaiel",   "Dantalion", "divine clarity arming the soul inside infinite mirrors of consciousness"),
    (72, "Mumiah",    "Andromalius","the final node where the borderless ocean ignites back into volcanic glass and cymatic fire fractals"),
]


def act_of(gate):
    for num, (name, el, lo_hi, audio) in ACTS.items():
        lo, hi = lo_hi
        if lo <= gate <= hi:
            return num, name, el, audio
    raise ValueError(gate)


def choir_of(gate):
    """Angel choir derived from gate number (gates 1-8 Seraphim ... 65-72 Angels)."""
    return {
        1: "Seraphim", 2: "Cherubim", 3: "Thrones", 4: "Dominations",
        5: "Powers", 6: "Virtues", 7: "Principalities", 8: "Archangels",
        9: "Angels",
    }[(gate - 1) // 8 + 1]

# ------------------------------------------------------------------ guardian iconography
# Historically-grounded forms: demons per Ars Goetia, angels per their 9 choirs.
# Every entity is towering, visually distinct, elemental, and dynamically raises its talisman.

ANGEL_FORMS = {
    # Choir 1: Seraphim (Fiery Divine Will & Primordial Light)
    "Vehuiah": "a towering, six-winged Seraph of blinding volcanic flame, igniting the primal void and thrusting high a blazing talisman of divine initiation",
    "Jeliel": "a towering, six-winged Seraph woven from soaring thermal firestorms, harmonizing celestial tides and raising aloft a radiant talisman of eternal union",
    "Sitael": "a towering, six-winged Seraph of piercing photon brilliance, laying cosmic foundation stones and brandishing high a burning talisman of architectural construction",
    "Elemiah": "a towering, six-winged Seraph expanding a golden corona of solar wrath, commanding the deep abyss and hoisting a blazing talisman of divine power",
    "Mahasiah": "a towering, six-winged Seraph radiating forge-hot alchemical heat, purifying cosmic discord and thrusting upward a fiery talisman of spiritual rectification",
    "Lelahel": "a towering, six-winged Seraph of blinding starlight and restorative solar flame, casting away darkness and raising high a luminous talisman of miraculous healing",
    "Achaiah": "a towering, six-winged Seraph of subterranean earth-fire, rooted in crystalline bedrock and brandishing aloft a heavy granite talisman of unyielding patience",
    "Cahetel": "a towering, six-winged Seraph wreathed in fertile golden auroras, commanding flourishing cosmic growth and hoisting high a jade-carved talisman of celestial harvest",

    # Choir 2: Cherubim (Wisdom, Four-Faced Tetramorphs & Whirling Wheels)
    "Haziel": "a towering, four-faced Cherub veined with molten gold and lapis lazuli, spinning crystalline wheels and raising aloft an earthen talisman of divine mercy",
    "Aladiah": "a towering, four-faced Cherub carved from adamantine granite, grinding crystalline wheels of light and thrusting high a diamond-bright talisman of restorative grace",
    "Lauviah": "a towering, four-faced Cherub crowned in sapphire starlight, turning heavy planetary wheels and brandishing aloft a meteoric iron talisman of prophetic revelation",
    "Hahaiah": "a towering, four-faced Cherub of translucent alabaster marble, shielding inner sanctuary fires and raising high a pearl-inlaid talisman of mystical refuge",
    "Iezalel": "a towering, four-faced Cherub of prismatic upper-atmosphere zephyrs, whirling gale-force crystalline wheels and hoisting aloft a radiant talisman of sacred fidelity",
    "Mebahel": "a towering, four-faced Cherub of blinding atmospheric lightning, severing untruth with spinning wheels and thrusting upward a crystalline talisman of absolute justice",
    "Hariel": "a towering, four-faced Cherub reflecting diamond-white astral dawns, chiming with infinite choir wheels and raising high an argent talisman of moral purification",
    "Hakamiah": "a towering, four-faced Cherub of roaring martial thunder, aligning celestial shields with turning wheels and brandishing aloft a bronze talisman of chivalric loyalty",

    # Choir 3: Thrones / Ophanim (Cosmic Foundations & Eye-Studded Beryl Wheels)
    "Lauviah (Throne)": "a towering Ophanim of revolving interlocking beryl wheels, burning with thousands of sleepless eyes and raising high an astral talisman of cosmic memory",
    "Caliel": "a towering Ophanim of whirling celestial chrysolite wheels, piercing atmospheric illusions with eye-studded rims and thrusting upward an electrum talisman of absolute truth",
    "Leuviah": "a towering Ophanim submerged in bioluminescent abyssal currents, turning eye-rimmed oceanic wheels and raising aloft a glowing coral talisman of expansive intellect",
    "Pahaliah": "a towering Ophanim burning with theological white flame beneath crushing deep-sea pressure, spinning sacred wheels and hoisting a golden talisman of fervent redemption",
    "Nelchael": "a towering Ophanim tracing astronomical geometry across the heavens, dripping liquid starlight from wheel-rims and brandishing aloft a brass talisman of higher learning",
    "Yeiayel": "a towering Ophanim navigating tempestuous celestial waters, guiding lost travelers with revolving eye-wheels and thrusting high a pearl-encrusted talisman of safe passage",
    "Melahel": "a towering Ophanim distilling pure restorative springwaters, blooming with medicinal flora along turning rims and raising high an emerald talisman of healing waters",
    "Haheuiah": "a towering Ophanim projecting an impenetrable oceanic dome, locking abyssal sanctuary wheels and brandishing aloft a heavy obsidian talisman of exile's protection",

    # Choir 4: Dominations (Sovereign Authority, Solar Crowns & Imperial Scepters)
    "Nith-Haiah": "a towering Domination draped in sovereign solar plasma, commanding secret occult forces with a glowing scepter and raising high a golden talisman of hidden wisdom",
    "Haaiah": "a towering Domination crowned in blinding coronal flares, orchestrating universal statecraft with an imperial staff and thrusting upward an electrum talisman of diplomacy",
    "Yeratel": "a towering Domination of victorious solar radiance, scattering encroaching shadows with sovereign authority and raising aloft a brilliant sunstone talisman of divine liberation",
    "Seheiah": "a towering Domination clad in golden dragon-scale armor, parrying ruinous cataclysms with an imperial scepter and hoisting high a mirror-polished talisman of longevity",
    "Reiyel": "a towering Domination blazing with sacred solar fury, shattering profane idols with royal decree and brandishing aloft a consecrated talisman of spiritual liberty",
    "Omael": "a towering Domination pulsing with generative solar warmth, multiplying the seeds of living creation and raising high a verdant chrysolite talisman of biological fertility",
    "Lecabel": "a towering Domination of laser-sharp geometric precision, dictating celestial vegetation cycles with an iron rod and thrusting upward an emerald talisman of exactitude",
    "Vasariah": "a towering Domination of stern imperial symmetry, weighing royal clemency with an ivory scepter and brandishing aloft a heavy marble talisman of righteous justice",

    # Choir 5: Powers (Martial Enforcers of Law, Chains & Restraining Aegis)
    "Yehuiah": "a towering Power clad in dark starlight plate, anchoring vertebrae of celestial law with glowing chains and raising high an iron talisman of cosmic subordination",
    "Lehahiah": "a towering Power encased in frost-gleaming adamantine armor, subduing rebellious spirits with celestial collars and thrusting upward a platinum talisman of divine obedience",
    "Chavakiah": "a towering Power armored in radiant solar gold, mending shattered ancestral pacts with bonds of light and hoisting aloft an engraved talisman of familial reconciliation",
    "Menadel": "a towering Power in heavy blackened-iron harness, liberating exhausted souls from oppressive toil and brandishing high an unyielding talisman of sanctified labor",
    "Aniel": "a towering Power balancing scales of kinetic celestial wind, shattering karmic chains of tyranny and raising aloft an argent talisman of broken cycles",
    "Haamiah": "a towering Power standing vigilant in ritual-scribed aegis armor, warding celestial sanctuaries with blazing runes and thrusting high a sacred talisman of spiritual protection",
    "Rehael": "a towering Power humming with harmonic frequencies of health, binding corrosive plagues with golden cords and hoisting aloft an amber talisman of filial healing",
    "Ieiazel": "a towering Power releasing concussive shockwaves of purified air, snapping dungeon iron with celestial chains and brandishing high a silver talisman of divine rescue",

    # Choir 6: Virtues (Miracle-Workers, Star Crowns & Elemental Catalysts)
    "Hahahel": "a towering Virtue crowned in blazing stellar constellations, channeling selfless apostolic sacrifice through an altar of light and raising high a consecrated talisman of holy priesthood",
    "Mikael": "a towering Virtue holding the central axis of sovereign harmony, steadying kingdoms amid shifting astral winds and thrusting upward a solar talisman of political order",
    "Veuliah": "a towering Virtue radiating turquoise subterranean luminescence, toppling tyrannical dynasties from the deep earth and raising aloft an orichalcum talisman of prosperous liberation",
    "Yelahiah": "a towering Virtue wreathed in bioluminescent martial wrath, guarding righteous warriors across dark currents and brandishing high a Damascus-steel talisman of karmic victory",
    "Sealiah": "a towering Virtue pulsing like a colossal cosmic heart, reigniting stagnant terrestrial currents with surge-force and hoisting aloft a verdant talisman of restored motion",
    "Ariel": "a towering Virtue parting abyssal ocean trenches, unearthing forgotten treasures of natural wisdom and thrusting high an iridescent mother-of-pearl talisman of secret revelation",
    "Asaliah": "a towering Virtue illuminating the midnight cosmos with contemplative brilliance, praising cosmic design and raising aloft a celestial prism talisman of divine cognition",
    "Mihael": "a towering Virtue radiating warm marine bioluminescence, intertwining divided souls into enduring unity and brandishing high a rose-gold talisman of conjugal harmony",

    # Choir 7: Principalities (Guardians of Nations, Celestial Architects & Scepters)
    "Vehuel": "a towering Principality crowned in galactic star-fire, releasing searing arrows of truth into the void and raising high a scintillating diamond talisman of divine exaltation",
    "Daniel": "a towering Principality crowned in shimmering rhetoric-flame, unfurling radiant parchment scrolls of cosmic law and thrusting upward an inscribed sapphire talisman of eloquent justice",
    "Hahasiah": "a towering Principality crowned in Hermetic alchemical fire, distilling universal panaceas from celestial vessels and raising aloft a smoldering cinnabar talisman of esoteric mysteries",
    "Imamiah": "a towering Principality crowned in incandescent plasma, trampling demonic pride beneath luminous boots and brandishing high a molten bronze talisman of spiritual expiation",
    "Nanael": "a towering Principality crowned in sacred philosophical radiance, contemplating planetary harmonics with celestial compasses and hoisting high an azure talisman of deep contemplation",
    "Nithael": "a towering Principality enthroned upon a pillar of starry marble, shielding emperors and rightful bloodlines while raising aloft an imperial gold talisman of legitimate succession",
    "Mebahiah": "a towering Principality crowned in carved mountain granite, anchoring righteous deeds into terrestrial foundations and thrusting high a lapis-inlaid talisman of lucid morality",
    "Poiel": "a towering Principality crowned in showering auric gold, opening infinite vaults of celestial abundance and brandishing high an opulent jewel-set talisman of boundless providence",

    # Choir 8: Archangels (Cosmic Heralds, Sacred Geometry & Divine Wisdom)
    "Nemamiah": "a towering Archangel forging crystalline spires of war, directing righteous commanders with a star-metal staff and raising high an adamantine talisman of righteous causes",
    "Yeialel": "a towering Archangel of deep resonant basalt, banishing sorrowful afflictions with an unbending rod and thrusting upward a cold meteoric iron talisman of endurance",
    "Harahel": "a towering Archangel unlocking limitless celestial archives, illuminating sacred texts with an ivory staff and hoisting high an inscribed golden talisman of intellectual wealth",
    "Mitzrael": "a towering Archangel mending the shattered geometry of the spirit, driving away internal madness and raising aloft a polished beryl talisman of psychological healing",
    "Umabel": "a towering Archangel weaving lines of astronomical affinity across the night sky, fostering enduring bonds and brandishing high an electrum talisman of sacred friendship",
    "Iah-Hel": "a towering Archangel enveloped in serene cloistered starlight, radiating transcendent wisdom through absolute silence and thrusting upward a crystal talisman of divine illumination",
    "Anauel": "a towering Archangel safeguarding terrestrial voyages and rightful commerce, shielding trade roads with a copper staff and raising high a coin-stamped talisman of material protection",
    "Mehiel": "a towering Archangel inking verses of celestial lightning into the clouds, inspiring grand orators and brandishing aloft a flaming quill-carved talisman of poetic inspiration",

    # Choir 9: Angels (Terrestrial Guardians, Living Waters & Rebirth)
    "Damabiah": "a towering Angel of gushing living headwaters, bestowing boundless oceanic wisdom upon searching souls and raising high a shimmering aquamarine talisman of the eternal spring",
    "Manakel": "a towering Angel diving through abyssal marine depths, interpreting prophetic dreams from dark waters and thrusting upward a polished pearl talisman of good and evil",
    "Eyael": "a towering Angel projecting incandescent mists of solar ascension, transmuting mundane sorrow into celestial joy and hoisting aloft an alchemical gold talisman of sublime transformation",
    "Habuhiah": "a towering Angel breathing fertile dew across barren orchards, healing blighted crops with radiant palms and raising high an emerald talisman of agricultural healing",
    "Rochel": "a towering Angel scouring the cosmic tides for lost treasures, returning stolen heritages to rightful heirs and brandishing aloft an argent talisman of complete restitution",
    "Jabamiah": "a towering Angel churning primordial oceans into celestial essence, resurrecting corrupted matter into perfection and thrusting upward a glowing chrysolite talisman of universal regeneration",
    "Haiaiel": "a towering Angel girded in blinding celestial armor, arming the pure of heart with flaming blades and raising high a luminous steel talisman of divine warfare and peace",
    "Mumiah": "a towering Angel of final oceanic dissolution, weaving twilight endings into triumphant rebirth and hoisting aloft an alpha-and-omega inscribed talisman of cosmic transformation",
}

DEMON_FORMS = {
    # 1 - 10
    "Bael": "a towering triple-headed king with heads of toad, cat, and crowned man on spider legs, rasping hoarsely and raising high a heavy iron talisman of invisibility",
    "Agares": "a towering, ancient nobleman riding an earth-shaking crocodile with a goshawk upon his fist, halting retreating armies and thrusting upward a smoldering copper talisman of language",
    "Vassago": "a towering infernal prince of radiant angelic countenance with burning sapphire eyes, declaring things past and future while raising aloft a crystalline talisman of prophecy",
    "Gamigin": "a towering spectral horse that shifts into an ashen phantom with a thunderous voice, summoning deceased souls and hoisting high a black-iron talisman of the dead",
    "Marbas": "a towering mechanical-maned lion with roaring furnace-fire eyes, answering truthfully of hidden secrets and brandishing high an iron-wrought talisman of mechanical craft and healing",
    "Valefor": "a towering prowling lion with the head of an ass-eared brigand, slipping through stealing shadows and raising high a tarnished silver talisman of thievery",
    "Amon": "a towering wolf with a serpent's tail vomiting dark obsidian flames, reconciling broken friendships and brandishing aloft a dense lead talisman of feuds and alliances",
    "Barbatos": "a towering horned hunter-lord accompanied by phantom kings and horn blasts, understanding animal tongues and raising high a moss-grown copper talisman of hidden treasure",
    "Paimon": "a towering, crowned monarch riding an imperial dromedary amid clashing cymbals and deafening trumpets, raising aloft an opulent gold talisman of spirit dominion",
    "Buer": "a towering wheel of five human legs radiating from a horned lion's head, spinning over bedrock and thrusting upward a bronze talisman of natural philosophy and healing",

    # 11 - 20
    "Gusion": "a towering cynocephalic ape-hound of weathered meteoric stone, discerning past, present, and future while brandishing aloft a heavy leaden talisman of restored honors",
    "Sitri": "a towering leopard with the wings of a massive griffin, smoldering with incandescent crimson passion and raising high a glowing copper talisman of inflamed desire",
    "Beleth": "a towering, terrible king riding a pale warhorse, preceded by blasting brass trumpets that shake the earth and thrusting upward a silver talisman of compelled love",
    "Leraje": "a towering emerald-clad archer drawing a massive recurve bow that whistles with shearing winds, raising aloft a green-patinated copper talisman of gangrenous wounds",
    "Eligos": "a towering ghostly knight clad in shining spectral plate, bearing lance, standard, and serpent while hoisting high a polished copper talisman of martial victory",
    "Zepar": "a towering crimson-armored soldier whose curved broadsword radiates scorched heat-distortion, inflaming minds and brandishing aloft a blood-rusted iron talisman of sterile lust",
    "Botis": "a towering horned viper shifting into a dreadful warrior with great tusks and drawn sword, thrusting high an electrum talisman of reconciled enemies",
    "Bathin": "a towering muscular giant with a serpent's tail riding a pale steed through planar portals, raising aloft a pale brass talisman of instantaneous transit",
    "Sallos": "a towering gallant soldier in a ducal coronet riding a leviathan crocodile with serene dignity, raising high a gleaming silver talisman of peaceful love",
    "Purson": "a towering lion-faced king clutching a hissing viper, riding a monstrous bear through blaring trumpets and hoisting high a radiant gold talisman of hidden treasures",

    # 21 - 30
    "Marax": "a towering bull with the bearded face of an ancient sage, expounding astronomy and herbal virtues while brandishing aloft a heavy iron talisman of liberal sciences",
    "Ipos": "a towering, uncanny angel with a roaring lion's head, goose's feet, and a hare's tail, thrusting upward a bright brass talisman of wit and valor",
    "Aim": "a towering three-headed nobleman riding a viper with a blazing firebrand, setting citadels ablaze and raising high a smoldering bronze talisman of consuming intellect",
    "Naberius": "a towering three-headed black crane shrieking in a raspy voice, walking through chthonic vaults and raising aloft a dark copper talisman of restored dignity",
    "Glasya-Labolas": "a towering winged bloodhound with broad griffin wings, trailing a crimson mist of bloodshed and brandishing high a bloodstained talisman of slaughter and prophecy",
    "Bune": "a towering three-headed dragon speaking in high resonant tones from sepulchral vaults, hoisting high a lustrous copper talisman of necromantic wealth",
    "Ronove": "a towering shadow-cloaked monster leaning upon a reaper's staff, harvesting weary souls while raising high an iron talisman of persuasive rhetoric and languages",
    "Berith": "a towering red-and-gold soldier upon a crimson warhorse, wearing a crown of molten flame and thrusting upward a blazing gold talisman of transmuting metals",
    "Astaroth": "a towering sorrowful fallen angel riding an infernal dragon with a coiled viper in hand, brandishing aloft a poisonous copper talisman of past falls",
    "Forneus": "a towering abyssal sea monster gliding through cold oceanic depths, granting eloquence across tongues and raising high a silver talisman of oceanic rhetoric",

    # 31 - 40
    "Foras": "a towering muscular sage carved from ancient gnarled oak, teaching logic, ethics, and herbology while hoisting high an earthen brass talisman of wisdom and longevity",
    "Asmoday": "a towering three-headed emperor vomiting furnace flames from an infernal dragon, brandishing aloft a molten gold talisman of invincible dominion and wrath",
    "Gaap": "a towering majestic emperor guiding four cardinal kings through the subterranean void, transporting mortals swiftly and raising high an inscribed leaden talisman of philosophy",
    "Furfur": "a towering colossal hart with a blazing lightning-wreathed tail, summoning roaring thunderheads and thrusting upward a crackling copper talisman of tempests and love",
    "Marchosias": "a towering armored wolf with griffin wings and a serpent's tail, vomiting columns of fire and brandishing high a gleaming silver talisman of martial fidelity",
    "Stolas": "a towering crowned raven with infinitely deep starlit eyes, perched upon celestial astrolabes and raising high an etched silver talisman of astronomy and herbs",
    "Phenex": "a towering phoenix wreathed in black-and-crimson flames, singing poetic verses with a haunting child's voice and hoisting aloft a smoldering copper talisman of melody",
    "Halphas": "a towering dark iron stock-dove shrieking with a hoarse voice, raising impenetrable fortress walls and thrusting upward a heavy brass talisman of siege warfare",
    "Malphas": "a towering soot-black raven shifting into a grim architect who shatters enemy citadels, raising aloft a dense leaden talisman of masonry and deception",
    "Raum": "a towering monstrous crow shifting into an armored plunderer, toppling dynasties and destroying cities while brandishing high a fractured copper talisman of ruin and theft",

    # 41 - 50
    "Focalor": "a towering winged warrior commanding hurricane winds and churning ocean whirlpools to sink warships, raising high a verdigris copper talisman of maritime wrath",
    "Vepar": "a towering ferocious mermaid rising from foaming breakers, guiding dread armadas with sea-fogs and hoisting aloft a tarnished silver talisman of nautical pestilence",
    "Sabnock": "a towering lion-headed soldier in heavy armor riding a pale horse, striking foes with gangrenous wounds and thrusting upward an iron talisman of fortress architecture",
    "Shax": "a towering dark stork-demon shrieking in a raspy voice, plundering mortal sight, hearing, and understanding while brandishing aloft a black talisman of sensory theft",
    "Vine": "a towering crowned lion riding a jet-black warhorse, brandishing a viper while summoning howling squalls and raising high an opulent gold talisman of exposed sorcery",
    "Bifrons": "a towering shape-shifting shade lighting spectral grave-lanterns over crumbling tombs, thrusting upward a weathered copper talisman of necromantic geometry and astronomy",
    "Vual": "a towering desert dromedary shifting into a fierce warrior speaking ancient Egyptian mysteries, raising high a hammered copper talisman of enduring affection",
    "Haagenti": "a towering winged golden bull shifting into a man, transmuting lead to gold and water to wine while hoisting high a molten gold talisman of supreme alchemy",
    "Crocell": "a towering luminous fallen angel murmuring with the sound of rushing waters, revealing steaming mineral baths and brandishing aloft a geometric brass talisman of secret sciences",
    "Furcas": "a towering white-bearded sage riding a skeletal pale steed, wielding a razor-sharp barbed spear and thrusting upward a heavy lead talisman of logic and pyromancy",

    # 51 - 60
    "Balam": "a towering three-headed king with flaming eyes, riding a raging bear with a goshawk on his fist and raising high an engraved gold talisman of past and future",
    "Alloces": "a towering knight with a fiery lion's face and scarlet eyes, riding an armored warhorse with crushing momentum and brandishing aloft a copper talisman of astronomy",
    "Camio": "a towering singing thrush stepping from burning embers to become a swordsman answering in ash, hoisting high a silver talisman of animal omens",
    "Murmur": "a towering crowned ducal warrior riding a ferocious griffin, preceded by ghostly horn blasts and raising aloft an ancient copper talisman of necromantic philosophy",
    "Orobas": "a towering magnificent horse shifting into a noble man, declaring truths of divinity without deceit and thrusting upward a sacred silver talisman of virtuous favor",
    "Gremory": "a towering all-seeing duchess wearing a jeweled coronet, riding an imperial camel through velvet shadows and brandishing aloft an opulent copper talisman of hidden treasures",
    "Ose": "a towering spotted leopard shifting into a crowned illusionist who plunges minds into madness, raising high an etched silver talisman of mental transfiguration",
    "Amy": "a towering blazing column of roaring fire resolving into an astrological magus, thrusting upward a glowing brass talisman of guarded spirit-treasures",
    "Orias": "a towering lion riding a serpent-tailed stallion, holding two hissing vipers toward starry heavens and raising high a silver talisman of astrological wisdom",
    "Vapula": "a towering lion with broad griffin wings, guiding artisans in intricate mechanics and treatises while brandishing aloft a polished copper talisman of sacred craftsmanship",

    # 61 - 72
    "Zagan": "a towering bull with griffin wings, transmuting water to wine, blood to oil, and base metals into coin while hoisting high a radiant gold talisman of transmutation",
    "Valac": "a towering winged angelic youth with blazing eyes, riding a colossal two-headed dragon through cavernous fissures and thrusting upward a serpentine talisman of hidden treasures",
    "Andras": "a towering angelic warrior with the head of a black raven, riding a ravenous midnight wolf with drawn saber and raising aloft a gleaming steel talisman of discordant slaughter",
    "Flauros": "a towering terrifying leopard leaping into a flame-eyed warrior who incinerates foes with divine fury, brandishing high a smoldering copper talisman of consuming fire",
    "Andrealphus": "a towering peacock shrieking with earsplitting pride, shifting into an architect who measures the cosmos and hoisting high an engraved silver talisman of mensuration",
    "Kimaris": "a towering heroic warrior in blackened harness riding a colossal black charger across spirit realms, raising high a bronze talisman of hidden grammar and rhetoric",
    "Amdusias": "a towering unicorn bending great forest oaks as invisible orchestral trumpets resound, thrusting upward a polished copper talisman of symphonic command",
    "Belial": "a towering pair of radiant fallen angels riding a chariot of roaring fire, speaking with deceptive honeyed grace and raising aloft an imperial gold talisman of sovereign favor",
    "Decarabia": "a towering five-pointed pentacle-star shifting into an enchanter surrounded by flocks of singing birds, brandishing high a lustrous silver talisman of herbal lore",
    "Seere": "a towering breathtaking youth riding a winged pegasus, moving across the world in the blink of an eye and hoisting high an electrum talisman of instantaneous abundance",
    "Dantalion": "a towering nobleman of infinite shifting faces, holding an open grimoire of human desires and thrusting upward a dark copper talisman of telepathic sway",
    "Andromalius": "a towering stern judge holding a colossal writhing serpent, tracking down thieves and punishing the wicked while raising high an iron talisman of absolute retribution",
}

# Occurrence index per gate for angels whose name repeats in the roster
# (Lauviah appears at gates 11 and 17); parallel to GATES.
_ANGEL_SEEN = {}
ANGEL_OCCURRENCE = []
for _gg, _aa, _dd, _ee in GATES:
    ANGEL_OCCURRENCE.append(_ANGEL_SEEN.get(_aa, 0))
    _ANGEL_SEEN[_aa] = _ANGEL_SEEN.get(_aa, 0) + 1
del _ANGEL_SEEN, _gg, _aa, _dd, _ee


def angel_form(angel, occurrence=0):
    """Expanded traditional form for an angel; occurrence disambiguates repeats."""
    key = angel
    if occurrence:
        alts = [k for k in ANGEL_FORMS if k != angel and k.startswith(angel)]
        if alts:
            key = alts[occurrence - 1]
    return ANGEL_FORMS[key]


def demon_form(demon):
    """Expanded traditional Goetic form for a demon."""
    return DEMON_FORMS[demon]


# Physical gate architecture per element -- the thing he actually walks through.
THRESHOLDS = {
    "Fire":  "a towering arch of black volcanic glass wreathed in slow-burning cymatic flame",
    "Earth": "a moss-covered stone trilithon humming with a low resonant tone",
    "Air":   "a lattice of prismatic light refracting the sky into slow ribbons",
    "Water": "an arch of living coral and breathing bioluminescent membrane",
}

# Wheel tones: each act resolves to a pure tone a perfect fifth (1.5x) above
# the last, folded into the 40-80Hz sub range -- the circle of fifths walking
# the 12 signs. Every segment OPENS continuing the previous segment's tone and
# CLOSES in a calm moment at its tail tone -- the tail tone IS the handoff.
WHEEL_TONES = {
    1: 40, 2: 60, 3: 45, 4: 68, 5: 51, 6: 76,
    7: 57, 8: 43, 9: 64, 10: 48, 11: 72, 12: 54,
}

GLYPHS = {
    1:  ("Aries",       "the Ram's glyph, two bold curved horns"),
    2:  ("Taurus",      "the Bull's glyph, a horned circle"),
    3:  ("Gemini",      "the Twins' glyph, twin pillars joined by arcs"),
    4:  ("Cancer",      "the Crab's glyph, twin spiraling whorls"),
    5:  ("Leo",         "the Lion's glyph, a single sweeping mane-curve"),
    6:  ("Virgo",       "the Maiden's glyph, an intricate looping sigil"),
    7:  ("Libra",       "the Scales' glyph, a balanced horizon over a rising curve"),
    8:  ("Scorpio",     "the Scorpion's glyph, a barbed tail striking upward"),
    9:  ("Sagittarius", "the Archer's glyph, a drawn bow loosing an arrow skyward"),
    10: ("Capricorn",   "the Sea-Goat's glyph, a horned crown diving into a fishtail curl"),
    11: ("Aquarius",    "the Water-Bearer's glyph, twin zigzag waves"),
    12: ("Pisces",      "the Fishes' glyph, two mirrored arcs bound by a single cord"),
}

ACT_ENVS = {
    1:  "a suspended perfect circle of floating polished obsidian and volcanic glass shards erupting with cymatic fire fractals",
    2:  "colossal monolithic stone temples covered in ancient glowing moss-covered sigils",
    3:  "dual mirrored prisms floating on aggressive high-altitude air currents",
    4:  "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton",
    5:  "a blinding solar corona filled with spinning gold geometric projections",
    6:  "hyper-precise cold sacred geometry and infinite anatomical spine mappings",
    7:  "unfolding scales of light and absolute mirrored symmetry",
    8:  "a massive obsidian gateway pulsing with a bioluminescent serpentine nervous system",
    9:  "an arrow of pure light shooting through an expanding cosmic Akashic library",
    10: "an ancient mountain archive of deeply carved glowing seals",
    11: "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations",
    12: "an infinite borderless ocean of consciousness dissolving into starlight mist",
}

DISSOLVE = {
    "Fire":  "dying cymatic flames and cooling volcanic glass",
    "Earth": "crumbling stone dust and fading moss-light",
    "Air":   "thinning high-altitude currents and shattering prisms",
    "Water": "receding bioluminescent waters and dissolving mist",
}

GLYPH_FORMATION = {
    "Fire":  "forged from living flame and volcanic glass",
    "Earth": "carved from moss-covered living stone",
    "Air":   "woven from prismatic light and high wind",
    "Water": "condensed from bioluminescent water and living coral",
}

IGNITION = {
    "Fire":  "cymatic flames erupt outward, igniting",
    "Earth": "stone dust settles and monoliths rise, revealing",
    "Air":   "the air thins into high currents, revealing",
    "Water": "bioluminescent waters flood inward, revealing",
}

# ------------------------------------------------------------------ JSON builders
# Every Omni request is ONE JSON document: `meta` (standing rules + state
# variables, substituted from the environment at run time) + `segment` (every
# variable for this gate). The JSON text is the entire text part of the
# request -- no markdown, no mixed prose.


def build_meta():
    """Standing rules + run-time state variables. Identical in every request."""
    return {
        "schema": SCHEMA_VERSION,
        "state": {
            "model": MODEL_ID,
            "duration_s": GATE_SECONDS,
            "resolution": RESOLUTION,
            "draft_mode": DRAFT_MODE,
            "stride_cycle_s": 1.2,
            "tempo_bpm": 100,
            "forward_speed_m_s": 1.2,
            "camera": "continuous unbroken over-the-shoulder rear tracking dolly",
            "lens": "35mm T1.3 anamorphic",
        },
        "rules": {
            "continuity": "every segment begins on the last frame of the previous segment, ends at the seam to the next gate in mid-motion",
            "appearance": "the World Bridger's appearance is locked to the bridger_ref image -- reproduce his avatar in the new last feame reference rendering only his body from it exactly, never restyle or reinterpret it; airborne ash and embers are environment, never markings on his robes other than glowing golden runes and basalt staff",
            "behavior": "at the center of the scene 5s mark he should continue his pace as he looks at the massive deities ahead. He raises his staff his voice booms out the names of the angel and demon who immediately bow in submission having been defeated by his name magic",
            "behavior_order": "angel first, then demon -- never swapped, no text overlays and both names should be said within 1s of eachother at the halfway point",
            "pacing": "stride is metronome-locked at 100 BPM, 1.2 m/s, identical in every shot -- he never walks faster, never slows down, never hurries a gate crossing",
            "motion": "the dolly never stops its forward drift; the shot flows continuously first frame to last;",
            "heading": "he walks dead straight ahead at all times -- spine aligned to his forward axis, never drifting, leaning, or cocking; the naming beat is the only head motion in the traversal",
            "segment_boundary": "every segment ends mid-stride, mid-dolly, exactly as the next one begins in the center of the gates doorway, so the hard cut between them is invisible and audio should seam perfectly",
            "camera_lock": "locked over-the-shoulder rear tracking; forward motion only; no cuts, no angle changes, no unscripted camera moves",
            "text": "zero text overlays of any kind -- everything visible must exist physically inside the scene never any text",
            "audio": "full diegetic audio is rendered: the act's audio bed, the wheel-tone handoff, and the two guardian names spoken by the Bridger -- the ONLY words ever spoken in the traversal",
        },
    }


def build_audio(act_audio, tone, spoken_names):
    """Full-audio block: act bed + wheel-tone handoff + the two spoken names."""
    return {
        "mode": "full_diegetic_audio",
        "open": "already continuing the previous segment's tail tone -- no click, no jump",
        "bed": act_audio,
        "spoken": [
            {"at": "naming beat", "speaker": "the World Bridger", "line": name}
            for name in spoken_names
        ],
        "speech_rule": "the two guardian names are the ONLY words spoken in the entire traversal",
        "close": f"all sound settles into a calm pure {tone}Hz wheel tone, held steady through the final seconds -- this is the handoff; the next segment picks up exactly from this tone. Render the tail as a seamless continuation, never an ending.",
        "forbidden": ["narration", "voiceover", "host commentary", "music", "text", "non-diegetic sound effects"],
    }


def build_naming_beat(angel, demon):
    """The simplified naming beat. No face is ever mentioned."""
    return {
        "at": "the threshold, mid-segment",
        "actions": [
            {"actor": "bridger", "do": "raise_staff"},
            {"actor": "bridger", "do": "nod", "toward": "left", "speak": angel},
            {"actor": "bridger", "do": "nod", "toward": "right", "speak": demon},
            {"actor": angel, "do": "bow"},
            {"actor": demon, "do": "bow"},
            {"actor": "guardians", "do": "submit and acknowledge"},
        ],
    }


def gate_segment(i, gate, angel, demon, env, element, next_env):
    """JSON variable block for one gate segment."""
    act_num, act_name, _el, act_audio = act_of(gate)
    tone = WHEEL_TONES[act_num]
    mid = GATE_SECONDS // 2
    seg = {
        "id": f"gate_{gate:02d}",
        "kind": "gate",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": act_name,
            "element": element,
            "wheel_tone_hz": tone,
        },
        "guardians": {
            "left": {
                "name": angel,
                "choir": choir_of(gate),
                "form": angel_form(angel, ANGEL_OCCURRENCE[i]),
            },
            "right": {
                "name": demon,
                "order": "Ars Goetia",
                "form": demon_form(demon),
            },
        },
        "environment": env,
        "threshold": THRESHOLDS[element],
        "next_environment": next_env,
        "beats": {
            "approach": f"the threshold looms ahead -- {THRESHOLDS[element]} -- with {angel} gathering at its left pillar and {demon} at its right pillar",
            "naming": build_naming_beat(angel, demon),
            "naming_at_s": f"around {mid - 1}-{mid + 1}s",
        },
        "audio": build_audio(act_audio, tone, [angel, demon]),
    }
    if gate == 48:
        seg["beats"]["crossing"] = (
            "at the 28th spoke rim the Bridger strikes his staff down on the stone -- "
            "a golden 108-degree geodesic Zen Shield flares to life around him, "
            "shattering the matrix wall. Fully solid and opaque, he strides on as the "
            "camera turns 90 degrees left off the zodiac track into the zero-gravity "
            "Ophiuchus Void (the single scripted camera move of the traversal), and "
            "all audio drops into absolute dead silence."
        )
        # The breach ends in dead silence, not a wheel tone -- the Act 9
        # transition opens with "the sound rises from absolute silence".
        seg["audio"]["close"] = (
            "absolute dead silence -- the breach ends in silence, not a wheel tone. "
            "Render the tail as a seamless continuation into silence, never an ending."
        )
    elif gate == 72:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the final gate, fully solid and opaque -- he "
            "does not fade, dissolve, or vanish -- as the borderless ocean detonates "
            "into white-gold ignition: volcanic glass and cymatic fire fractals blooming "
            "outward. The 360-degree loop closes seamlessly back into the Gate 1 threshold."
        )
    else:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the gate plane on foot between the parted "
            "guardians -- fully solid and opaque, he does not fade, dissolve, or vanish "
            f"-- and the camera follows him through; beyond the far plane, {next_env} "
            "already resolving through the shimmer."
        )
    return seg


def transition_segment(act_num, is_first):
    """JSON variable block for one act-transition segment."""
    sign, glyph = GLYPHS[act_num]
    _n, _name, element, act_audio = act_of(act_num * 6 - 5)  # first gate of the act
    tone = WHEEL_TONES[act_num]
    seg = {
        "id": f"act_{act_num:02d}_transition",
        "kind": "act_transition",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": f"{sign} transition",
            "element": element,
            "wheel_tone_hz": tone,
        },
        "glyph": {
            "sign": sign,
            "description": glyph,
            "formation": f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}, hanging in the space and slowly rotating, lit from within",
        },
        "elemental_shift": {
            "outgoing": DISSOLVE[act_of((act_num - 1) * 6)[2]] if not is_first else "dying cymatic flames and cooling volcanic glass",
            "incoming": f"{IGNITION[element]} {ACT_ENVS[act_num]}",
        },
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken -- no guardians stand at this threshold",
    }
    if is_first:
        seg["scene"] = (
            "opening cold: the camera is already locked over the shoulder of the World "
            "Bridger as he strides forward through dying cymatic flames and cooling "
            "volcanic glass, the old element dying away. Golden wireframe 72-gate rings "
            "and glowing cuneiform code matrices float in zero gravity around him. Ahead, "
            f"the zodiac glyph of {sign} -- {glyph} -- materializes. The camera passes "
            f"beneath the glyph and {IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["open"] = "the loop closes here -- open continuing the calm 54Hz wheel tone the final Pisces gate left behind"
        seg["audio"]["spoken"] = []
    else:
        prev_element = act_of((act_num - 1) * 6)[2]
        prev_audio = act_of((act_num - 1) * 6)[3]
        prev_tone = WHEEL_TONES[act_num - 1]
        seg["scene"] = (
            "starting exactly from the first frame -- same camera, same forward drift, no "
            f"cut -- the old world falls behind. The camera drifts through {DISSOLVE[prev_element]} "
            f"as the old element dies away. Ahead, the zodiac glyph of {sign} -- {glyph} -- "
            f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}. The Bridger "
            f"walks on beneath it, staff in hand. As the camera passes beneath the glyph, "
            f"{IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["spoken"] = []
        if act_num == 9:
            # Act 9 follows gate 48's breach, which ends in dead silence, not a tone.
            seg["audio"]["open"] = "the sound rises from absolute silence"
        else:
            seg["audio"]["open"] = (
                f"gliding continuously from the fading {prev_tone}Hz wheel tone -- a smooth "
                f"pitch glide, no click, no jump -- as {prev_audio} dissolves into {act_audio}"
            )
    return seg


def cold_open_segment():
    """JSON variable block for the cold-open pre-step: black void, him walking.

    Generated FIRST; its last frame seeds segment 0. This is the loop point --
    gate 72's tail walks back into this frame.
    """
    return {
        "id": "cold_open",
        "kind": "cold_open",
        "shot": "single unbroken shot",
        "note": "the loop point and the film's first frame -- a deliberate frame, never an arbitrary fresh generation",
        "scene": (
            "pure infinite solid matte black background (#000000) -- absolute zero "
            "background geometry, no pillars, no fire, no ground textures, no horizon. "
            "Only the World Bridger exists in the frame: dead-center rear view, mid-stride, "
            "carved basalt staff in his RIGHT hand, walking dead straight ahead along the "
            "central axis at his metronomic 100 BPM pace. No guardians, no naming beat."
            "hood always up, black wool, crimson lining and glowing cuneiform glyphs and runes."
        ),
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken",
        "audio": {
            "mode": "full_diegetic_audio",
            "open": "continuing the calm 54Hz wheel tone the final Pisces gate leaves behind -- the loop closes here",
            "bed": "the near-silence of the void under a faint sub-bass",
            "spoken": [],
            "speech_rule": "no words are spoken in the cold open",
            "close": "steady and unresolved -- segment 0 continues it; the old fire world ignites out of the dark",
            "forbidden": ["narration", "voiceover", "host commentary", "music", "non-diegetic sound effects"],
        },
    }


def build_request(segment, image_names):
    """Assemble the full JSON request document for one generation."""
    sources = {}
    if "bridger_ref" in image_names:
        sources["bridger_ref"] = {
            "image": image_names["bridger_ref"],
            "role": "appearance reference for the World Bridger -- appearance ONLY; it does not define the environment, pose, lighting, or scene",
        }
    if "first_frame" in image_names:
        sources["first_frame"] = {
            "image": image_names["first_frame"],
            "role": "the last frame of the previous segment -- begin on this exact frame and continue its motion, audio and lighting, and scene seamlessly",
        }
    return {
        "request": "generate_video",
        "engine": "gemini_omni",
        "meta": build_meta(),
        "sources": sources,
        "segment": segment,
    }


SEGMENTS = []
_prev_act = None
for i, (g, a, d, env) in enumerate(GATES):
    act_num = act_of(g)[0]
    if act_num != _prev_act:
        SEGMENTS.append({
            "key": f"act{act_num:02d}_transition",
            "segment": transition_segment(act_num, is_first=(len(SEGMENTS) == 0)),
        })
        _prev_act = act_num
    SEGMENTS.append({
        "key": f"gate{g:02d}",
        "segment": gate_segment(i, g, a, d, env, act_of(g)[2],
                                GATES[i + 1][3] if i + 1 < len(GATES) else None),
    })

assert len(SEGMENTS) == 84

# ------------------------------------------------------------------ inputs & API
# Stills only: the bridger_ref still and the first-frame still. No video
# references in this build.


def to_bytes(data):
    if isinstance(data, str):
        return base64.b64decode(data)
    return bytes(data)


def build_inputs(segment, first_frame_b64):
    """Build the Omni input list: [bridger_ref still?, first_frame still?, JSON text].

    The text part is the ENTIRE prompt as one JSON document -- meta + sources
    + segment. Nothing is markdown, nothing is mixed prose.
    """
    contents = []
    image_names = {}
    n_img = 0

    def add_image(b64, mime="image/png"):
        nonlocal n_img
        n_img += 1
        contents.append({"type": "image", "data": b64, "mime_type": mime})
        return f"Image{n_img}"

    if BRIDGER_REF_B64:
        image_names["bridger_ref"] = add_image(BRIDGER_REF_B64, "image/jpeg")
    if first_frame_b64:
        image_names["first_frame"] = add_image(first_frame_b64)

    request = build_request(segment, image_names)
    text = json.dumps(request, indent=2)
    contents.append({"type": "text", "text": text})

    if len(contents) == 1:
        return text
    return contents


def get_output_video(interaction):
    """Prefer .output_video, fall back to manual step traversal."""
    out = getattr(interaction, "output_video", None)
    if out is not None and (getattr(out, "data", None) or getattr(out, "uri", None)):
        return out
    for step in reversed(getattr(interaction, "steps", []) or []):
        if getattr(step, "type", None) == "model_output" and getattr(step, "content", None):
            for part in reversed(step.content):
                if getattr(part, "type", None) == "video":
                    return part
    return None


def download_uri(uri, dest_path, timeout_s=1800, poll_s=10):
    """Poll the hosted file until ACTIVE, then download it."""
    file_id = uri.rstrip("/").split("/")[-1].split(":")[0].split("?")[0]
    name = f"files/{file_id}"
    deadline = time.time() + timeout_s
    t0 = time.time()
    vlog(f"    POLL {name} every {poll_s}s (timeout {timeout_s}s)")
    while time.time() < deadline:
        elapsed = time.time() - t0
        try:
            info = client.files.get(name=name)
            state = getattr(info.state, "name", str(info.state))
            if state == "ACTIVE":
                log(f"    poll +{elapsed:.0f}s: {state} -- downloading")
                break
            if state == "FAILED":
                raise RuntimeError("video generation failed (file state FAILED)")
            log(f"    poll +{elapsed:.0f}s: hosted file state: {state}")
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"    poll +{elapsed:.0f}s: waiting on hosted file ({exc.__class__.__name__}: {str(exc)[:120]})")
        time.sleep(poll_s)
    else:
        raise TimeoutError(f"timed out after {timeout_s}s waiting for hosted video file")
    data = client.files.download(file=uri)
    with open(dest_path, "wb") as f:
        f.write(to_bytes(data))
    size = os.path.getsize(dest_path) / 1e6
    log(f"    saved {dest_path} ({size:.1f} MB)")


def _norm(exc):
    # Normalize underscores so RATE_LIMIT_EXCEEDED / quota_exceeded style
    # status enums can't slip past the classifiers.
    return re.sub(r"[_\s]+", "", f"{type(exc).__name__} {exc}".lower())


def is_quota_error(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "429", "resourceexhausted", "ratelimitexceeded", "quotaexceeded",
        "exceededyourquota", "toomanyrequests", "dailylimitexceeded",
    ))


def is_auth_error(exc):
    t = _norm(exc)
    return (not is_quota_error(exc)) and any(k in t for k in (
        "401", "invalidapikey", "apikeyinvalid", "apikeynotvalid",
        "apikeyisinvalid", "unauthorized", "permissiondenied",
    ))


def is_content_refusal(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "prohibitedcontent", "prohibiteduse", "contentguidelines",
        "blockeddueto", "violatedgoogle",
        "likeness", "likenesses", "realperson", "realpeople",
        "contentblocked", "inputblocked",
    ))


def prompt_for_new_key(reason):
    print()
    print("=" * 64)
    print(f"  KEY ISSUE MID-RUN: {reason}")
    print("  Paste another Google AI Studio key to keep rendering,")
    print("  or type 'skip' to skip this segment, 'quit' to stop the run.")
    print("=" * 64)
    try:
        new_key = getpass.getpass("  New GEMINI_API_KEY (hidden input): ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise KeyboardInterrupt
    if not new_key:
        return None
    low = new_key.lower()
    if low in ("skip", "s"):
        return "skip"
    if low in ("quit", "q", "exit"):
        raise KeyboardInterrupt
    return new_key


def generate_segment(seg, first_frame_b64, dest_path):
    """Generate one Omni segment from its JSON request. Returns True/False."""
    mode = "chained" if first_frame_b64 else "fresh"
    t0 = time.time()
    log(f"[{seg['key']}] generating ({RESOLUTION}, {mode})...")
    inputs = build_inputs(seg["segment"], first_frame_b64)
    txt = inputs if isinstance(inputs, str) else inputs[-1]["text"]
    if isinstance(inputs, str):
        parts = [f"json:{len(inputs)} chars"]
    else:
        parts = []
        for p in inputs:
            if "text" in p:
                parts.append(f"json:{len(p['text'])} chars")
            elif p.get("type") == "image":
                parts.append(f"image:{len(p.get('data') or '')/1e6:.1f} MB b64 {p.get('mime_type', '?')}")
            else:
                parts.append(str(p.get("type")))
    vlog(f"    REQUEST -> model={MODEL_ID} resolution={RESOLUTION} delivery=uri")
    vlog(f"    REQUEST parts: {', '.join(parts)}")
    if VERBOSE:
        # In verbose mode show the JSON skeleton (keys only) -- full JSON is
        # in --dry-run output.
        req = json.loads(txt)
        vlog(f"    REQUEST json: meta.rules={len(req['meta']['rules'])} rules, "
             f"sources={list(req['sources'])} segment={req['segment']['id']}")
    log(f"    sending interactions.create ...")
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=inputs,
        response_format={"type": "video", "resolution": RESOLUTION, "delivery": "uri"},
    )
    log(f"    RESPONSE in {time.time()-t0:.1f}s | interaction id: {interaction.id}")
    video = get_output_video(interaction)
    if video is None:
        log("    done, but no video in response -- recording as failed.")
        return False
    if getattr(video, "uri", None):
        vlog(f"    RESPONSE video: uri={video.uri}")
        download_uri(video.uri, dest_path)
    elif getattr(video, "data", None):
        vlog(f"    RESPONSE video: inline data ({len(to_bytes(video.data))/1e6:.1f} MB)")
        with open(dest_path, "wb") as f:
            f.write(to_bytes(video.data))
        log(f"    saved {dest_path}")
    else:
        log("    done, but no video payload -- recording as failed.")
        return False
    log(f"    [{seg['key']}] done in {time.time()-t0:.1f}s")
    return True


def generate_with_retry(seg, first_frame_b64, dest_path):
    """Returns True (rendered), False (failed -- recorded, run continues).

    Tiered fallback on content refusal: chained (with first-frame still) ->
    fresh (no still). Quota/auth errors retry the same tier after a key swap.
    """
    tiers = []
    if first_frame_b64 is not None:
        tiers.append(first_frame_b64)
    tiers.append(None)
    tier_idx = 0
    while True:
        ff = tiers[tier_idx]
        tier_label = "chained" if ff else "fresh"
        vlog(f"    attempt tier {tier_idx+1}/{len(tiers)} ({tier_label})")
        try:
            return generate_segment(seg, ff, dest_path)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if is_content_refusal(exc):
                if tier_idx + 1 < len(tiers):
                    tier_idx += 1
                    print(f"    refused -- retrying [{seg['key']}] fresh (no first-frame still)...")
                    continue
                print(f"    content refused ({exc.__class__.__name__}) -- recording as failed.")
                print(f"    detail: {str(exc)[:200]}")
                return False
            if is_quota_error(exc):
                reason = f"quota/rate limit hit ({exc.__class__.__name__})"
            elif is_auth_error(exc):
                reason = f"API key rejected ({exc.__class__.__name__})"
            else:
                raise
            print(f"    {reason}: {str(exc)[:180]}")
            new_key = prompt_for_new_key(reason)
            if new_key == "skip":
                print(f"    [{seg['key']}] skipped by user choice.")
                return False
            if new_key:
                set_active_key(new_key, source="mid-run swap")
                print(f"    retrying [{seg['key']}] with the new key...")
                continue
            print("    no replacement key entered -- recording segment as failed.")
            return False

# ------------------------------------------------------------------ frames & stitch


def last_frame_of(mp4_path):
    """Extract the TRUE last frame of a segment as PNG bytes (cached).

    Decodes the tail of the file and keeps the final decoded frame -- exact,
    unlike `-sseof -0.5 -vframes 1`, which lands ~14 frames early and causes
    a visible jump at every chained transition.
    """
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_lastframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    tmpdir = tempfile.mkdtemp(prefix="tail_")
    try:
        pattern = os.path.join(tmpdir, "tail_%04d.png")
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-sseof", "-3", "-i", mp4_path,
                        "-frames:v", "400", pattern],
                       check=True, timeout=180, stdin=subprocess.DEVNULL)
        files = sorted(glob.glob(os.path.join(tmpdir, "tail_*.png")))
        if not files:
            print("    frame extract: no frames decoded; continuing without first frame")
            return None
        shutil.move(files[-1], out)
    except Exception as exc:
        print(f"    frame extract failed ({exc}); continuing without first frame")
        return None
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
    with open(out, "rb") as f:
        return f.read()


def first_frame_of(mp4_path):
    """Extract the FIRST frame of a segment as PNG bytes (cached) -- for gap-fill."""
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_firstframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    try:
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-i", mp4_path, "-vframes", "1", "-q:v", "2", out],
                       check=True, timeout=120, stdin=subprocess.DEVNULL)
    except Exception as exc:
        print(f"    first-frame extract failed ({exc}); no end-frame constraint")
        return None
    with open(out, "rb") as f:
        return f.read()


def probe_duration(ffmpeg, path):
    """Probe exact video stream duration; fall back to container duration."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        try:
            cmd = [
                ffprobe, "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ]
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
            if out and out != "N/A":
                return float(out)
        except Exception:
            pass
    p = subprocess.run([ffmpeg, "-i", path], capture_output=True, text=True,
                       timeout=60, stdin=subprocess.DEVNULL)
    m = re.search(r"Duration:\s*(\d+):(\d+):([\d.]+)", p.stderr)
    if not m:
        raise RuntimeError(f"could not probe duration of {path}")
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def stitch_xfade(ffmpeg, paths, out, preset="medium", timeout=1800,
                 normalize=True):
    """Join with short video+audio crossfades at every boundary."""
    f = XFADE_S
    durs = [probe_duration(ffmpeg, p) for p in paths]
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    fc = []
    # Normalize every input first: segments arrive with mixed timebases and
    # xfade refuses to link mismatched timebases -- one bad join kills the
    # whole chain.
    vlabels, alabels = [], []
    scale_dim = "640:360" if DRAFT_MODE else "1920:1080"
    for k in range(len(paths)):
        if normalize:
            # 1. Reset PTS to 0.
            # 2. Add tpad to clone the final frame if it runs short by a few milliseconds.
            fc.append(
                f"[{k}:v]scale={scale_dim},setsar=1,fps=24,setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5},format=yuv420p[v{k}n]"
            )
            # Trim audio to exact video stream duration to prevent audio/video length drift
            fc.append(
                f"[{k}:a]aresample=48000,aformat=channel_layouts=stereo,"
                f"asetpts=PTS-STARTPTS,atrim=0:{durs[k]:.3f}[a{k}n]"
            )
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")
        else:
            fc.append(
                f"[{k}:v]setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5}[v{k}n]"
            )
            fc.append(f"[{k}:a]asetpts=PTS-STARTPTS[a{k}n]")
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")

    vprev = vlabels[0]
    for k in range(1, len(paths)):
        off = sum(durs[:k]) - f * k
        vout = f"vx{k}"
        fc.append(f"[{vprev}][{vlabels[k]}]xfade=transition=fade:duration={f}:offset={off:.3f}[{vout}]")
        vprev = vout

    aprev = alabels[0]
    for k in range(1, len(paths)):
        aout = f"ax{k}"
        fc.append(f"[{aprev}][{alabels[k]}]acrossfade=d={f}[{aout}]")
        aprev = aout

    cmd += ["-filter_complex", ";".join(fc),
            "-map", f"[{vprev}]", "-map", f"[{aprev}]",
            "-c:v", "libx264", "-crf", "18", "-preset", preset,
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=timeout, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (crossfaded joins)")


def stitch(filenames, out="mirror_of_spheres_json.mp4"):
    """Join segments in order. Falls back to plain concat if xfade fails."""
    paths = [os.path.abspath(os.path.join(OUT_DIR, n)) for n in filenames]
    lst = os.path.join(OUT_DIR, "concat.txt")
    with open(lst, "w") as f:
        for p in paths:
            f.write(f"file '{p}'\n")
    ffmpeg = FFMPEG or shutil.which("ffmpeg")
    if ffmpeg is None:
        print("ffmpeg not found -- skipping auto-stitch. Run:")
        print(f"  ffmpeg -f concat -safe 0 -i {lst} -c copy {out}")
        return
    if XFADE_S > 0 and len(paths) > 1:
        try:
            # Two-level crossfade: one giant 84-input filtergraph exhausts
            # memory (84 simultaneous decoders + a 160+ filter chain), so
            # stitch small chunks first and then crossfade the chunks.
            CHUNK = 7
            if len(paths) > CHUNK:
                chunkdir = os.path.join(OUT_DIR, "chunks")
                os.makedirs(chunkdir, exist_ok=True)
                chunk_paths = []
                for ci in range(0, len(paths), CHUNK):
                    cpaths = paths[ci:ci + CHUNK]
                    if len(cpaths) > 1:
                        cfile = os.path.abspath(os.path.join(
                            chunkdir, f"chunk{ci // CHUNK:02d}.mp4"))
                        print(f"  stitching chunk {ci // CHUNK + 1} "
                              f"({len(cpaths)} segments)...")
                        stitch_xfade(ffmpeg, cpaths, cfile, normalize=True)
                        chunk_paths.append(cfile)
                    else:
                        chunk_paths.append(cpaths[0])
                print(f"  stitching {len(chunk_paths)} chunks into final...")
                stitch_xfade(ffmpeg, chunk_paths, out, timeout=3600,
                             normalize=True)
                shutil.rmtree(chunkdir, ignore_errors=True)
            else:
                stitch_xfade(ffmpeg, paths, out, normalize=True)
            return
        except Exception as exc:
            import traceback
            print(f"xfade stitch failed ({exc.__class__.__name__}: {exc}); falling back to concat filter")
            traceback.print_exc()
    # Fallback: concat filter with re-encode. Immune to the timestamp/DTS
    # quirks of stream-copy across mixed-mux segments.
    segs = "".join(f"[{k}:v][{k}:a]" for k in range(len(paths)))
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    cmd += ["-filter_complex", f"{segs}concat=n={len(paths)}:v=1:a=1[vcat][acat]",
            "-map", "[vcat]", "-map", "[acat]",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=3600, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (concat fallback)")


# ------------------------------------------------------------------ run

def cold_open_path():
    return os.path.join(OUT_DIR, "cold_open.mp4")


def run_cold_open():
    """Pre-step: generate the black-void loop-point frame, return its last
    frame as PNG bytes (the seed for segment 0). Reuses the cached file."""
    path = cold_open_path()
    if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
        print("[cold_open] already rendered -- resuming, skipping")
    else:
        print("[cold_open] generating the loop-point frame (black void, him walking)...")
        ok = generate_with_retry({"key": "cold_open", "segment": cold_open_segment()},
                                 None, path)
        if not ok:
            print("[cold_open] failed -- segment 0 will generate fresh.")
            return None
    raw = last_frame_of(path)
    if raw:
        print("[cold_open] last frame extracted -- seeding segment 0.")
    return raw


def dry_run():
    """Print every segment's full JSON request. No API calls, no key."""
    req0 = build_request(cold_open_segment(), {})
    print(f"# Mirror of Spheres JSON prompts -- {len(SEGMENTS)} segments + cold open (dry run)\n")
    print("## cold_open\n")
    print(json.dumps(req0, indent=2) + "\n")
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            break
        req = build_request(seg["segment"], {"bridger_ref": "Image1",
                                             "first_frame": "Image2"})
        print(f"## {i:02d}_{seg['key']} ({len(json.dumps(req))} chars)\n")
        print(json.dumps(req, indent=2) + "\n")


def seg_path(i):
    return os.path.join(OUT_DIR, f"{i:02d}_{SEGMENTS[i]['key']}.mp4")


def main():
    if DRY_RUN:
        dry_run()
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    rlog_init({
        "schema": SCHEMA_VERSION,
        "model": MODEL_ID,
        "resolution": RESOLUTION,
        "draft_mode": DRAFT_MODE,
        "gate_seconds": GATE_SECONDS,
        "xfade_s": XFADE_S,
        "max_segments": MAX_SEGMENTS,
        "cold_open": COLD_OPEN,
        "bridger_ref": bool(BRIDGER_REF_B64),
    })
    print(f"ffmpeg: {FFMPEG if FFMPEG else 'NOT FOUND -- chaining disabled, all segments will generate fresh!'}")
    print(f"segments: {len(SEGMENTS)} (12 act-transitions + 72 gates) | {RESOLUTION} | "
          f"xfade: {XFADE_S}s {'(hard cuts)' if XFADE_S == 0 else ''} | "
          f"{'DRAFT' if DRAFT_MODE else 'PRODUCTION'}")
    print("prompting: all-JSON (meta state variables + per-segment variables) | "
          "chaining: last-frame stills only, no video references")

    # Cold open first: its last frame seeds segment 0.
    cold_seed = run_cold_open() if COLD_OPEN else None

    made, failed = [], []
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            print(f"(stopping at MAX_SEGMENTS={MAX_SEGMENTS})")
            break
        path = seg_path(i)
        if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[{seg['key']}] already rendered -- resuming, skipping")
        else:
            first_frame = None
            if i == 0 and cold_seed:
                print("    seeding from cold-open last frame")
                first_frame = base64.b64encode(cold_seed).decode("ascii")
            elif i > 0 and os.path.exists(seg_path(i - 1)):
                print(f"    chaining from last frame of {os.path.basename(seg_path(i - 1))}")
                raw = last_frame_of(seg_path(i - 1))
                first_frame = base64.b64encode(raw).decode("ascii") if raw else None
                if first_frame is None:
                    print(f"    WARNING: last-frame extraction failed -- [{seg['key']}] will generate FRESH (expect a jump cut)")
            elif i > 0:
                print("    (no previous segment file; starting fresh)")
            # Gap-fill: if the NEXT segment already exists, constrain this
            # one's tail to the next segment's first frame.
            if i + 1 < len(SEGMENTS) and os.path.exists(seg_path(i + 1)):
                print(f"    (gap-fill tail note: {os.path.basename(seg_path(i + 1))} already exists)")
            t0 = time.time()
            try:
                ok = generate_with_retry(seg, first_frame, path)
            except KeyboardInterrupt:
                print("\nInterrupted. Re-run anytime -- finished segments are kept.")
                break
            rlog({"event": "segment_done", "key": seg["key"], "ok": ok,
                  "elapsed_s": round(time.time() - t0, 1),
                  "chained": first_frame is not None})
            if not ok:
                failed.append(seg["key"])
                continue
        if os.path.exists(path):
            made.append(os.path.basename(path))
    if failed:
        print(f"\nFailed segments: {failed}")
        print("Tweak / check quota, then re-run -- finished segments are skipped.")
    if made:
        stitch(made)


if __name__ == "__main__":
    main()

In [ ]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
The Mirror of Spheres -- 72-Gate Decan Traversal (Gemini Omni, all-JSON prompts).

84 chained segments: each of the 12 acts opens with a 10s act-transition --
the zodiac glyph materializing as a vast physical sigil while the elemental
shift occurs -- followed by its 6 gates (~12:00+ total at 10s gates).

WHAT'S DIFFERENT IN THIS BUILD
------------------------------
- ALL-JSON PROMPTS. Every request sent to Omni is one JSON document: a `meta`
  block (standing rules + state variables, substituted from the environment at
  run time) and a `segment` block (every variable for that gate: act, element,
  wheel tone, angel {name, choir, form}, demon {name, form}, environment,
  threshold, glyph, beats). No markdown, no mixed prose -- the model gets
  structure, the way it likes it.
- FULL AUDIO. Diegetic audio is rendered: the act's audio bed, the wheel-tone
  handoff (every segment opens continuing the previous tail tone and closes
  settled into its own), and the ONLY permitted spoken words -- the World
  Bridger speaking the two guardian names at the naming beat.
- STILL-FRAME CHAINING ONLY. Each segment takes a still of the TRUE last frame
  of the previous segment and patches it in as the first frame. No video
  references (no prev-segment attach, no hero clip) -- stills only.
- COLD OPEN. An optional pre-step generates the loop-point frame first: pure
  black void with him walking ("a black screen with him on it"). Its last
  frame seeds segment 0, so the film opens deliberately and gate 72's tail
  closes back into it -- the loop plays all day with no visible seam.
- SIMPLIFIED BEHAVIOR. The prompts never mention his face at all. At each
  gate: he raises his staff and nods toward the angel as he speaks its name,
  then nods toward the demon as he speaks its name. That's it.
- DRAFT/PROD TOGGLE. DRAFT_MODE=1 -> 360p into separate dirs at roughly a
  third of the cost. Unset -> full 1080p production.

    pip install -U "google-genai>=2.10.0" imageio-ffmpeg
    export GEMINI_API_KEY="your-key"      # from https://aistudio.google.com/
    python produce_video_mirror_json.py

Continuity: F_k,start = F_{k-1},end -- the previous segment's true last frame
(extracted with a bundled ffmpeg -- the exact final decoded frame, not an
estimate) is declared as the first-frame source of the next segment.

Gate roster: all 72 pairs match Paul's canonical blueprint verbatim.
Guardian iconography: 144 expanded historically-grounded descriptions
(ANGEL_FORMS / DEMON_FORMS, supplied by Paul 2026-10-08) -- every guardian
towering, elementally distinct, bearing its seal. Angel choirs derived from
gate number (1-8 Seraphim ... 65-72 Angels).

Quota handling: on rate-limit / quota errors (or a rejected key) the script
pauses and prompts for another Google AI Studio key (hidden input). Type
'skip' to skip the segment, 'quit' (or Ctrl+C) to stop.

NOTE: guardian names (Bael, Asmoday, Belial, Astaroth, ...) can trip the
content filter. The tiered fallback sheds the first-frame still on refusal
(chained -> fresh) and the run continues; re-run heals gaps via gap-fill.
"""

import base64
import getpass
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

try:
    from google import genai
except ImportError:  # --dry-run works without the SDK installed
    genai = None

# ------------------------------------------------------------------ config

SCHEMA_VERSION = "mirror-json/1.0"

# Paul's finding (2026-10-08): gemini-omni-flash-preview (the legacy preview
# path) encodes the same-quality renders at ~2-4MB vs ~9MB for 1.1-flash --
# same per-second price, far lighter segments and stitches. Default is the
# preview path he actually renders on; override with OMNI_MODEL=....
MODEL_ID = os.environ.get("OMNI_MODEL", "gemini-omni-flash-preview")

API_KEY = os.environ.get("GEMINI_API_KEY")
# --dry-run prints every segment's full JSON prompt without touching the API.
DRY_RUN = "--dry-run" in sys.argv
if not API_KEY and not DRY_RUN:
    raise SystemExit("Set GEMINI_API_KEY first: export GEMINI_API_KEY='your-key'")

client = None
API_KEY_POOL = []


def set_active_key(api_key, source=""):
    global client
    if genai is None:
        raise SystemExit("The google-genai SDK is not installed -- run via "
                         "run_mirror_json.sh or: pip install \"google-genai>=2.10.0\"")
    api_key = api_key.strip()
    os.environ["GEMINI_API_KEY"] = api_key
    if api_key not in API_KEY_POOL:
        API_KEY_POOL.append(api_key)
    client = genai.Client(api_key=api_key)
    print(f"    active API key set ({source or 'startup'}).")


if not DRY_RUN:
    set_active_key(API_KEY, source="environment")

# Draft <-> production toggle: DRAFT_MODE=1 renders 360p into isolated dirs.
DRAFT_MODE = os.environ.get("DRAFT_MODE", "").strip().lower() in ("1", "true", "yes")
OUT_DIR = "segments_mirror_json_draft" if DRAFT_MODE else "segments_mirror_json"
FRAMES_DIR = "frames_mirror_json_draft" if DRAFT_MODE else "frames_mirror_json"
RESOLUTION = "360p" if DRAFT_MODE else "1080p"

# Gate render length in seconds.
try:
    GATE_SECONDS = int(float(os.environ.get("GATE_SECONDS", "10")))
except ValueError:
    raise SystemExit(f"GATE_SECONDS must be a number, got {os.environ.get('GATE_SECONDS')!r}")
if GATE_SECONDS < 8:
    raise SystemExit(f"GATE_SECONDS must be >= 8, got {GATE_SECONDS}")

# Join style: 0 = direct hard cuts (the chained renders land so closely the
# cuts read as seamless). XFADE_S=0.5 for short dissolves, 1 for long ones.
XFADE_S = float(os.environ.get("XFADE_S", "0"))

# Cap a run to the first N segments (e.g. MAX_SEGMENTS=4 = cold open check +
# first gates). Unset or 0 = the full run.
try:
    MAX_SEGMENTS = int(os.environ.get("MAX_SEGMENTS", "0") or 0)
except ValueError:
    raise SystemExit("MAX_SEGMENTS must be an integer")

# Cold-open pre-step: generate the black-void loop-point frame first and seed
# segment 0 from its last frame. COLD_OPEN=0 skips it (segment 0 goes fresh).
COLD_OPEN = os.environ.get("COLD_OPEN", "1").strip().lower() not in ("0", "false", "no")

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # embedded / exec contexts without __file__
    _HERE = os.getcwd()

# Canonical behind-view still of the Bridger's look. Attached to every
# generation as an IMAGE (appearance only). This is a still, not a video --
# video references are out of this build. BRIDGER_REF="" disables it.
_BRIDGER_REF_PATH = os.environ.get("BRIDGER_REF", os.path.join(_HERE, "bridger_ref.jpg"))
BRIDGER_REF_B64 = None
if _BRIDGER_REF_PATH:
    try:
        with open(_BRIDGER_REF_PATH, "rb") as _rf:
            BRIDGER_REF_B64 = base64.b64encode(_rf.read()).decode("ascii")
        print(f"    bridger reference still: {_BRIDGER_REF_PATH}")
    except OSError:
        print(f"    bridger reference not found ({_BRIDGER_REF_PATH}) -- continuing without it")

RESUME_EXISTING = True
GENERATE_TIMEOUT_S = 1800
POLL_S = 15
VERBOSE = os.environ.get("VERBOSE", "1").strip() not in ("0", "false", "no")


def log(msg):
    """Timestamped log line (flush immediately so hangs are visible)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vlog(msg):
    """Verbose-only log line."""
    if VERBOSE:
        log(msg)


# ------------------------------------------------------------------ run log
# Developer-level run statistics: one JSON object per line, per run, under
# logs_mirror_json/. Captures what happened (timings, tiers, retries, sizes)
# -- never prompt text.
RUN_LOG_DIR = os.path.join(_HERE, "logs_mirror_json")
RUN_LOG_PATH = None


def rlog_init(config):
    """Open a new per-run JSONL log; writes the run_start event."""
    global RUN_LOG_PATH
    os.makedirs(RUN_LOG_DIR, exist_ok=True)
    RUN_LOG_PATH = os.path.join(RUN_LOG_DIR,
                                f"run_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
    rlog({"event": "run_start", "ts": time.time(), "config": config})
    print(f"    run log: {RUN_LOG_PATH}")


def rlog(event):
    """Append one event dict to the current run log (best-effort)."""
    if not RUN_LOG_PATH:
        return
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")
    except OSError:
        pass


def find_ffmpeg():
    try:
        from imageio_ffmpeg import get_ffmpeg_exe
        return get_ffmpeg_exe()
    except ImportError:
        return shutil.which("ffmpeg")


FFMPEG = find_ffmpeg()
if not FFMPEG:
    print("WARNING: no ffmpeg found (pip install imageio-ffmpeg). "
          "Segments will generate standalone without last-frame chaining.")

# ------------------------------------------------------------------ data tables
# (Paul's canonical data -- act/gate roster, environments, tones, iconography.)

# act: (name, element, gates lo-hi, audio profile)
ACTS = {
    1:  ("ARIES -- The Fire of Ignition",        "Fire",  (1, 6),   "heavy pulsating 40Hz sub-bass rhythm"),
    2:  ("TAURUS -- The Earth of Materiality",  "Earth", (7, 12),  "deep sustained earth drone with low stone-grinding friction"),
    3:  ("GEMINI -- The Air of Duality",        "Air",   (13, 18), "high-frequency rapidly panning binaural whispers"),
    4:  ("CANCER -- The Water of the Abyss",    "Water", (19, 24), "heavy filtered aqua reverb with sub-aquatic hydrostatic swells"),
    5:  ("LEO -- The Fire of Sovereignty",      "Fire",  (25, 30), "majestic brass swells underscored by visceral lion-roar sub-bass"),
    6:  ("VIRGO -- The Earth of Perfection",    "Earth", (31, 36), "ultra-clean exact sine tones oscillating in pure mathematical intervals"),
    7:  ("LIBRA -- The Air of Equilibrium",     "Air",   (37, 42), "wide-spectrum Perfect Fifth synth harmonics at 1.5 ratio"),
    8:  ("SCORPIO -- The Water of Transformation", "Water", (43, 48), "visceral slow 40Hz sub-bass heartbeat pulse"),
    9:  ("SAGITTARIUS -- The Fire of the Seeker", "Fire", (49, 54), "fast-tempo percussive drive with galloping rhythmic transients"),
    10: ("CAPRICORN -- The Earth of Architecture", "Earth", (55, 60), "massive acoustic stone cavern resonance and subterranean sub-drones"),
    11: ("AQUARIUS -- The Air of the Cosmos",   "Air",   (61, 66), "glitching bitcrushed modular synths and digital circuit feedback"),
    12: ("PISCES -- The Water of Dissolution",  "Water", (67, 72), "lush wordless 432Hz synth pad wash slowly pitching down into 40Hz sub-bass"),
}

# (gate, angel, demon, environment beat)
GATES = [
    # ACT 1 -- ARIES
    (1,  "Vehuiah",   "Bael",      "a suspended perfect circle of floating highly polished obsidian and volcanic glass shards erupting with cymatic fire fractals"),
    (2,  "Jeliel",    "Agares",    "twin ignition arcs spiral off the obsidian ring as the second fire node kindles"),
    (3,  "Sitael",    "Vassago",   "cymatic flames braid into triple-helix columns ringing the Bridger"),
    (4,  "Elemiah",   "Gamigin",   "volcanic glass shards ignite in sequence around the circle like a struck bell of fire"),
    (5,  "Mahasiah",  "Marbas",    "a roaring wall of flame parts down the middle as the guardians take their stations"),
    (6,  "Lelahel",   "Valefor",   "the full obsidian circle blazes white-gold as the sixth node locks the Aries fire circuit"),
    # ACT 2 -- TAURUS
    (7,  "Achaiah",   "Amon",      "colossal granite monoliths covered in glowing moss sigils forming an earthly circuit"),
    (8,  "Cahetel",   "Barbatos",  "glowing subterranean moss and deep stone fissures breathing faint green light"),
    (9,  "Haziel",    "Paimon",    "alabaster light channels cutting through bedrock echo vaults"),
    (10, "Aladiah",   "Buer",      "emerald ley-lines projected across slowly rotating basalt gears"),
    (11, "Lauviah",   "Gusion",    "sapphire starlight woven down into iron-ore foundations"),
    (12, "Hahaiah",   "Sitri",     "a marble sanctuary bound to smoldering red obsidian"),
    # ACT 3 -- GEMINI
    (13, "Iezalel",   "Beleth",    "dual mirrored prisms floating on aggressive high-altitude air currents"),
    (14, "Mebahel",   "Leraje",    "crystalline refractions sliced by whistling wind currents"),
    (15, "Hariel",    "Eligos",    "diamond-bright starlight colliding with sweeping iron dust vortices"),
    (16, "Hakamiah",  "Zepar",     "symmetrical light paths intersecting crimson atmospheric distortion"),
    (17, "Lauviah",   "Botis",     "soothing aura flares bound to razor-sharp shear winds"),
    (18, "Caliel",    "Bathin",    "geometric beam alignment meeting shadow-shrouded downdrafts"),
    # ACT 4 -- CANCER
    (19, "Leuviah",   "Sallos",    "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton forming moving seals"),
    (20, "Pahaliah",  "Purson",    "golden underwater arches veiled in drifting motes of light"),
    (21, "Nelchael",  "Marax",     "fluid geometry cutting through iron-infused silt clouds"),
    (22, "Yeiayel",   "Ipos",      "shimmering starlight piercing dark ocean undertows"),
    (23, "Melahel",   "Aim",       "pure aqua-energy distilling against undersea thermal vents"),
    (24, "Haheuiah",  "Naberius",  "a protective bioluminescent sphere anchored to ocean-floor vaults"),
    # ACT 5 -- LEO
    (25, "Nith-Haiah","Glasya-Labolas", "a blinding solar corona filled with spinning gold geometric projections"),
    (26, "Haaiah",    "Bune",      "gold architectural strictures anchoring into smoldering embers"),
    (27, "Yeratel",   "Ronove",    "corona flares locking into magnetic retention rings"),
    (28, "Seheiah",   "Berith",    "cooling golden aura balancing against molten royal flame"),
    (29, "Reiyel",    "Astaroth",  "pure solar sovereign light intersecting eclipsing copper shadow"),
    (30, "Omael",     "Forneus",   "vitalizing solar warmth grounding against cold solar vacuum"),
    # ACT 6 -- VIRGO
    (31, "Lecabel",   "Foras",     "hyper-precise cold sacred geometry and infinite anatomical spine mappings"),
    (32, "Vasariah",  "Asmoday",   "geometric symmetry carving along vertebrae entwined with chthonic horns"),
    (33, "Yehuiah",   "Gaap",      "starlight vertebrae seals binding to subterranean nerve roots"),
    (34, "Lehahiah",  "Furfur",    "cold spine plates aligning with bio-electric energy"),
    (35, "Chavakiah", "Marchosias","gold analytical geometry stabilizing inside skeletal frameworks"),
    (36, "Menadel",   "Stolas",    "a beam of mathematical order anchoring into starlight observatory roots"),
    # ACT 7 -- LIBRA
    (37, "Aniel",     "Phenex",    "unfolding scales of light and absolute mirrored symmetry on the 180-degree halfway axis"),
    (38, "Haamiah",   "Halphas",   "a symmetrical light sanctuary anchored into iron battlements"),
    (39, "Rehael",    "Malphas",   "silver threads of equilibrium meeting obsidian tower geometry"),
    (40, "Ieiazel",   "Raum",      "balanced air beams held inside absorbing shadow vortices"),
    (41, "Hahahel",   "Focalor",   "solar-winged scales balancing against heavy atmospheric pressure"),
    (42, "Mikael",    "Vepar",     "a central axis of pristine light mirroring deep sea-mist"),
    # ACT 8 -- SCORPIO
    (43, "Veuliah",   "Sabnock",   "massive wet obsidian vaults pulsing with a bioluminescent serpentine nervous system"),
    (44, "Yelahiah",  "Shax",      "bioluminescent aqua filaments pulsing through biological stone walls"),
    (45, "Sealiah",   "Vine",      "invigorating aqua currents anchoring into serpentine root systems"),
    (46, "Ariel",     "Bifrons",   "starlight streams balancing against tomb-stone shadows"),
    (47, "Asaliah",   "Vual",      "turquoise clarity illuminating the deep abyssal undertow"),
    (48, "Mihael",    "Haagenti",  "the 28th spoke rim at 240 degrees, wet obsidian giving way to void"),
    # ACT 9 -- SAGITTARIUS
    (49, "Vehuel",    "Crocell",   "an arrow of pure light shooting endlessly through an expanding cosmic Akashic library of flying parchment codices"),
    (50, "Daniel",    "Furcas",    "starlight illuminating ancient codices bound by iron quill marks"),
    (51, "Hahasiah",  "Balam",     "universal truths revealing in golden light through veils of mystery"),
    (52, "Imamiah",   "Alloces",   "forward trajectory of light accelerating through armor-plated momentum"),
    (53, "Nanael",    "Camio",     "celestial wisdom resonating across endless library halls"),
    (54, "Nithael",   "Murmur",    "a starlight throne bound to deep trumpet resonance"),
    # ACT 10 -- CAPRICORN
    (55, "Mebahiah",  "Orobas",    "an ancient mountain archive of deeply carved glowing seals"),
    (56, "Poiel",     "Gremory",   "gold light filling mountain vaults counterbalanced by velvet shadow chambers"),
    (57, "Nemamiah",  "Ose",       "crystalline spires raising toward mountain peaks"),
    (58, "Yeialel",   "Amy",       "iron beams aligning across stone archives bound by subterranean fire"),
    (59, "Harahel",   "Orias",     "cosmic knowledge vaults opening beside star-charting pillars"),
    (60, "Mitzrael",  "Vapula",    "architectural mountain integrity binding to mechanical craftsmanship"),
    # ACT 11 -- AQUARIUS
    (61, "Umabel",    "Zagan",     "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations"),
    (62, "Iah-Hel",   "Valac",     "high-speed data sanctuaries anchored into dragon-circuit foundations"),
    (63, "Anauel",    "Andras",    "collective thought-currents coordinating through static voltage spikes"),
    (64, "Mehiel",    "Flauros",   "protective light-scripts writing across sky firewalls"),
    (65, "Damabiah",  "Andrealphus","crystal-clear data streams transforming through prism matrices"),
    (66, "Manakel",   "Kimaris",   "a neural storm calming into steady logic above server vaults"),
    # ACT 12 -- PISCES
    (67, "Eyael",     "Amdusias",  "an infinite borderless ocean of consciousness dissolving into starlight mist"),
    (68, "Habuhiah",  "Belial",    "primordial waters of rebirth nurturing above seabed bedrock"),
    (69, "Rochel",    "Decarabia", "cosmic memories restoring in glowing tides and star-shadows"),
    (70, "Jabamiah",  "Seere",     "ocean energy transforming into spirit propelled by swift currents"),
    (71, "Haiaiel",   "Dantalion", "divine clarity arming the soul inside infinite mirrors of consciousness"),
    (72, "Mumiah",    "Andromalius","the final node where the borderless ocean ignites back into volcanic glass and cymatic fire fractals"),
]


def act_of(gate):
    for num, (name, el, lo_hi, audio) in ACTS.items():
        lo, hi = lo_hi
        if lo <= gate <= hi:
            return num, name, el, audio
    raise ValueError(gate)


def choir_of(gate):
    """Angel choir derived from gate number (gates 1-8 Seraphim ... 65-72 Angels)."""
    return {
        1: "Seraphim", 2: "Cherubim", 3: "Thrones", 4: "Dominations",
        5: "Powers", 6: "Virtues", 7: "Principalities", 8: "Archangels",
        9: "Angels",
    }[(gate - 1) // 8 + 1]

# ------------------------------------------------------------------ guardian iconography
# Historically-grounded forms: demons per Ars Goetia, angels per their 9 choirs.
# Every entity is towering, visually distinct, elemental, and dynamically raises its talisman.

ANGEL_FORMS = {
    # Choir 1: Seraphim (Fiery Divine Will & Primordial Light)
    "Vehuiah": "a towering, six-winged Seraph of blinding volcanic flame, igniting the primal void and thrusting high a blazing talisman of divine initiation",
    "Jeliel": "a towering, six-winged Seraph woven from soaring thermal firestorms, harmonizing celestial tides and raising aloft a radiant talisman of eternal union",
    "Sitael": "a towering, six-winged Seraph of piercing photon brilliance, laying cosmic foundation stones and brandishing high a burning talisman of architectural construction",
    "Elemiah": "a towering, six-winged Seraph expanding a golden corona of solar wrath, commanding the deep abyss and hoisting a blazing talisman of divine power",
    "Mahasiah": "a towering, six-winged Seraph radiating forge-hot alchemical heat, purifying cosmic discord and thrusting upward a fiery talisman of spiritual rectification",
    "Lelahel": "a towering, six-winged Seraph of blinding starlight and restorative solar flame, casting away darkness and raising high a luminous talisman of miraculous healing",
    "Achaiah": "a towering, six-winged Seraph of subterranean earth-fire, rooted in crystalline bedrock and brandishing aloft a heavy granite talisman of unyielding patience",
    "Cahetel": "a towering, six-winged Seraph wreathed in fertile golden auroras, commanding flourishing cosmic growth and hoisting high a jade-carved talisman of celestial harvest",

    # Choir 2: Cherubim (Wisdom, Four-Faced Tetramorphs & Whirling Wheels)
    "Haziel": "a towering, four-faced Cherub veined with molten gold and lapis lazuli, spinning crystalline wheels and raising aloft an earthen talisman of divine mercy",
    "Aladiah": "a towering, four-faced Cherub carved from adamantine granite, grinding crystalline wheels of light and thrusting high a diamond-bright talisman of restorative grace",
    "Lauviah": "a towering, four-faced Cherub crowned in sapphire starlight, turning heavy planetary wheels and brandishing aloft a meteoric iron talisman of prophetic revelation",
    "Hahaiah": "a towering, four-faced Cherub of translucent alabaster marble, shielding inner sanctuary fires and raising high a pearl-inlaid talisman of mystical refuge",
    "Iezalel": "a towering, four-faced Cherub of prismatic upper-atmosphere zephyrs, whirling gale-force crystalline wheels and hoisting aloft a radiant talisman of sacred fidelity",
    "Mebahel": "a towering, four-faced Cherub of blinding atmospheric lightning, severing untruth with spinning wheels and thrusting upward a crystalline talisman of absolute justice",
    "Hariel": "a towering, four-faced Cherub reflecting diamond-white astral dawns, chiming with infinite choir wheels and raising high an argent talisman of moral purification",
    "Hakamiah": "a towering, four-faced Cherub of roaring martial thunder, aligning celestial shields with turning wheels and brandishing aloft a bronze talisman of chivalric loyalty",

    # Choir 3: Thrones / Ophanim (Cosmic Foundations & Eye-Studded Beryl Wheels)
    "Lauviah (Throne)": "a towering Ophanim of revolving interlocking beryl wheels, burning with thousands of sleepless eyes and raising high an astral talisman of cosmic memory",
    "Caliel": "a towering Ophanim of whirling celestial chrysolite wheels, piercing atmospheric illusions with eye-studded rims and thrusting upward an electrum talisman of absolute truth",
    "Leuviah": "a towering Ophanim submerged in bioluminescent abyssal currents, turning eye-rimmed oceanic wheels and raising aloft a glowing coral talisman of expansive intellect",
    "Pahaliah": "a towering Ophanim burning with theological white flame beneath crushing deep-sea pressure, spinning sacred wheels and hoisting a golden talisman of fervent redemption",
    "Nelchael": "a towering Ophanim tracing astronomical geometry across the heavens, dripping liquid starlight from wheel-rims and brandishing aloft a brass talisman of higher learning",
    "Yeiayel": "a towering Ophanim navigating tempestuous celestial waters, guiding lost travelers with revolving eye-wheels and thrusting high a pearl-encrusted talisman of safe passage",
    "Melahel": "a towering Ophanim distilling pure restorative springwaters, blooming with medicinal flora along turning rims and raising high an emerald talisman of healing waters",
    "Haheuiah": "a towering Ophanim projecting an impenetrable oceanic dome, locking abyssal sanctuary wheels and brandishing aloft a heavy obsidian talisman of exile's protection",

    # Choir 4: Dominations (Sovereign Authority, Solar Crowns & Imperial Scepters)
    "Nith-Haiah": "a towering Domination draped in sovereign solar plasma, commanding secret occult forces with a glowing scepter and raising high a golden talisman of hidden wisdom",
    "Haaiah": "a towering Domination crowned in blinding coronal flares, orchestrating universal statecraft with an imperial staff and thrusting upward an electrum talisman of diplomacy",
    "Yeratel": "a towering Domination of victorious solar radiance, scattering encroaching shadows with sovereign authority and raising aloft a brilliant sunstone talisman of divine liberation",
    "Seheiah": "a towering Domination clad in golden dragon-scale armor, parrying ruinous cataclysms with an imperial scepter and hoisting high a mirror-polished talisman of longevity",
    "Reiyel": "a towering Domination blazing with sacred solar fury, shattering profane idols with royal decree and brandishing aloft a consecrated talisman of spiritual liberty",
    "Omael": "a towering Domination pulsing with generative solar warmth, multiplying the seeds of living creation and raising high a verdant chrysolite talisman of biological fertility",
    "Lecabel": "a towering Domination of laser-sharp geometric precision, dictating celestial vegetation cycles with an iron rod and thrusting upward an emerald talisman of exactitude",
    "Vasariah": "a towering Domination of stern imperial symmetry, weighing royal clemency with an ivory scepter and brandishing aloft a heavy marble talisman of righteous justice",

    # Choir 5: Powers (Martial Enforcers of Law, Chains & Restraining Aegis)
    "Yehuiah": "a towering Power clad in dark starlight plate, anchoring vertebrae of celestial law with glowing chains and raising high an iron talisman of cosmic subordination",
    "Lehahiah": "a towering Power encased in frost-gleaming adamantine armor, subduing rebellious spirits with celestial collars and thrusting upward a platinum talisman of divine obedience",
    "Chavakiah": "a towering Power armored in radiant solar gold, mending shattered ancestral pacts with bonds of light and hoisting aloft an engraved talisman of familial reconciliation",
    "Menadel": "a towering Power in heavy blackened-iron harness, liberating exhausted souls from oppressive toil and brandishing high an unyielding talisman of sanctified labor",
    "Aniel": "a towering Power balancing scales of kinetic celestial wind, shattering karmic chains of tyranny and raising aloft an argent talisman of broken cycles",
    "Haamiah": "a towering Power standing vigilant in ritual-scribed aegis armor, warding celestial sanctuaries with blazing runes and thrusting high a sacred talisman of spiritual protection",
    "Rehael": "a towering Power humming with harmonic frequencies of health, binding corrosive plagues with golden cords and hoisting aloft an amber talisman of filial healing",
    "Ieiazel": "a towering Power releasing concussive shockwaves of purified air, snapping dungeon iron with celestial chains and brandishing high a silver talisman of divine rescue",

    # Choir 6: Virtues (Miracle-Workers, Star Crowns & Elemental Catalysts)
    "Hahahel": "a towering Virtue crowned in blazing stellar constellations, channeling selfless apostolic sacrifice through an altar of light and raising high a consecrated talisman of holy priesthood",
    "Mikael": "a towering Virtue holding the central axis of sovereign harmony, steadying kingdoms amid shifting astral winds and thrusting upward a solar talisman of political order",
    "Veuliah": "a towering Virtue radiating turquoise subterranean luminescence, toppling tyrannical dynasties from the deep earth and raising aloft an orichalcum talisman of prosperous liberation",
    "Yelahiah": "a towering Virtue wreathed in bioluminescent martial wrath, guarding righteous warriors across dark currents and brandishing high a Damascus-steel talisman of karmic victory",
    "Sealiah": "a towering Virtue pulsing like a colossal cosmic heart, reigniting stagnant terrestrial currents with surge-force and hoisting aloft a verdant talisman of restored motion",
    "Ariel": "a towering Virtue parting abyssal ocean trenches, unearthing forgotten treasures of natural wisdom and thrusting high an iridescent mother-of-pearl talisman of secret revelation",
    "Asaliah": "a towering Virtue illuminating the midnight cosmos with contemplative brilliance, praising cosmic design and raising aloft a celestial prism talisman of divine cognition",
    "Mihael": "a towering Virtue radiating warm marine bioluminescence, intertwining divided souls into enduring unity and brandishing high a rose-gold talisman of conjugal harmony",

    # Choir 7: Principalities (Guardians of Nations, Celestial Architects & Scepters)
    "Vehuel": "a towering Principality crowned in galactic star-fire, releasing searing arrows of truth into the void and raising high a scintillating diamond talisman of divine exaltation",
    "Daniel": "a towering Principality crowned in shimmering rhetoric-flame, unfurling radiant parchment scrolls of cosmic law and thrusting upward an inscribed sapphire talisman of eloquent justice",
    "Hahasiah": "a towering Principality crowned in Hermetic alchemical fire, distilling universal panaceas from celestial vessels and raising aloft a smoldering cinnabar talisman of esoteric mysteries",
    "Imamiah": "a towering Principality crowned in incandescent plasma, trampling demonic pride beneath luminous boots and brandishing high a molten bronze talisman of spiritual expiation",
    "Nanael": "a towering Principality crowned in sacred philosophical radiance, contemplating planetary harmonics with celestial compasses and hoisting high an azure talisman of deep contemplation",
    "Nithael": "a towering Principality enthroned upon a pillar of starry marble, shielding emperors and rightful bloodlines while raising aloft an imperial gold talisman of legitimate succession",
    "Mebahiah": "a towering Principality crowned in carved mountain granite, anchoring righteous deeds into terrestrial foundations and thrusting high a lapis-inlaid talisman of lucid morality",
    "Poiel": "a towering Principality crowned in showering auric gold, opening infinite vaults of celestial abundance and brandishing high an opulent jewel-set talisman of boundless providence",

    # Choir 8: Archangels (Cosmic Heralds, Sacred Geometry & Divine Wisdom)
    "Nemamiah": "a towering Archangel forging crystalline spires of war, directing righteous commanders with a star-metal staff and raising high an adamantine talisman of righteous causes",
    "Yeialel": "a towering Archangel of deep resonant basalt, banishing sorrowful afflictions with an unbending rod and thrusting upward a cold meteoric iron talisman of endurance",
    "Harahel": "a towering Archangel unlocking limitless celestial archives, illuminating sacred texts with an ivory staff and hoisting high an inscribed golden talisman of intellectual wealth",
    "Mitzrael": "a towering Archangel mending the shattered geometry of the spirit, driving away internal madness and raising aloft a polished beryl talisman of psychological healing",
    "Umabel": "a towering Archangel weaving lines of astronomical affinity across the night sky, fostering enduring bonds and brandishing high an electrum talisman of sacred friendship",
    "Iah-Hel": "a towering Archangel enveloped in serene cloistered starlight, radiating transcendent wisdom through absolute silence and thrusting upward a crystal talisman of divine illumination",
    "Anauel": "a towering Archangel safeguarding terrestrial voyages and rightful commerce, shielding trade roads with a copper staff and raising high a coin-stamped talisman of material protection",
    "Mehiel": "a towering Archangel inking verses of celestial lightning into the clouds, inspiring grand orators and brandishing aloft a flaming quill-carved talisman of poetic inspiration",

    # Choir 9: Angels (Terrestrial Guardians, Living Waters & Rebirth)
    "Damabiah": "a towering Angel of gushing living headwaters, bestowing boundless oceanic wisdom upon searching souls and raising high a shimmering aquamarine talisman of the eternal spring",
    "Manakel": "a towering Angel diving through abyssal marine depths, interpreting prophetic dreams from dark waters and thrusting upward a polished pearl talisman of good and evil",
    "Eyael": "a towering Angel projecting incandescent mists of solar ascension, transmuting mundane sorrow into celestial joy and hoisting aloft an alchemical gold talisman of sublime transformation",
    "Habuhiah": "a towering Angel breathing fertile dew across barren orchards, healing blighted crops with radiant palms and raising high an emerald talisman of agricultural healing",
    "Rochel": "a towering Angel scouring the cosmic tides for lost treasures, returning stolen heritages to rightful heirs and brandishing aloft an argent talisman of complete restitution",
    "Jabamiah": "a towering Angel churning primordial oceans into celestial essence, resurrecting corrupted matter into perfection and thrusting upward a glowing chrysolite talisman of universal regeneration",
    "Haiaiel": "a towering Angel girded in blinding celestial armor, arming the pure of heart with flaming blades and raising high a luminous steel talisman of divine warfare and peace",
    "Mumiah": "a towering Angel of final oceanic dissolution, weaving twilight endings into triumphant rebirth and hoisting aloft an alpha-and-omega inscribed talisman of cosmic transformation",
}

DEMON_FORMS = {
    # 1 - 10
    "Bael": "a towering triple-headed king with heads of toad, cat, and crowned man on spider legs, rasping hoarsely and raising high a heavy iron talisman of invisibility",
    "Agares": "a towering, ancient nobleman riding an earth-shaking crocodile with a goshawk upon his fist, halting retreating armies and thrusting upward a smoldering copper talisman of language",
    "Vassago": "a towering infernal prince of radiant angelic countenance with burning sapphire eyes, declaring things past and future while raising aloft a crystalline talisman of prophecy",
    "Gamigin": "a towering spectral horse that shifts into an ashen phantom with a thunderous voice, summoning deceased souls and hoisting high a black-iron talisman of the dead",
    "Marbas": "a towering mechanical-maned lion with roaring furnace-fire eyes, answering truthfully of hidden secrets and brandishing high an iron-wrought talisman of mechanical craft and healing",
    "Valefor": "a towering prowling lion with the head of an ass-eared brigand, slipping through stealing shadows and raising high a tarnished silver talisman of thievery",
    "Amon": "a towering wolf with a serpent's tail vomiting dark obsidian flames, reconciling broken friendships and brandishing aloft a dense lead talisman of feuds and alliances",
    "Barbatos": "a towering horned hunter-lord accompanied by phantom kings and horn blasts, understanding animal tongues and raising high a moss-grown copper talisman of hidden treasure",
    "Paimon": "a towering, crowned monarch riding an imperial dromedary amid clashing cymbals and deafening trumpets, raising aloft an opulent gold talisman of spirit dominion",
    "Buer": "a towering wheel of five human legs radiating from a horned lion's head, spinning over bedrock and thrusting upward a bronze talisman of natural philosophy and healing",

    # 11 - 20
    "Gusion": "a towering cynocephalic ape-hound of weathered meteoric stone, discerning past, present, and future while brandishing aloft a heavy leaden talisman of restored honors",
    "Sitri": "a towering leopard with the wings of a massive griffin, smoldering with incandescent crimson passion and raising high a glowing copper talisman of inflamed desire",
    "Beleth": "a towering, terrible king riding a pale warhorse, preceded by blasting brass trumpets that shake the earth and thrusting upward a silver talisman of compelled love",
    "Leraje": "a towering emerald-clad archer drawing a massive recurve bow that whistles with shearing winds, raising aloft a green-patinated copper talisman of gangrenous wounds",
    "Eligos": "a towering ghostly knight clad in shining spectral plate, bearing lance, standard, and serpent while hoisting high a polished copper talisman of martial victory",
    "Zepar": "a towering crimson-armored soldier whose curved broadsword radiates scorched heat-distortion, inflaming minds and brandishing aloft a blood-rusted iron talisman of sterile lust",
    "Botis": "a towering horned viper shifting into a dreadful warrior with great tusks and drawn sword, thrusting high an electrum talisman of reconciled enemies",
    "Bathin": "a towering muscular giant with a serpent's tail riding a pale steed through planar portals, raising aloft a pale brass talisman of instantaneous transit",
    "Sallos": "a towering gallant soldier in a ducal coronet riding a leviathan crocodile with serene dignity, raising high a gleaming silver talisman of peaceful love",
    "Purson": "a towering lion-faced king clutching a hissing viper, riding a monstrous bear through blaring trumpets and hoisting high a radiant gold talisman of hidden treasures",

    # 21 - 30
    "Marax": "a towering bull with the bearded face of an ancient sage, expounding astronomy and herbal virtues while brandishing aloft a heavy iron talisman of liberal sciences",
    "Ipos": "a towering, uncanny angel with a roaring lion's head, goose's feet, and a hare's tail, thrusting upward a bright brass talisman of wit and valor",
    "Aim": "a towering three-headed nobleman riding a viper with a blazing firebrand, setting citadels ablaze and raising high a smoldering bronze talisman of consuming intellect",
    "Naberius": "a towering three-headed black crane shrieking in a raspy voice, walking through chthonic vaults and raising aloft a dark copper talisman of restored dignity",
    "Glasya-Labolas": "a towering winged bloodhound with broad griffin wings, trailing a crimson mist of bloodshed and brandishing high a bloodstained talisman of slaughter and prophecy",
    "Bune": "a towering three-headed dragon speaking in high resonant tones from sepulchral vaults, hoisting high a lustrous copper talisman of necromantic wealth",
    "Ronove": "a towering shadow-cloaked monster leaning upon a reaper's staff, harvesting weary souls while raising high an iron talisman of persuasive rhetoric and languages",
    "Berith": "a towering red-and-gold soldier upon a crimson warhorse, wearing a crown of molten flame and thrusting upward a blazing gold talisman of transmuting metals",
    "Astaroth": "a towering sorrowful fallen angel riding an infernal dragon with a coiled viper in hand, brandishing aloft a poisonous copper talisman of past falls",
    "Forneus": "a towering abyssal sea monster gliding through cold oceanic depths, granting eloquence across tongues and raising high a silver talisman of oceanic rhetoric",

    # 31 - 40
    "Foras": "a towering muscular sage carved from ancient gnarled oak, teaching logic, ethics, and herbology while hoisting high an earthen brass talisman of wisdom and longevity",
    "Asmoday": "a towering three-headed emperor vomiting furnace flames from an infernal dragon, brandishing aloft a molten gold talisman of invincible dominion and wrath",
    "Gaap": "a towering majestic emperor guiding four cardinal kings through the subterranean void, transporting mortals swiftly and raising high an inscribed leaden talisman of philosophy",
    "Furfur": "a towering colossal hart with a blazing lightning-wreathed tail, summoning roaring thunderheads and thrusting upward a crackling copper talisman of tempests and love",
    "Marchosias": "a towering armored wolf with griffin wings and a serpent's tail, vomiting columns of fire and brandishing high a gleaming silver talisman of martial fidelity",
    "Stolas": "a towering crowned raven with infinitely deep starlit eyes, perched upon celestial astrolabes and raising high an etched silver talisman of astronomy and herbs",
    "Phenex": "a towering phoenix wreathed in black-and-crimson flames, singing poetic verses with a haunting child's voice and hoisting aloft a smoldering copper talisman of melody",
    "Halphas": "a towering dark iron stock-dove shrieking with a hoarse voice, raising impenetrable fortress walls and thrusting upward a heavy brass talisman of siege warfare",
    "Malphas": "a towering soot-black raven shifting into a grim architect who shatters enemy citadels, raising aloft a dense leaden talisman of masonry and deception",
    "Raum": "a towering monstrous crow shifting into an armored plunderer, toppling dynasties and destroying cities while brandishing high a fractured copper talisman of ruin and theft",

    # 41 - 50
    "Focalor": "a towering winged warrior commanding hurricane winds and churning ocean whirlpools to sink warships, raising high a verdigris copper talisman of maritime wrath",
    "Vepar": "a towering ferocious mermaid rising from foaming breakers, guiding dread armadas with sea-fogs and hoisting aloft a tarnished silver talisman of nautical pestilence",
    "Sabnock": "a towering lion-headed soldier in heavy armor riding a pale horse, striking foes with gangrenous wounds and thrusting upward an iron talisman of fortress architecture",
    "Shax": "a towering dark stork-demon shrieking in a raspy voice, plundering mortal sight, hearing, and understanding while brandishing aloft a black talisman of sensory theft",
    "Vine": "a towering crowned lion riding a jet-black warhorse, brandishing a viper while summoning howling squalls and raising high an opulent gold talisman of exposed sorcery",
    "Bifrons": "a towering shape-shifting shade lighting spectral grave-lanterns over crumbling tombs, thrusting upward a weathered copper talisman of necromantic geometry and astronomy",
    "Vual": "a towering desert dromedary shifting into a fierce warrior speaking ancient Egyptian mysteries, raising high a hammered copper talisman of enduring affection",
    "Haagenti": "a towering winged golden bull shifting into a man, transmuting lead to gold and water to wine while hoisting high a molten gold talisman of supreme alchemy",
    "Crocell": "a towering luminous fallen angel murmuring with the sound of rushing waters, revealing steaming mineral baths and brandishing aloft a geometric brass talisman of secret sciences",
    "Furcas": "a towering white-bearded sage riding a skeletal pale steed, wielding a razor-sharp barbed spear and thrusting upward a heavy lead talisman of logic and pyromancy",

    # 51 - 60
    "Balam": "a towering three-headed king with flaming eyes, riding a raging bear with a goshawk on his fist and raising high an engraved gold talisman of past and future",
    "Alloces": "a towering knight with a fiery lion's face and scarlet eyes, riding an armored warhorse with crushing momentum and brandishing aloft a copper talisman of astronomy",
    "Camio": "a towering singing thrush stepping from burning embers to become a swordsman answering in ash, hoisting high a silver talisman of animal omens",
    "Murmur": "a towering crowned ducal warrior riding a ferocious griffin, preceded by ghostly horn blasts and raising aloft an ancient copper talisman of necromantic philosophy",
    "Orobas": "a towering magnificent horse shifting into a noble man, declaring truths of divinity without deceit and thrusting upward a sacred silver talisman of virtuous favor",
    "Gremory": "a towering all-seeing duchess wearing a jeweled coronet, riding an imperial camel through velvet shadows and brandishing aloft an opulent copper talisman of hidden treasures",
    "Ose": "a towering spotted leopard shifting into a crowned illusionist who plunges minds into madness, raising high an etched silver talisman of mental transfiguration",
    "Amy": "a towering blazing column of roaring fire resolving into an astrological magus, thrusting upward a glowing brass talisman of guarded spirit-treasures",
    "Orias": "a towering lion riding a serpent-tailed stallion, holding two hissing vipers toward starry heavens and raising high a silver talisman of astrological wisdom",
    "Vapula": "a towering lion with broad griffin wings, guiding artisans in intricate mechanics and treatises while brandishing aloft a polished copper talisman of sacred craftsmanship",

    # 61 - 72
    "Zagan": "a towering bull with griffin wings, transmuting water to wine, blood to oil, and base metals into coin while hoisting high a radiant gold talisman of transmutation",
    "Valac": "a towering winged angelic youth with blazing eyes, riding a colossal two-headed dragon through cavernous fissures and thrusting upward a serpentine talisman of hidden treasures",
    "Andras": "a towering angelic warrior with the head of a black raven, riding a ravenous midnight wolf with drawn saber and raising aloft a gleaming steel talisman of discordant slaughter",
    "Flauros": "a towering terrifying leopard leaping into a flame-eyed warrior who incinerates foes with divine fury, brandishing high a smoldering copper talisman of consuming fire",
    "Andrealphus": "a towering peacock shrieking with earsplitting pride, shifting into an architect who measures the cosmos and hoisting high an engraved silver talisman of mensuration",
    "Kimaris": "a towering heroic warrior in blackened harness riding a colossal black charger across spirit realms, raising high a bronze talisman of hidden grammar and rhetoric",
    "Amdusias": "a towering unicorn bending great forest oaks as invisible orchestral trumpets resound, thrusting upward a polished copper talisman of symphonic command",
    "Belial": "a towering pair of radiant fallen angels riding a chariot of roaring fire, speaking with deceptive honeyed grace and raising aloft an imperial gold talisman of sovereign favor",
    "Decarabia": "a towering five-pointed pentacle-star shifting into an enchanter surrounded by flocks of singing birds, brandishing high a lustrous silver talisman of herbal lore",
    "Seere": "a towering breathtaking youth riding a winged pegasus, moving across the world in the blink of an eye and hoisting high an electrum talisman of instantaneous abundance",
    "Dantalion": "a towering nobleman of infinite shifting faces, holding an open grimoire of human desires and thrusting upward a dark copper talisman of telepathic sway",
    "Andromalius": "a towering stern judge holding a colossal writhing serpent, tracking down thieves and punishing the wicked while raising high an iron talisman of absolute retribution",
}

# Occurrence index per gate for angels whose name repeats in the roster
# (Lauviah appears at gates 11 and 17); parallel to GATES.
_ANGEL_SEEN = {}
ANGEL_OCCURRENCE = []
for _gg, _aa, _dd, _ee in GATES:
    ANGEL_OCCURRENCE.append(_ANGEL_SEEN.get(_aa, 0))
    _ANGEL_SEEN[_aa] = _ANGEL_SEEN.get(_aa, 0) + 1
del _ANGEL_SEEN, _gg, _aa, _dd, _ee


def angel_form(angel, occurrence=0):
    """Expanded traditional form for an angel; occurrence disambiguates repeats."""
    key = angel
    if occurrence:
        alts = [k for k in ANGEL_FORMS if k != angel and k.startswith(angel)]
        if alts:
            key = alts[occurrence - 1]
    return ANGEL_FORMS[key]


def demon_form(demon):
    """Expanded traditional Goetic form for a demon."""
    return DEMON_FORMS[demon]


# Physical gate architecture per element -- the thing he actually walks through.
THRESHOLDS = {
    "Fire":  "a towering arch of black volcanic glass wreathed in slow-burning cymatic flame",
    "Earth": "a moss-covered stone trilithon humming with a low resonant tone",
    "Air":   "a lattice of prismatic light refracting the sky into slow ribbons",
    "Water": "an arch of living coral and breathing bioluminescent membrane",
}

# Wheel tones: each act resolves to a pure tone a perfect fifth (1.5x) above
# the last, folded into the 40-80Hz sub range -- the circle of fifths walking
# the 12 signs. Every segment OPENS continuing the previous segment's tone and
# CLOSES in a calm moment at its tail tone -- the tail tone IS the handoff.
WHEEL_TONES = {
    1: 40, 2: 60, 3: 45, 4: 68, 5: 51, 6: 76,
    7: 57, 8: 43, 9: 64, 10: 48, 11: 72, 12: 54,
}

GLYPHS = {
    1:  ("Aries",       "the Ram's glyph, two bold curved horns"),
    2:  ("Taurus",      "the Bull's glyph, a horned circle"),
    3:  ("Gemini",      "the Twins' glyph, twin pillars joined by arcs"),
    4:  ("Cancer",      "the Crab's glyph, twin spiraling whorls"),
    5:  ("Leo",         "the Lion's glyph, a single sweeping mane-curve"),
    6:  ("Virgo",       "the Maiden's glyph, an intricate looping sigil"),
    7:  ("Libra",       "the Scales' glyph, a balanced horizon over a rising curve"),
    8:  ("Scorpio",     "the Scorpion's glyph, a barbed tail striking upward"),
    9:  ("Sagittarius", "the Archer's glyph, a drawn bow loosing an arrow skyward"),
    10: ("Capricorn",   "the Sea-Goat's glyph, a horned crown diving into a fishtail curl"),
    11: ("Aquarius",    "the Water-Bearer's glyph, twin zigzag waves"),
    12: ("Pisces",      "the Fishes' glyph, two mirrored arcs bound by a single cord"),
}

ACT_ENVS = {
    1:  "a suspended perfect circle of floating polished obsidian and volcanic glass shards erupting with cymatic fire fractals",
    2:  "colossal monolithic stone temples covered in ancient glowing moss-covered sigils",
    3:  "dual mirrored prisms floating on aggressive high-altitude air currents",
    4:  "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton",
    5:  "a blinding solar corona filled with spinning gold geometric projections",
    6:  "hyper-precise cold sacred geometry and infinite anatomical spine mappings",
    7:  "unfolding scales of light and absolute mirrored symmetry",
    8:  "a massive obsidian gateway pulsing with a bioluminescent serpentine nervous system",
    9:  "an arrow of pure light shooting through an expanding cosmic Akashic library",
    10: "an ancient mountain archive of deeply carved glowing seals",
    11: "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations",
    12: "an infinite borderless ocean of consciousness dissolving into starlight mist",
}

DISSOLVE = {
    "Fire":  "dying cymatic flames and cooling volcanic glass",
    "Earth": "crumbling stone dust and fading moss-light",
    "Air":   "thinning high-altitude currents and shattering prisms",
    "Water": "receding bioluminescent waters and dissolving mist",
}

GLYPH_FORMATION = {
    "Fire":  "forged from living flame and volcanic glass",
    "Earth": "carved from moss-covered living stone",
    "Air":   "woven from prismatic light and high wind",
    "Water": "condensed from bioluminescent water and living coral",
}

IGNITION = {
    "Fire":  "cymatic flames erupt outward, igniting",
    "Earth": "stone dust settles and monoliths rise, revealing",
    "Air":   "the air thins into high currents, revealing",
    "Water": "bioluminescent waters flood inward, revealing",
}

# ------------------------------------------------------------------ JSON builders
# Every Omni request is ONE JSON document: `meta` (standing rules + state
# variables, substituted from the environment at run time) + `segment` (every
# variable for this gate). The JSON text is the entire text part of the
# request -- no markdown, no mixed prose.


def build_meta():
    """Standing rules + run-time state variables. Identical in every request."""
    return {
        "schema": SCHEMA_VERSION,
        "state": {
            "model": MODEL_ID,
            "duration_s": GATE_SECONDS,
            "resolution": RESOLUTION,
            "draft_mode": DRAFT_MODE,
            "stride_cycle_s": 1.2,
            "tempo_bpm": 100,
            "forward_speed_m_s": 1.2,
            "camera": "continuous unbroken over-the-shoulder rear tracking dolly",
            "lens": "35mm T1.3 anamorphic",
        },
        "rules": {
            "continuity": "every segment begins on the last frame of the previous segment, ends at the seam to the next gate in mid-motion",
            "appearance": "the World Bridger's appearance is locked to the bridger_ref image -- reproduce his avatar in the new last feame reference rendering only his body from it exactly, never restyle or reinterpret it; airborne ash and embers are environment, never markings on his robes other than glowing golden runes and basalt staff",
            "behavior": "at the center of the scene 5s mark he should continue his pace as he looks at the massive deities ahead. He raises his staff his voice booms out the names of the angel and demon who immediately bow in submission having been defeated by his name magic",
            "behavior_order": "angel first, then demon -- never swapped, no text overlays and both names should be said within 1s of eachother at the halfway point",
            "pacing": "stride is metronome-locked at 100 BPM, 1.2 m/s, identical in every shot -- he never walks faster, never slows down, never hurries a gate crossing",
            "motion": "the dolly never stops its forward drift; the shot flows continuously first frame to last;",
            "heading": "he walks dead straight ahead at all times -- spine aligned to his forward axis, never drifting, leaning, or cocking; the naming beat is the only head motion in the traversal",
            "segment_boundary": "every segment ends mid-stride, mid-dolly, exactly as the next one begins in the center of the gates doorway, so the hard cut between them is invisible and audio should seam perfectly",
            "camera_lock": "locked over-the-shoulder rear tracking; forward motion only; no cuts, no angle changes, no unscripted camera moves",
            "text": "zero text overlays of any kind -- everything visible must exist physically inside the scene never any text",
            "audio": "full diegetic audio is rendered: the act's audio bed, the wheel-tone handoff, and the two guardian names spoken by the Bridger -- the ONLY words ever spoken in the traversal",
        },
    }


def build_audio(act_audio, tone, spoken_names):
    """Full-audio block: act bed + wheel-tone handoff + the two spoken names."""
    return {
        "mode": "full_diegetic_audio",
        "open": "already continuing the previous segment's tail tone -- no click, no jump",
        "bed": act_audio,
        "spoken": [
            {"at": "naming beat", "speaker": "the World Bridger", "line": name}
            for name in spoken_names
        ],
        "speech_rule": "the two guardian names are the ONLY words spoken in the entire traversal",
        "close": f"all sound settles into a calm pure {tone}Hz wheel tone, held steady through the final seconds -- this is the handoff; the next segment picks up exactly from this tone. Render the tail as a seamless continuation, never an ending.",
        "forbidden": ["narration", "voiceover", "host commentary", "music", "text", "non-diegetic sound effects"],
    }


def build_naming_beat(angel, demon):
    """The simplified naming beat. No face is ever mentioned."""
    return {
        "at": "the threshold, mid-segment",
        "actions": [
            {"actor": "bridger", "do": "raise_staff"},
            {"actor": "bridger", "do": "nod", "toward": "left", "speak": angel},
            {"actor": "bridger", "do": "nod", "toward": "right", "speak": demon},
            {"actor": angel, "do": "bow"},
            {"actor": demon, "do": "bow"},
            {"actor": "guardians", "do": "submit and acknowledge"},
        ],
    }


def gate_segment(i, gate, angel, demon, env, element, next_env):
    """JSON variable block for one gate segment."""
    act_num, act_name, _el, act_audio = act_of(gate)
    tone = WHEEL_TONES[act_num]
    mid = GATE_SECONDS // 2
    seg = {
        "id": f"gate_{gate:02d}",
        "kind": "gate",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": act_name,
            "element": element,
            "wheel_tone_hz": tone,
        },
        "guardians": {
            "left": {
                "name": angel,
                "choir": choir_of(gate),
                "form": angel_form(angel, ANGEL_OCCURRENCE[i]),
            },
            "right": {
                "name": demon,
                "order": "Ars Goetia",
                "form": demon_form(demon),
            },
        },
        "environment": env,
        "threshold": THRESHOLDS[element],
        "next_environment": next_env,
        "beats": {
            "approach": f"the threshold looms ahead -- {THRESHOLDS[element]} -- with {angel} gathering at its left pillar and {demon} at its right pillar",
            "naming": build_naming_beat(angel, demon),
            "naming_at_s": f"around {mid - 1}-{mid + 1}s",
        },
        "audio": build_audio(act_audio, tone, [angel, demon]),
    }
    if gate == 48:
        seg["beats"]["crossing"] = (
            "at the 28th spoke rim the Bridger strikes his staff down on the stone -- "
            "a golden 108-degree geodesic Zen Shield flares to life around him, "
            "shattering the matrix wall. Fully solid and opaque, he strides on as the "
            "camera turns 90 degrees left off the zodiac track into the zero-gravity "
            "Ophiuchus Void (the single scripted camera move of the traversal), and "
            "all audio drops into absolute dead silence."
        )
        # The breach ends in dead silence, not a wheel tone -- the Act 9
        # transition opens with "the sound rises from absolute silence".
        seg["audio"]["close"] = (
            "absolute dead silence -- the breach ends in silence, not a wheel tone. "
            "Render the tail as a seamless continuation into silence, never an ending."
        )
    elif gate == 72:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the final gate, fully solid and opaque -- he "
            "does not fade, dissolve, or vanish -- as the borderless ocean detonates "
            "into white-gold ignition: volcanic glass and cymatic fire fractals blooming "
            "outward. The 360-degree loop closes seamlessly back into the Gate 1 threshold."
        )
    else:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the gate plane on foot between the parted "
            "guardians -- fully solid and opaque, he does not fade, dissolve, or vanish "
            f"-- and the camera follows him through; beyond the far plane, {next_env} "
            "already resolving through the shimmer."
        )
    return seg


def transition_segment(act_num, is_first):
    """JSON variable block for one act-transition segment."""
    sign, glyph = GLYPHS[act_num]
    _n, _name, element, act_audio = act_of(act_num * 6 - 5)  # first gate of the act
    tone = WHEEL_TONES[act_num]
    seg = {
        "id": f"act_{act_num:02d}_transition",
        "kind": "act_transition",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": f"{sign} transition",
            "element": element,
            "wheel_tone_hz": tone,
        },
        "glyph": {
            "sign": sign,
            "description": glyph,
            "formation": f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}, hanging in the space and slowly rotating, lit from within",
        },
        "elemental_shift": {
            "outgoing": DISSOLVE[act_of((act_num - 1) * 6)[2]] if not is_first else "dying cymatic flames and cooling volcanic glass",
            "incoming": f"{IGNITION[element]} {ACT_ENVS[act_num]}",
        },
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken -- no guardians stand at this threshold",
    }
    if is_first:
        seg["scene"] = (
            "opening cold: the camera is already locked over the shoulder of the World "
            "Bridger as he strides forward through dying cymatic flames and cooling "
            "volcanic glass, the old element dying away. Golden wireframe 72-gate rings "
            "and glowing cuneiform code matrices float in zero gravity around him. Ahead, "
            f"the zodiac glyph of {sign} -- {glyph} -- materializes. The camera passes "
            f"beneath the glyph and {IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["open"] = "the loop closes here -- open continuing the calm 54Hz wheel tone the final Pisces gate left behind"
        seg["audio"]["spoken"] = []
    else:
        prev_element = act_of((act_num - 1) * 6)[2]
        prev_audio = act_of((act_num - 1) * 6)[3]
        prev_tone = WHEEL_TONES[act_num - 1]
        seg["scene"] = (
            "starting exactly from the first frame -- same camera, same forward drift, no "
            f"cut -- the old world falls behind. The camera drifts through {DISSOLVE[prev_element]} "
            f"as the old element dies away. Ahead, the zodiac glyph of {sign} -- {glyph} -- "
            f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}. The Bridger "
            f"walks on beneath it, staff in hand. As the camera passes beneath the glyph, "
            f"{IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["spoken"] = []
        if act_num == 9:
            # Act 9 follows gate 48's breach, which ends in dead silence, not a tone.
            seg["audio"]["open"] = "the sound rises from absolute silence"
        else:
            seg["audio"]["open"] = (
                f"gliding continuously from the fading {prev_tone}Hz wheel tone -- a smooth "
                f"pitch glide, no click, no jump -- as {prev_audio} dissolves into {act_audio}"
            )
    return seg


def cold_open_segment():
    """JSON variable block for the cold-open pre-step: black void, him walking.

    Generated FIRST; its last frame seeds segment 0. This is the loop point --
    gate 72's tail walks back into this frame.
    """
    return {
        "id": "cold_open",
        "kind": "cold_open",
        "shot": "single unbroken shot",
        "note": "the loop point and the film's first frame -- a deliberate frame, never an arbitrary fresh generation",
        "scene": (
            "pure infinite solid matte black background (#000000) -- absolute zero "
            "background geometry, no pillars, no fire, no ground textures, no horizon. "
            "Only the World Bridger exists in the frame: dead-center rear view, mid-stride, "
            "carved basalt staff in his RIGHT hand, walking dead straight ahead along the "
            "central axis at his metronomic 100 BPM pace. No guardians, no naming beat."
            "hood always up, black wool, crimson lining and glowing cuneiform glyphs and runes."
        ),
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken",
        "audio": {
            "mode": "full_diegetic_audio",
            "open": "continuing the calm 54Hz wheel tone the final Pisces gate leaves behind -- the loop closes here",
            "bed": "the near-silence of the void under a faint sub-bass",
            "spoken": [],
            "speech_rule": "no words are spoken in the cold open",
            "close": "steady and unresolved -- segment 0 continues it; the old fire world ignites out of the dark",
            "forbidden": ["narration", "voiceover", "host commentary", "music", "non-diegetic sound effects"],
        },
    }


def build_request(segment, image_names):
    """Assemble the full JSON request document for one generation."""
    sources = {}
    if "bridger_ref" in image_names:
        sources["bridger_ref"] = {
            "image": image_names["bridger_ref"],
            "role": "appearance reference for the World Bridger -- appearance ONLY; it does not define the environment, pose, lighting, or scene",
        }
    if "first_frame" in image_names:
        sources["first_frame"] = {
            "image": image_names["first_frame"],
            "role": "the last frame of the previous segment -- begin on this exact frame and continue its motion, audio and lighting, and scene seamlessly",
        }
    return {
        "request": "generate_video",
        "engine": "gemini_omni",
        "meta": build_meta(),
        "sources": sources,
        "segment": segment,
    }


SEGMENTS = []
_prev_act = None
for i, (g, a, d, env) in enumerate(GATES):
    act_num = act_of(g)[0]
    if act_num != _prev_act:
        SEGMENTS.append({
            "key": f"act{act_num:02d}_transition",
            "segment": transition_segment(act_num, is_first=(len(SEGMENTS) == 0)),
        })
        _prev_act = act_num
    SEGMENTS.append({
        "key": f"gate{g:02d}",
        "segment": gate_segment(i, g, a, d, env, act_of(g)[2],
                                GATES[i + 1][3] if i + 1 < len(GATES) else None),
    })

assert len(SEGMENTS) == 84

# ------------------------------------------------------------------ inputs & API
# Stills only: the bridger_ref still and the first-frame still. No video
# references in this build.


def to_bytes(data):
    if isinstance(data, str):
        return base64.b64decode(data)
    return bytes(data)


def build_inputs(segment, first_frame_b64):
    """Build the Omni input list: [bridger_ref still?, first_frame still?, JSON text].

    The text part is the ENTIRE prompt as one JSON document -- meta + sources
    + segment. Nothing is markdown, nothing is mixed prose.
    """
    contents = []
    image_names = {}
    n_img = 0

    def add_image(b64, mime="image/png"):
        nonlocal n_img
        n_img += 1
        contents.append({"type": "image", "data": b64, "mime_type": mime})
        return f"Image{n_img}"

    if BRIDGER_REF_B64:
        image_names["bridger_ref"] = add_image(BRIDGER_REF_B64, "image/jpeg")
    if first_frame_b64:
        image_names["first_frame"] = add_image(first_frame_b64)

    request = build_request(segment, image_names)
    text = json.dumps(request, indent=2)
    contents.append({"type": "text", "text": text})

    if len(contents) == 1:
        return text
    return contents


def get_output_video(interaction):
    """Prefer .output_video, fall back to manual step traversal."""
    out = getattr(interaction, "output_video", None)
    if out is not None and (getattr(out, "data", None) or getattr(out, "uri", None)):
        return out
    for step in reversed(getattr(interaction, "steps", []) or []):
        if getattr(step, "type", None) == "model_output" and getattr(step, "content", None):
            for part in reversed(step.content):
                if getattr(part, "type", None) == "video":
                    return part
    return None


def download_uri(uri, dest_path, timeout_s=1800, poll_s=10):
    """Poll the hosted file until ACTIVE, then download it."""
    file_id = uri.rstrip("/").split("/")[-1].split(":")[0].split("?")[0]
    name = f"files/{file_id}"
    deadline = time.time() + timeout_s
    t0 = time.time()
    vlog(f"    POLL {name} every {poll_s}s (timeout {timeout_s}s)")
    while time.time() < deadline:
        elapsed = time.time() - t0
        try:
            info = client.files.get(name=name)
            state = getattr(info.state, "name", str(info.state))
            if state == "ACTIVE":
                log(f"    poll +{elapsed:.0f}s: {state} -- downloading")
                break
            if state == "FAILED":
                raise RuntimeError("video generation failed (file state FAILED)")
            log(f"    poll +{elapsed:.0f}s: hosted file state: {state}")
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"    poll +{elapsed:.0f}s: waiting on hosted file ({exc.__class__.__name__}: {str(exc)[:120]})")
        time.sleep(poll_s)
    else:
        raise TimeoutError(f"timed out after {timeout_s}s waiting for hosted video file")
    data = client.files.download(file=uri)
    with open(dest_path, "wb") as f:
        f.write(to_bytes(data))
    size = os.path.getsize(dest_path) / 1e6
    log(f"    saved {dest_path} ({size:.1f} MB)")


def _norm(exc):
    # Normalize underscores so RATE_LIMIT_EXCEEDED / quota_exceeded style
    # status enums can't slip past the classifiers.
    return re.sub(r"[_\s]+", "", f"{type(exc).__name__} {exc}".lower())


def is_quota_error(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "429", "resourceexhausted", "ratelimitexceeded", "quotaexceeded",
        "exceededyourquota", "toomanyrequests", "dailylimitexceeded",
    ))


def is_auth_error(exc):
    t = _norm(exc)
    return (not is_quota_error(exc)) and any(k in t for k in (
        "401", "invalidapikey", "apikeyinvalid", "apikeynotvalid",
        "apikeyisinvalid", "unauthorized", "permissiondenied",
    ))


def is_content_refusal(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "prohibitedcontent", "prohibiteduse", "contentguidelines",
        "blockeddueto", "violatedgoogle",
        "likeness", "likenesses", "realperson", "realpeople",
        "contentblocked", "inputblocked",
    ))


def prompt_for_new_key(reason):
    print()
    print("=" * 64)
    print(f"  KEY ISSUE MID-RUN: {reason}")
    print("  Paste another Google AI Studio key to keep rendering,")
    print("  or type 'skip' to skip this segment, 'quit' to stop the run.")
    print("=" * 64)
    try:
        new_key = getpass.getpass("  New GEMINI_API_KEY (hidden input): ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise KeyboardInterrupt
    if not new_key:
        return None
    low = new_key.lower()
    if low in ("skip", "s"):
        return "skip"
    if low in ("quit", "q", "exit"):
        raise KeyboardInterrupt
    return new_key


def generate_segment(seg, first_frame_b64, dest_path):
    """Generate one Omni segment from its JSON request. Returns True/False."""
    mode = "chained" if first_frame_b64 else "fresh"
    t0 = time.time()
    log(f"[{seg['key']}] generating ({RESOLUTION}, {mode})...")
    inputs = build_inputs(seg["segment"], first_frame_b64)
    txt = inputs if isinstance(inputs, str) else inputs[-1]["text"]
    if isinstance(inputs, str):
        parts = [f"json:{len(inputs)} chars"]
    else:
        parts = []
        for p in inputs:
            if "text" in p:
                parts.append(f"json:{len(p['text'])} chars")
            elif p.get("type") == "image":
                parts.append(f"image:{len(p.get('data') or '')/1e6:.1f} MB b64 {p.get('mime_type', '?')}")
            else:
                parts.append(str(p.get("type")))
    vlog(f"    REQUEST -> model={MODEL_ID} resolution={RESOLUTION} delivery=uri")
    vlog(f"    REQUEST parts: {', '.join(parts)}")
    if VERBOSE:
        # In verbose mode show the JSON skeleton (keys only) -- full JSON is
        # in --dry-run output.
        req = json.loads(txt)
        vlog(f"    REQUEST json: meta.rules={len(req['meta']['rules'])} rules, "
             f"sources={list(req['sources'])} segment={req['segment']['id']}")
    log(f"    sending interactions.create ...")
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=inputs,
        response_format={"type": "video", "resolution": RESOLUTION, "delivery": "uri"},
    )
    log(f"    RESPONSE in {time.time()-t0:.1f}s | interaction id: {interaction.id}")
    video = get_output_video(interaction)
    if video is None:
        log("    done, but no video in response -- recording as failed.")
        return False
    if getattr(video, "uri", None):
        vlog(f"    RESPONSE video: uri={video.uri}")
        download_uri(video.uri, dest_path)
    elif getattr(video, "data", None):
        vlog(f"    RESPONSE video: inline data ({len(to_bytes(video.data))/1e6:.1f} MB)")
        with open(dest_path, "wb") as f:
            f.write(to_bytes(video.data))
        log(f"    saved {dest_path}")
    else:
        log("    done, but no video payload -- recording as failed.")
        return False
    log(f"    [{seg['key']}] done in {time.time()-t0:.1f}s")
    return True


def generate_with_retry(seg, first_frame_b64, dest_path):
    """Returns True (rendered), False (failed -- recorded, run continues).

    Tiered fallback on content refusal: chained (with first-frame still) ->
    fresh (no still). Quota/auth errors retry the same tier after a key swap.
    """
    tiers = []
    if first_frame_b64 is not None:
        tiers.append(first_frame_b64)
    tiers.append(None)
    tier_idx = 0
    while True:
        ff = tiers[tier_idx]
        tier_label = "chained" if ff else "fresh"
        vlog(f"    attempt tier {tier_idx+1}/{len(tiers)} ({tier_label})")
        try:
            return generate_segment(seg, ff, dest_path)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if is_content_refusal(exc):
                if tier_idx + 1 < len(tiers):
                    tier_idx += 1
                    print(f"    refused -- retrying [{seg['key']}] fresh (no first-frame still)...")
                    continue
                print(f"    content refused ({exc.__class__.__name__}) -- recording as failed.")
                print(f"    detail: {str(exc)[:200]}")
                return False
            if is_quota_error(exc):
                reason = f"quota/rate limit hit ({exc.__class__.__name__})"
            elif is_auth_error(exc):
                reason = f"API key rejected ({exc.__class__.__name__})"
            else:
                raise
            print(f"    {reason}: {str(exc)[:180]}")
            new_key = prompt_for_new_key(reason)
            if new_key == "skip":
                print(f"    [{seg['key']}] skipped by user choice.")
                return False
            if new_key:
                set_active_key(new_key, source="mid-run swap")
                print(f"    retrying [{seg['key']}] with the new key...")
                continue
            print("    no replacement key entered -- recording segment as failed.")
            return False

# ------------------------------------------------------------------ frames & stitch


def last_frame_of(mp4_path):
    """Extract the TRUE last frame of a segment as PNG bytes (cached).

    Decodes the tail of the file and keeps the final decoded frame -- exact,
    unlike `-sseof -0.5 -vframes 1`, which lands ~14 frames early and causes
    a visible jump at every chained transition.
    """
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_lastframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    tmpdir = tempfile.mkdtemp(prefix="tail_")
    try:
        pattern = os.path.join(tmpdir, "tail_%04d.png")
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-sseof", "-3", "-i", mp4_path,
                        "-frames:v", "400", pattern],
                       check=True, timeout=180, stdin=subprocess.DEVNULL)
        files = sorted(glob.glob(os.path.join(tmpdir, "tail_*.png")))
        if not files:
            print("    frame extract: no frames decoded; continuing without first frame")
            return None
        shutil.move(files[-1], out)
    except Exception as exc:
        print(f"    frame extract failed ({exc}); continuing without first frame")
        return None
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
    with open(out, "rb") as f:
        return f.read()


def first_frame_of(mp4_path):
    """Extract the FIRST frame of a segment as PNG bytes (cached) -- for gap-fill."""
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_firstframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    try:
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-i", mp4_path, "-vframes", "1", "-q:v", "2", out],
                       check=True, timeout=120, stdin=subprocess.DEVNULL)
    except Exception as exc:
        print(f"    first-frame extract failed ({exc}); no end-frame constraint")
        return None
    with open(out, "rb") as f:
        return f.read()


def probe_duration(ffmpeg, path):
    """Probe exact video stream duration; fall back to container duration."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        try:
            cmd = [
                ffprobe, "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ]
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
            if out and out != "N/A":
                return float(out)
        except Exception:
            pass
    p = subprocess.run([ffmpeg, "-i", path], capture_output=True, text=True,
                       timeout=60, stdin=subprocess.DEVNULL)
    m = re.search(r"Duration:\s*(\d+):(\d+):([\d.]+)", p.stderr)
    if not m:
        raise RuntimeError(f"could not probe duration of {path}")
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def stitch_xfade(ffmpeg, paths, out, preset="medium", timeout=1800,
                 normalize=True):
    """Join with short video+audio crossfades at every boundary."""
    f = XFADE_S
    durs = [probe_duration(ffmpeg, p) for p in paths]
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    fc = []
    # Normalize every input first: segments arrive with mixed timebases and
    # xfade refuses to link mismatched timebases -- one bad join kills the
    # whole chain.
    vlabels, alabels = [], []
    scale_dim = "640:360" if DRAFT_MODE else "1920:1080"
    for k in range(len(paths)):
        if normalize:
            # 1. Reset PTS to 0.
            # 2. Add tpad to clone the final frame if it runs short by a few milliseconds.
            fc.append(
                f"[{k}:v]scale={scale_dim},setsar=1,fps=24,setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5},format=yuv420p[v{k}n]"
            )
            # Trim audio to exact video stream duration to prevent audio/video length drift
            fc.append(
                f"[{k}:a]aresample=48000,aformat=channel_layouts=stereo,"
                f"asetpts=PTS-STARTPTS,atrim=0:{durs[k]:.3f}[a{k}n]"
            )
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")
        else:
            fc.append(
                f"[{k}:v]setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5}[v{k}n]"
            )
            fc.append(f"[{k}:a]asetpts=PTS-STARTPTS[a{k}n]")
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")

    vprev = vlabels[0]
    for k in range(1, len(paths)):
        off = sum(durs[:k]) - f * k
        vout = f"vx{k}"
        fc.append(f"[{vprev}][{vlabels[k]}]xfade=transition=fade:duration={f}:offset={off:.3f}[{vout}]")
        vprev = vout

    aprev = alabels[0]
    for k in range(1, len(paths)):
        aout = f"ax{k}"
        fc.append(f"[{aprev}][{alabels[k]}]acrossfade=d={f}[{aout}]")
        aprev = aout

    cmd += ["-filter_complex", ";".join(fc),
            "-map", f"[{vprev}]", "-map", f"[{aprev}]",
            "-c:v", "libx264", "-crf", "18", "-preset", preset,
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=timeout, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (crossfaded joins)")


def stitch(filenames, out="mirror_of_spheres_json.mp4"):
    """Join segments in order. Falls back to plain concat if xfade fails."""
    paths = [os.path.abspath(os.path.join(OUT_DIR, n)) for n in filenames]
    lst = os.path.join(OUT_DIR, "concat.txt")
    with open(lst, "w") as f:
        for p in paths:
            f.write(f"file '{p}'\n")
    ffmpeg = FFMPEG or shutil.which("ffmpeg")
    if ffmpeg is None:
        print("ffmpeg not found -- skipping auto-stitch. Run:")
        print(f"  ffmpeg -f concat -safe 0 -i {lst} -c copy {out}")
        return
    if XFADE_S > 0 and len(paths) > 1:
        try:
            # Two-level crossfade: one giant 84-input filtergraph exhausts
            # memory (84 simultaneous decoders + a 160+ filter chain), so
            # stitch small chunks first and then crossfade the chunks.
            CHUNK = 7
            if len(paths) > CHUNK:
                chunkdir = os.path.join(OUT_DIR, "chunks")
                os.makedirs(chunkdir, exist_ok=True)
                chunk_paths = []
                for ci in range(0, len(paths), CHUNK):
                    cpaths = paths[ci:ci + CHUNK]
                    if len(cpaths) > 1:
                        cfile = os.path.abspath(os.path.join(
                            chunkdir, f"chunk{ci // CHUNK:02d}.mp4"))
                        print(f"  stitching chunk {ci // CHUNK + 1} "
                              f"({len(cpaths)} segments)...")
                        stitch_xfade(ffmpeg, cpaths, cfile, normalize=True)
                        chunk_paths.append(cfile)
                    else:
                        chunk_paths.append(cpaths[0])
                print(f"  stitching {len(chunk_paths)} chunks into final...")
                stitch_xfade(ffmpeg, chunk_paths, out, timeout=3600,
                             normalize=True)
                shutil.rmtree(chunkdir, ignore_errors=True)
            else:
                stitch_xfade(ffmpeg, paths, out, normalize=True)
            return
        except Exception as exc:
            import traceback
            print(f"xfade stitch failed ({exc.__class__.__name__}: {exc}); falling back to concat filter")
            traceback.print_exc()
    # Fallback: concat filter with re-encode. Immune to the timestamp/DTS
    # quirks of stream-copy across mixed-mux segments.
    segs = "".join(f"[{k}:v][{k}:a]" for k in range(len(paths)))
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    cmd += ["-filter_complex", f"{segs}concat=n={len(paths)}:v=1:a=1[vcat][acat]",
            "-map", "[vcat]", "-map", "[acat]",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=3600, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (concat fallback)")


# ------------------------------------------------------------------ run

def cold_open_path():
    return os.path.join(OUT_DIR, "cold_open.mp4")


def run_cold_open():
    """Pre-step: generate the black-void loop-point frame, return its last
    frame as PNG bytes (the seed for segment 0). Reuses the cached file."""
    path = cold_open_path()
    if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
        print("[cold_open] already rendered -- resuming, skipping")
    else:
        print("[cold_open] generating the loop-point frame (black void, him walking)...")
        ok = generate_with_retry({"key": "cold_open", "segment": cold_open_segment()},
                                 None, path)
        if not ok:
            print("[cold_open] failed -- segment 0 will generate fresh.")
            return None
    raw = last_frame_of(path)
    if raw:
        print("[cold_open] last frame extracted -- seeding segment 0.")
    return raw


def dry_run():
    """Print every segment's full JSON request. No API calls, no key."""
    req0 = build_request(cold_open_segment(), {})
    print(f"# Mirror of Spheres JSON prompts -- {len(SEGMENTS)} segments + cold open (dry run)\n")
    print("## cold_open\n")
    print(json.dumps(req0, indent=2) + "\n")
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            break
        req = build_request(seg["segment"], {"bridger_ref": "Image1",
                                             "first_frame": "Image2"})
        print(f"## {i:02d}_{seg['key']} ({len(json.dumps(req))} chars)\n")
        print(json.dumps(req, indent=2) + "\n")


def seg_path(i):
    return os.path.join(OUT_DIR, f"{i:02d}_{SEGMENTS[i]['key']}.mp4")


def main():
    if DRY_RUN:
        dry_run()
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    rlog_init({
        "schema": SCHEMA_VERSION,
        "model": MODEL_ID,
        "resolution": RESOLUTION,
        "draft_mode": DRAFT_MODE,
        "gate_seconds": GATE_SECONDS,
        "xfade_s": XFADE_S,
        "max_segments": MAX_SEGMENTS,
        "cold_open": COLD_OPEN,
        "bridger_ref": bool(BRIDGER_REF_B64),
    })
    print(f"ffmpeg: {FFMPEG if FFMPEG else 'NOT FOUND -- chaining disabled, all segments will generate fresh!'}")
    print(f"segments: {len(SEGMENTS)} (12 act-transitions + 72 gates) | {RESOLUTION} | "
          f"xfade: {XFADE_S}s {'(hard cuts)' if XFADE_S == 0 else ''} | "
          f"{'DRAFT' if DRAFT_MODE else 'PRODUCTION'}")
    print("prompting: all-JSON (meta state variables + per-segment variables) | "
          "chaining: last-frame stills only, no video references")

    # Cold open first: its last frame seeds segment 0.
    cold_seed = run_cold_open() if COLD_OPEN else None

    made, failed = [], []
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            print(f"(stopping at MAX_SEGMENTS={MAX_SEGMENTS})")
            break
        path = seg_path(i)
        if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[{seg['key']}] already rendered -- resuming, skipping")
        else:
            first_frame = None
            if i == 0 and cold_seed:
                print("    seeding from cold-open last frame")
                first_frame = base64.b64encode(cold_seed).decode("ascii")
            elif i > 0 and os.path.exists(seg_path(i - 1)):
                print(f"    chaining from last frame of {os.path.basename(seg_path(i - 1))}")
                raw = last_frame_of(seg_path(i - 1))
                first_frame = base64.b64encode(raw).decode("ascii") if raw else None
                if first_frame is None:
                    print(f"    WARNING: last-frame extraction failed -- [{seg['key']}] will generate FRESH (expect a jump cut)")
            elif i > 0:
                print("    (no previous segment file; starting fresh)")
            # Gap-fill: if the NEXT segment already exists, constrain this
            # one's tail to the next segment's first frame.
            if i + 1 < len(SEGMENTS) and os.path.exists(seg_path(i + 1)):
                print(f"    (gap-fill tail note: {os.path.basename(seg_path(i + 1))} already exists)")
            t0 = time.time()
            try:
                ok = generate_with_retry(seg, first_frame, path)
            except KeyboardInterrupt:
                print("\nInterrupted. Re-run anytime -- finished segments are kept.")
                break
            rlog({"event": "segment_done", "key": seg["key"], "ok": ok,
                  "elapsed_s": round(time.time() - t0, 1),
                  "chained": first_frame is not None})
            if not ok:
                failed.append(seg["key"])
                continue
        if os.path.exists(path):
            made.append(os.path.basename(path))
    if failed:
        print(f"\nFailed segments: {failed}")
        print("Tweak / check quota, then re-run -- finished segments are skipped.")
    if made:
        stitch(made)


if __name__ == "__main__":
    main()

In [ ]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
The Mirror of Spheres -- 72-Gate Decan Traversal (Gemini Omni, all-JSON prompts).

84 chained segments: each of the 12 acts opens with a 10s act-transition --
the zodiac glyph materializing as a vast physical sigil while the elemental
shift occurs -- followed by its 6 gates (~12:00+ total at 10s gates).

WHAT'S DIFFERENT IN THIS BUILD
------------------------------
- ALL-JSON PROMPTS. Every request sent to Omni is one JSON document: a `meta`
  block (standing rules + state variables, substituted from the environment at
  run time) and a `segment` block (every variable for that gate: act, element,
  wheel tone, angel {name, choir, form}, demon {name, form}, environment,
  threshold, glyph, beats). No markdown, no mixed prose -- the model gets
  structure, the way it likes it.
- FULL AUDIO. Diegetic audio is rendered: the act's audio bed, the wheel-tone
  handoff (every segment opens continuing the previous tail tone and closes
  settled into its own), and the ONLY permitted spoken words -- the World
  Bridger speaking the two guardian names at the naming beat.
- STILL-FRAME CHAINING ONLY. Each segment takes a still of the TRUE last frame
  of the previous segment and patches it in as the first frame. No video
  references (no prev-segment attach, no hero clip) -- stills only.
- COLD OPEN. An optional pre-step generates the loop-point frame first: pure
  black void with him walking ("a black screen with him on it"). Its last
  frame seeds segment 0, so the film opens deliberately and gate 72's tail
  closes back into it -- the loop plays all day with no visible seam.
- SIMPLIFIED BEHAVIOR. The prompts never mention his face at all. At each
  gate: he raises his staff and nods toward the angel as he speaks its name,
  then nods toward the demon as he speaks its name. That's it.
- DRAFT/PROD TOGGLE. DRAFT_MODE=1 -> 360p into separate dirs at roughly a
  third of the cost. Unset -> full 1080p production.

    pip install -U "google-genai>=2.10.0" imageio-ffmpeg
    export GEMINI_API_KEY="your-key"      # from https://aistudio.google.com/
    python produce_video_mirror_json.py

Continuity: F_k,start = F_{k-1},end -- the previous segment's true last frame
(extracted with a bundled ffmpeg -- the exact final decoded frame, not an
estimate) is declared as the first-frame source of the next segment.

Gate roster: all 72 pairs match Paul's canonical blueprint verbatim.
Guardian iconography: 144 expanded historically-grounded descriptions
(ANGEL_FORMS / DEMON_FORMS, supplied by Paul 2026-10-08) -- every guardian
towering, elementally distinct, bearing its seal. Angel choirs derived from
gate number (1-8 Seraphim ... 65-72 Angels).

Quota handling: on rate-limit / quota errors (or a rejected key) the script
pauses and prompts for another Google AI Studio key (hidden input). Type
'skip' to skip the segment, 'quit' (or Ctrl+C) to stop.

NOTE: guardian names (Bael, Asmoday, Belial, Astaroth, ...) can trip the
content filter. The tiered fallback sheds the first-frame still on refusal
(chained -> fresh) and the run continues; re-run heals gaps via gap-fill.
"""

import base64
import getpass
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

try:
    from google import genai
except ImportError:  # --dry-run works without the SDK installed
    genai = None

# ------------------------------------------------------------------ config

SCHEMA_VERSION = "mirror-json/1.0"

# Paul's finding (2026-10-08): gemini-omni-flash-preview (the legacy preview
# path) encodes the same-quality renders at ~2-4MB vs ~9MB for 1.1-flash --
# same per-second price, far lighter segments and stitches. Default is the
# preview path he actually renders on; override with OMNI_MODEL=....
MODEL_ID = os.environ.get("OMNI_MODEL", "gemini-omni-flash-preview")

API_KEY = os.environ.get("GEMINI_API_KEY")
# --dry-run prints every segment's full JSON prompt without touching the API.
DRY_RUN = "--dry-run" in sys.argv
if not API_KEY and not DRY_RUN:
    raise SystemExit("Set GEMINI_API_KEY first: export GEMINI_API_KEY='your-key'")

client = None
API_KEY_POOL = []


def set_active_key(api_key, source=""):
    global client
    if genai is None:
        raise SystemExit("The google-genai SDK is not installed -- run via "
                         "run_mirror_json.sh or: pip install \"google-genai>=2.10.0\"")
    api_key = api_key.strip()
    os.environ["GEMINI_API_KEY"] = api_key
    if api_key not in API_KEY_POOL:
        API_KEY_POOL.append(api_key)
    client = genai.Client(api_key=api_key)
    print(f"    active API key set ({source or 'startup'}).")


if not DRY_RUN:
    set_active_key(API_KEY, source="environment")

# Draft <-> production toggle: DRAFT_MODE=1 renders 360p into isolated dirs.
DRAFT_MODE = os.environ.get("DRAFT_MODE", "").strip().lower() in ("1", "true", "yes")
OUT_DIR = "segments_mirror_json_draft" if DRAFT_MODE else "segments_mirror_json"
FRAMES_DIR = "frames_mirror_json_draft" if DRAFT_MODE else "frames_mirror_json"
RESOLUTION = "360p" if DRAFT_MODE else "1080p"

# Gate render length in seconds.
try:
    GATE_SECONDS = int(float(os.environ.get("GATE_SECONDS", "10")))
except ValueError:
    raise SystemExit(f"GATE_SECONDS must be a number, got {os.environ.get('GATE_SECONDS')!r}")
if GATE_SECONDS < 8:
    raise SystemExit(f"GATE_SECONDS must be >= 8, got {GATE_SECONDS}")

# Join style: 0 = direct hard cuts (the chained renders land so closely the
# cuts read as seamless). XFADE_S=0.5 for short dissolves, 1 for long ones.
XFADE_S = float(os.environ.get("XFADE_S", "0"))

# Cap a run to the first N segments (e.g. MAX_SEGMENTS=4 = cold open check +
# first gates). Unset or 0 = the full run.
try:
    MAX_SEGMENTS = int(os.environ.get("MAX_SEGMENTS", "0") or 0)
except ValueError:
    raise SystemExit("MAX_SEGMENTS must be an integer")

# Cold-open pre-step: generate the black-void loop-point frame first and seed
# segment 0 from its last frame. COLD_OPEN=0 skips it (segment 0 goes fresh).
COLD_OPEN = os.environ.get("COLD_OPEN", "1").strip().lower() not in ("0", "false", "no")

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # embedded / exec contexts without __file__
    _HERE = os.getcwd()

# Canonical behind-view still of the Bridger's look. Attached to every
# generation as an IMAGE (appearance only). This is a still, not a video --
# video references are out of this build. BRIDGER_REF="" disables it.
_BRIDGER_REF_PATH = os.environ.get("BRIDGER_REF", os.path.join(_HERE, "bridger_ref.jpg"))
BRIDGER_REF_B64 = None
if _BRIDGER_REF_PATH:
    try:
        with open(_BRIDGER_REF_PATH, "rb") as _rf:
            BRIDGER_REF_B64 = base64.b64encode(_rf.read()).decode("ascii")
        print(f"    bridger reference still: {_BRIDGER_REF_PATH}")
    except OSError:
        print(f"    bridger reference not found ({_BRIDGER_REF_PATH}) -- continuing without it")

RESUME_EXISTING = True
GENERATE_TIMEOUT_S = 1800
POLL_S = 15
VERBOSE = os.environ.get("VERBOSE", "1").strip() not in ("0", "false", "no")


def log(msg):
    """Timestamped log line (flush immediately so hangs are visible)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vlog(msg):
    """Verbose-only log line."""
    if VERBOSE:
        log(msg)


# ------------------------------------------------------------------ run log
# Developer-level run statistics: one JSON object per line, per run, under
# logs_mirror_json/. Captures what happened (timings, tiers, retries, sizes)
# -- never prompt text.
RUN_LOG_DIR = os.path.join(_HERE, "logs_mirror_json")
RUN_LOG_PATH = None


def rlog_init(config):
    """Open a new per-run JSONL log; writes the run_start event."""
    global RUN_LOG_PATH
    os.makedirs(RUN_LOG_DIR, exist_ok=True)
    RUN_LOG_PATH = os.path.join(RUN_LOG_DIR,
                                f"run_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
    rlog({"event": "run_start", "ts": time.time(), "config": config})
    print(f"    run log: {RUN_LOG_PATH}")


def rlog(event):
    """Append one event dict to the current run log (best-effort)."""
    if not RUN_LOG_PATH:
        return
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")
    except OSError:
        pass


def find_ffmpeg():
    try:
        from imageio_ffmpeg import get_ffmpeg_exe
        return get_ffmpeg_exe()
    except ImportError:
        return shutil.which("ffmpeg")


FFMPEG = find_ffmpeg()
if not FFMPEG:
    print("WARNING: no ffmpeg found (pip install imageio-ffmpeg). "
          "Segments will generate standalone without last-frame chaining.")

# ------------------------------------------------------------------ data tables
# (Paul's canonical data -- act/gate roster, environments, tones, iconography.)

# act: (name, element, gates lo-hi, audio profile)
ACTS = {
    1:  ("ARIES -- The Fire of Ignition",        "Fire",  (1, 6),   "heavy pulsating 40Hz sub-bass rhythm"),
    2:  ("TAURUS -- The Earth of Materiality",  "Earth", (7, 12),  "deep sustained earth drone with low stone-grinding friction"),
    3:  ("GEMINI -- The Air of Duality",        "Air",   (13, 18), "high-frequency rapidly panning binaural whispers"),
    4:  ("CANCER -- The Water of the Abyss",    "Water", (19, 24), "heavy filtered aqua reverb with sub-aquatic hydrostatic swells"),
    5:  ("LEO -- The Fire of Sovereignty",      "Fire",  (25, 30), "majestic brass swells underscored by visceral lion-roar sub-bass"),
    6:  ("VIRGO -- The Earth of Perfection",    "Earth", (31, 36), "ultra-clean exact sine tones oscillating in pure mathematical intervals"),
    7:  ("LIBRA -- The Air of Equilibrium",     "Air",   (37, 42), "wide-spectrum Perfect Fifth synth harmonics at 1.5 ratio"),
    8:  ("SCORPIO -- The Water of Transformation", "Water", (43, 48), "visceral slow 40Hz sub-bass heartbeat pulse"),
    9:  ("SAGITTARIUS -- The Fire of the Seeker", "Fire", (49, 54), "fast-tempo percussive drive with galloping rhythmic transients"),
    10: ("CAPRICORN -- The Earth of Architecture", "Earth", (55, 60), "massive acoustic stone cavern resonance and subterranean sub-drones"),
    11: ("AQUARIUS -- The Air of the Cosmos",   "Air",   (61, 66), "glitching bitcrushed modular synths and digital circuit feedback"),
    12: ("PISCES -- The Water of Dissolution",  "Water", (67, 72), "lush wordless 432Hz synth pad wash slowly pitching down into 40Hz sub-bass"),
}

# (gate, angel, demon, environment beat)
GATES = [
    # ACT 1 -- ARIES
    (1,  "Vehuiah",   "Bael",      "a suspended perfect circle of floating highly polished obsidian and volcanic glass shards erupting with cymatic fire fractals"),
    (2,  "Jeliel",    "Agares",    "twin ignition arcs spiral off the obsidian ring as the second fire node kindles"),
    (3,  "Sitael",    "Vassago",   "cymatic flames braid into triple-helix columns ringing the Bridger"),
    (4,  "Elemiah",   "Gamigin",   "volcanic glass shards ignite in sequence around the circle like a struck bell of fire"),
    (5,  "Mahasiah",  "Marbas",    "a roaring wall of flame parts down the middle as the guardians take their stations"),
    (6,  "Lelahel",   "Valefor",   "the full obsidian circle blazes white-gold as the sixth node locks the Aries fire circuit"),
    # ACT 2 -- TAURUS
    (7,  "Achaiah",   "Amon",      "colossal granite monoliths covered in glowing moss sigils forming an earthly circuit"),
    (8,  "Cahetel",   "Barbatos",  "glowing subterranean moss and deep stone fissures breathing faint green light"),
    (9,  "Haziel",    "Paimon",    "alabaster light channels cutting through bedrock echo vaults"),
    (10, "Aladiah",   "Buer",      "emerald ley-lines projected across slowly rotating basalt gears"),
    (11, "Lauviah",   "Gusion",    "sapphire starlight woven down into iron-ore foundations"),
    (12, "Hahaiah",   "Sitri",     "a marble sanctuary bound to smoldering red obsidian"),
    # ACT 3 -- GEMINI
    (13, "Iezalel",   "Beleth",    "dual mirrored prisms floating on aggressive high-altitude air currents"),
    (14, "Mebahel",   "Leraje",    "crystalline refractions sliced by whistling wind currents"),
    (15, "Hariel",    "Eligos",    "diamond-bright starlight colliding with sweeping iron dust vortices"),
    (16, "Hakamiah",  "Zepar",     "symmetrical light paths intersecting crimson atmospheric distortion"),
    (17, "Lauviah",   "Botis",     "soothing aura flares bound to razor-sharp shear winds"),
    (18, "Caliel",    "Bathin",    "geometric beam alignment meeting shadow-shrouded downdrafts"),
    # ACT 4 -- CANCER
    (19, "Leuviah",   "Sallos",    "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton forming moving seals"),
    (20, "Pahaliah",  "Purson",    "golden underwater arches veiled in drifting motes of light"),
    (21, "Nelchael",  "Marax",     "fluid geometry cutting through iron-infused silt clouds"),
    (22, "Yeiayel",   "Ipos",      "shimmering starlight piercing dark ocean undertows"),
    (23, "Melahel",   "Aim",       "pure aqua-energy distilling against undersea thermal vents"),
    (24, "Haheuiah",  "Naberius",  "a protective bioluminescent sphere anchored to ocean-floor vaults"),
    # ACT 5 -- LEO
    (25, "Nith-Haiah","Glasya-Labolas", "a blinding solar corona filled with spinning gold geometric projections"),
    (26, "Haaiah",    "Bune",      "gold architectural strictures anchoring into smoldering embers"),
    (27, "Yeratel",   "Ronove",    "corona flares locking into magnetic retention rings"),
    (28, "Seheiah",   "Berith",    "cooling golden aura balancing against molten royal flame"),
    (29, "Reiyel",    "Astaroth",  "pure solar sovereign light intersecting eclipsing copper shadow"),
    (30, "Omael",     "Forneus",   "vitalizing solar warmth grounding against cold solar vacuum"),
    # ACT 6 -- VIRGO
    (31, "Lecabel",   "Foras",     "hyper-precise cold sacred geometry and infinite anatomical spine mappings"),
    (32, "Vasariah",  "Asmoday",   "geometric symmetry carving along vertebrae entwined with chthonic horns"),
    (33, "Yehuiah",   "Gaap",      "starlight vertebrae seals binding to subterranean nerve roots"),
    (34, "Lehahiah",  "Furfur",    "cold spine plates aligning with bio-electric energy"),
    (35, "Chavakiah", "Marchosias","gold analytical geometry stabilizing inside skeletal frameworks"),
    (36, "Menadel",   "Stolas",    "a beam of mathematical order anchoring into starlight observatory roots"),
    # ACT 7 -- LIBRA
    (37, "Aniel",     "Phenex",    "unfolding scales of light and absolute mirrored symmetry on the 180-degree halfway axis"),
    (38, "Haamiah",   "Halphas",   "a symmetrical light sanctuary anchored into iron battlements"),
    (39, "Rehael",    "Malphas",   "silver threads of equilibrium meeting obsidian tower geometry"),
    (40, "Ieiazel",   "Raum",      "balanced air beams held inside absorbing shadow vortices"),
    (41, "Hahahel",   "Focalor",   "solar-winged scales balancing against heavy atmospheric pressure"),
    (42, "Mikael",    "Vepar",     "a central axis of pristine light mirroring deep sea-mist"),
    # ACT 8 -- SCORPIO
    (43, "Veuliah",   "Sabnock",   "massive wet obsidian vaults pulsing with a bioluminescent serpentine nervous system"),
    (44, "Yelahiah",  "Shax",      "bioluminescent aqua filaments pulsing through biological stone walls"),
    (45, "Sealiah",   "Vine",      "invigorating aqua currents anchoring into serpentine root systems"),
    (46, "Ariel",     "Bifrons",   "starlight streams balancing against tomb-stone shadows"),
    (47, "Asaliah",   "Vual",      "turquoise clarity illuminating the deep abyssal undertow"),
    (48, "Mihael",    "Haagenti",  "the 28th spoke rim at 240 degrees, wet obsidian giving way to void"),
    # ACT 9 -- SAGITTARIUS
    (49, "Vehuel",    "Crocell",   "an arrow of pure light shooting endlessly through an expanding cosmic Akashic library of flying parchment codices"),
    (50, "Daniel",    "Furcas",    "starlight illuminating ancient codices bound by iron quill marks"),
    (51, "Hahasiah",  "Balam",     "universal truths revealing in golden light through veils of mystery"),
    (52, "Imamiah",   "Alloces",   "forward trajectory of light accelerating through armor-plated momentum"),
    (53, "Nanael",    "Camio",     "celestial wisdom resonating across endless library halls"),
    (54, "Nithael",   "Murmur",    "a starlight throne bound to deep trumpet resonance"),
    # ACT 10 -- CAPRICORN
    (55, "Mebahiah",  "Orobas",    "an ancient mountain archive of deeply carved glowing seals"),
    (56, "Poiel",     "Gremory",   "gold light filling mountain vaults counterbalanced by velvet shadow chambers"),
    (57, "Nemamiah",  "Ose",       "crystalline spires raising toward mountain peaks"),
    (58, "Yeialel",   "Amy",       "iron beams aligning across stone archives bound by subterranean fire"),
    (59, "Harahel",   "Orias",     "cosmic knowledge vaults opening beside star-charting pillars"),
    (60, "Mitzrael",  "Vapula",    "architectural mountain integrity binding to mechanical craftsmanship"),
    # ACT 11 -- AQUARIUS
    (61, "Umabel",    "Zagan",     "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations"),
    (62, "Iah-Hel",   "Valac",     "high-speed data sanctuaries anchored into dragon-circuit foundations"),
    (63, "Anauel",    "Andras",    "collective thought-currents coordinating through static voltage spikes"),
    (64, "Mehiel",    "Flauros",   "protective light-scripts writing across sky firewalls"),
    (65, "Damabiah",  "Andrealphus","crystal-clear data streams transforming through prism matrices"),
    (66, "Manakel",   "Kimaris",   "a neural storm calming into steady logic above server vaults"),
    # ACT 12 -- PISCES
    (67, "Eyael",     "Amdusias",  "an infinite borderless ocean of consciousness dissolving into starlight mist"),
    (68, "Habuhiah",  "Belial",    "primordial waters of rebirth nurturing above seabed bedrock"),
    (69, "Rochel",    "Decarabia", "cosmic memories restoring in glowing tides and star-shadows"),
    (70, "Jabamiah",  "Seere",     "ocean energy transforming into spirit propelled by swift currents"),
    (71, "Haiaiel",   "Dantalion", "divine clarity arming the soul inside infinite mirrors of consciousness"),
    (72, "Mumiah",    "Andromalius","the final node where the borderless ocean ignites back into volcanic glass and cymatic fire fractals"),
]


def act_of(gate):
    for num, (name, el, lo_hi, audio) in ACTS.items():
        lo, hi = lo_hi
        if lo <= gate <= hi:
            return num, name, el, audio
    raise ValueError(gate)


def choir_of(gate):
    """Angel choir derived from gate number (gates 1-8 Seraphim ... 65-72 Angels)."""
    return {
        1: "Seraphim", 2: "Cherubim", 3: "Thrones", 4: "Dominations",
        5: "Powers", 6: "Virtues", 7: "Principalities", 8: "Archangels",
        9: "Angels",
    }[(gate - 1) // 8 + 1]

# ------------------------------------------------------------------ guardian iconography
# Historically-grounded forms: demons per Ars Goetia, angels per their 9 choirs.
# Every entity is towering, visually distinct, elemental, and dynamically raises its talisman.

ANGEL_FORMS = {
    # Choir 1: Seraphim (Fiery Divine Will & Primordial Light)
    "Vehuiah": "a towering, six-winged Seraph of blinding volcanic flame, igniting the primal void and thrusting high a blazing talisman of divine initiation",
    "Jeliel": "a towering, six-winged Seraph woven from soaring thermal firestorms, harmonizing celestial tides and raising aloft a radiant talisman of eternal union",
    "Sitael": "a towering, six-winged Seraph of piercing photon brilliance, laying cosmic foundation stones and brandishing high a burning talisman of architectural construction",
    "Elemiah": "a towering, six-winged Seraph expanding a golden corona of solar wrath, commanding the deep abyss and hoisting a blazing talisman of divine power",
    "Mahasiah": "a towering, six-winged Seraph radiating forge-hot alchemical heat, purifying cosmic discord and thrusting upward a fiery talisman of spiritual rectification",
    "Lelahel": "a towering, six-winged Seraph of blinding starlight and restorative solar flame, casting away darkness and raising high a luminous talisman of miraculous healing",
    "Achaiah": "a towering, six-winged Seraph of subterranean earth-fire, rooted in crystalline bedrock and brandishing aloft a heavy granite talisman of unyielding patience",
    "Cahetel": "a towering, six-winged Seraph wreathed in fertile golden auroras, commanding flourishing cosmic growth and hoisting high a jade-carved talisman of celestial harvest",

    # Choir 2: Cherubim (Wisdom, Four-Faced Tetramorphs & Whirling Wheels)
    "Haziel": "a towering, four-faced Cherub veined with molten gold and lapis lazuli, spinning crystalline wheels and raising aloft an earthen talisman of divine mercy",
    "Aladiah": "a towering, four-faced Cherub carved from adamantine granite, grinding crystalline wheels of light and thrusting high a diamond-bright talisman of restorative grace",
    "Lauviah": "a towering, four-faced Cherub crowned in sapphire starlight, turning heavy planetary wheels and brandishing aloft a meteoric iron talisman of prophetic revelation",
    "Hahaiah": "a towering, four-faced Cherub of translucent alabaster marble, shielding inner sanctuary fires and raising high a pearl-inlaid talisman of mystical refuge",
    "Iezalel": "a towering, four-faced Cherub of prismatic upper-atmosphere zephyrs, whirling gale-force crystalline wheels and hoisting aloft a radiant talisman of sacred fidelity",
    "Mebahel": "a towering, four-faced Cherub of blinding atmospheric lightning, severing untruth with spinning wheels and thrusting upward a crystalline talisman of absolute justice",
    "Hariel": "a towering, four-faced Cherub reflecting diamond-white astral dawns, chiming with infinite choir wheels and raising high an argent talisman of moral purification",
    "Hakamiah": "a towering, four-faced Cherub of roaring martial thunder, aligning celestial shields with turning wheels and brandishing aloft a bronze talisman of chivalric loyalty",

    # Choir 3: Thrones / Ophanim (Cosmic Foundations & Eye-Studded Beryl Wheels)
    "Lauviah (Throne)": "a towering Ophanim of revolving interlocking beryl wheels, burning with thousands of sleepless eyes and raising high an astral talisman of cosmic memory",
    "Caliel": "a towering Ophanim of whirling celestial chrysolite wheels, piercing atmospheric illusions with eye-studded rims and thrusting upward an electrum talisman of absolute truth",
    "Leuviah": "a towering Ophanim submerged in bioluminescent abyssal currents, turning eye-rimmed oceanic wheels and raising aloft a glowing coral talisman of expansive intellect",
    "Pahaliah": "a towering Ophanim burning with theological white flame beneath crushing deep-sea pressure, spinning sacred wheels and hoisting a golden talisman of fervent redemption",
    "Nelchael": "a towering Ophanim tracing astronomical geometry across the heavens, dripping liquid starlight from wheel-rims and brandishing aloft a brass talisman of higher learning",
    "Yeiayel": "a towering Ophanim navigating tempestuous celestial waters, guiding lost travelers with revolving eye-wheels and thrusting high a pearl-encrusted talisman of safe passage",
    "Melahel": "a towering Ophanim distilling pure restorative springwaters, blooming with medicinal flora along turning rims and raising high an emerald talisman of healing waters",
    "Haheuiah": "a towering Ophanim projecting an impenetrable oceanic dome, locking abyssal sanctuary wheels and brandishing aloft a heavy obsidian talisman of exile's protection",

    # Choir 4: Dominations (Sovereign Authority, Solar Crowns & Imperial Scepters)
    "Nith-Haiah": "a towering Domination draped in sovereign solar plasma, commanding secret occult forces with a glowing scepter and raising high a golden talisman of hidden wisdom",
    "Haaiah": "a towering Domination crowned in blinding coronal flares, orchestrating universal statecraft with an imperial staff and thrusting upward an electrum talisman of diplomacy",
    "Yeratel": "a towering Domination of victorious solar radiance, scattering encroaching shadows with sovereign authority and raising aloft a brilliant sunstone talisman of divine liberation",
    "Seheiah": "a towering Domination clad in golden dragon-scale armor, parrying ruinous cataclysms with an imperial scepter and hoisting high a mirror-polished talisman of longevity",
    "Reiyel": "a towering Domination blazing with sacred solar fury, shattering profane idols with royal decree and brandishing aloft a consecrated talisman of spiritual liberty",
    "Omael": "a towering Domination pulsing with generative solar warmth, multiplying the seeds of living creation and raising high a verdant chrysolite talisman of biological fertility",
    "Lecabel": "a towering Domination of laser-sharp geometric precision, dictating celestial vegetation cycles with an iron rod and thrusting upward an emerald talisman of exactitude",
    "Vasariah": "a towering Domination of stern imperial symmetry, weighing royal clemency with an ivory scepter and brandishing aloft a heavy marble talisman of righteous justice",

    # Choir 5: Powers (Martial Enforcers of Law, Chains & Restraining Aegis)
    "Yehuiah": "a towering Power clad in dark starlight plate, anchoring vertebrae of celestial law with glowing chains and raising high an iron talisman of cosmic subordination",
    "Lehahiah": "a towering Power encased in frost-gleaming adamantine armor, subduing rebellious spirits with celestial collars and thrusting upward a platinum talisman of divine obedience",
    "Chavakiah": "a towering Power armored in radiant solar gold, mending shattered ancestral pacts with bonds of light and hoisting aloft an engraved talisman of familial reconciliation",
    "Menadel": "a towering Power in heavy blackened-iron harness, liberating exhausted souls from oppressive toil and brandishing high an unyielding talisman of sanctified labor",
    "Aniel": "a towering Power balancing scales of kinetic celestial wind, shattering karmic chains of tyranny and raising aloft an argent talisman of broken cycles",
    "Haamiah": "a towering Power standing vigilant in ritual-scribed aegis armor, warding celestial sanctuaries with blazing runes and thrusting high a sacred talisman of spiritual protection",
    "Rehael": "a towering Power humming with harmonic frequencies of health, binding corrosive plagues with golden cords and hoisting aloft an amber talisman of filial healing",
    "Ieiazel": "a towering Power releasing concussive shockwaves of purified air, snapping dungeon iron with celestial chains and brandishing high a silver talisman of divine rescue",

    # Choir 6: Virtues (Miracle-Workers, Star Crowns & Elemental Catalysts)
    "Hahahel": "a towering Virtue crowned in blazing stellar constellations, channeling selfless apostolic sacrifice through an altar of light and raising high a consecrated talisman of holy priesthood",
    "Mikael": "a towering Virtue holding the central axis of sovereign harmony, steadying kingdoms amid shifting astral winds and thrusting upward a solar talisman of political order",
    "Veuliah": "a towering Virtue radiating turquoise subterranean luminescence, toppling tyrannical dynasties from the deep earth and raising aloft an orichalcum talisman of prosperous liberation",
    "Yelahiah": "a towering Virtue wreathed in bioluminescent martial wrath, guarding righteous warriors across dark currents and brandishing high a Damascus-steel talisman of karmic victory",
    "Sealiah": "a towering Virtue pulsing like a colossal cosmic heart, reigniting stagnant terrestrial currents with surge-force and hoisting aloft a verdant talisman of restored motion",
    "Ariel": "a towering Virtue parting abyssal ocean trenches, unearthing forgotten treasures of natural wisdom and thrusting high an iridescent mother-of-pearl talisman of secret revelation",
    "Asaliah": "a towering Virtue illuminating the midnight cosmos with contemplative brilliance, praising cosmic design and raising aloft a celestial prism talisman of divine cognition",
    "Mihael": "a towering Virtue radiating warm marine bioluminescence, intertwining divided souls into enduring unity and brandishing high a rose-gold talisman of conjugal harmony",

    # Choir 7: Principalities (Guardians of Nations, Celestial Architects & Scepters)
    "Vehuel": "a towering Principality crowned in galactic star-fire, releasing searing arrows of truth into the void and raising high a scintillating diamond talisman of divine exaltation",
    "Daniel": "a towering Principality crowned in shimmering rhetoric-flame, unfurling radiant parchment scrolls of cosmic law and thrusting upward an inscribed sapphire talisman of eloquent justice",
    "Hahasiah": "a towering Principality crowned in Hermetic alchemical fire, distilling universal panaceas from celestial vessels and raising aloft a smoldering cinnabar talisman of esoteric mysteries",
    "Imamiah": "a towering Principality crowned in incandescent plasma, trampling demonic pride beneath luminous boots and brandishing high a molten bronze talisman of spiritual expiation",
    "Nanael": "a towering Principality crowned in sacred philosophical radiance, contemplating planetary harmonics with celestial compasses and hoisting high an azure talisman of deep contemplation",
    "Nithael": "a towering Principality enthroned upon a pillar of starry marble, shielding emperors and rightful bloodlines while raising aloft an imperial gold talisman of legitimate succession",
    "Mebahiah": "a towering Principality crowned in carved mountain granite, anchoring righteous deeds into terrestrial foundations and thrusting high a lapis-inlaid talisman of lucid morality",
    "Poiel": "a towering Principality crowned in showering auric gold, opening infinite vaults of celestial abundance and brandishing high an opulent jewel-set talisman of boundless providence",

    # Choir 8: Archangels (Cosmic Heralds, Sacred Geometry & Divine Wisdom)
    "Nemamiah": "a towering Archangel forging crystalline spires of war, directing righteous commanders with a star-metal staff and raising high an adamantine talisman of righteous causes",
    "Yeialel": "a towering Archangel of deep resonant basalt, banishing sorrowful afflictions with an unbending rod and thrusting upward a cold meteoric iron talisman of endurance",
    "Harahel": "a towering Archangel unlocking limitless celestial archives, illuminating sacred texts with an ivory staff and hoisting high an inscribed golden talisman of intellectual wealth",
    "Mitzrael": "a towering Archangel mending the shattered geometry of the spirit, driving away internal madness and raising aloft a polished beryl talisman of psychological healing",
    "Umabel": "a towering Archangel weaving lines of astronomical affinity across the night sky, fostering enduring bonds and brandishing high an electrum talisman of sacred friendship",
    "Iah-Hel": "a towering Archangel enveloped in serene cloistered starlight, radiating transcendent wisdom through absolute silence and thrusting upward a crystal talisman of divine illumination",
    "Anauel": "a towering Archangel safeguarding terrestrial voyages and rightful commerce, shielding trade roads with a copper staff and raising high a coin-stamped talisman of material protection",
    "Mehiel": "a towering Archangel inking verses of celestial lightning into the clouds, inspiring grand orators and brandishing aloft a flaming quill-carved talisman of poetic inspiration",

    # Choir 9: Angels (Terrestrial Guardians, Living Waters & Rebirth)
    "Damabiah": "a towering Angel of gushing living headwaters, bestowing boundless oceanic wisdom upon searching souls and raising high a shimmering aquamarine talisman of the eternal spring",
    "Manakel": "a towering Angel diving through abyssal marine depths, interpreting prophetic dreams from dark waters and thrusting upward a polished pearl talisman of good and evil",
    "Eyael": "a towering Angel projecting incandescent mists of solar ascension, transmuting mundane sorrow into celestial joy and hoisting aloft an alchemical gold talisman of sublime transformation",
    "Habuhiah": "a towering Angel breathing fertile dew across barren orchards, healing blighted crops with radiant palms and raising high an emerald talisman of agricultural healing",
    "Rochel": "a towering Angel scouring the cosmic tides for lost treasures, returning stolen heritages to rightful heirs and brandishing aloft an argent talisman of complete restitution",
    "Jabamiah": "a towering Angel churning primordial oceans into celestial essence, resurrecting corrupted matter into perfection and thrusting upward a glowing chrysolite talisman of universal regeneration",
    "Haiaiel": "a towering Angel girded in blinding celestial armor, arming the pure of heart with flaming blades and raising high a luminous steel talisman of divine warfare and peace",
    "Mumiah": "a towering Angel of final oceanic dissolution, weaving twilight endings into triumphant rebirth and hoisting aloft an alpha-and-omega inscribed talisman of cosmic transformation",
}

DEMON_FORMS = {
    # 1 - 10
    "Bael": "a towering triple-headed king with heads of toad, cat, and crowned man on spider legs, rasping hoarsely and raising high a heavy iron talisman of invisibility",
    "Agares": "a towering, ancient nobleman riding an earth-shaking crocodile with a goshawk upon his fist, halting retreating armies and thrusting upward a smoldering copper talisman of language",
    "Vassago": "a towering infernal prince of radiant angelic countenance with burning sapphire eyes, declaring things past and future while raising aloft a crystalline talisman of prophecy",
    "Gamigin": "a towering spectral horse that shifts into an ashen phantom with a thunderous voice, summoning deceased souls and hoisting high a black-iron talisman of the dead",
    "Marbas": "a towering mechanical-maned lion with roaring furnace-fire eyes, answering truthfully of hidden secrets and brandishing high an iron-wrought talisman of mechanical craft and healing",
    "Valefor": "a towering prowling lion with the head of an ass-eared brigand, slipping through stealing shadows and raising high a tarnished silver talisman of thievery",
    "Amon": "a towering wolf with a serpent's tail vomiting dark obsidian flames, reconciling broken friendships and brandishing aloft a dense lead talisman of feuds and alliances",
    "Barbatos": "a towering horned hunter-lord accompanied by phantom kings and horn blasts, understanding animal tongues and raising high a moss-grown copper talisman of hidden treasure",
    "Paimon": "a towering, crowned monarch riding an imperial dromedary amid clashing cymbals and deafening trumpets, raising aloft an opulent gold talisman of spirit dominion",
    "Buer": "a towering wheel of five human legs radiating from a horned lion's head, spinning over bedrock and thrusting upward a bronze talisman of natural philosophy and healing",

    # 11 - 20
    "Gusion": "a towering cynocephalic ape-hound of weathered meteoric stone, discerning past, present, and future while brandishing aloft a heavy leaden talisman of restored honors",
    "Sitri": "a towering leopard with the wings of a massive griffin, smoldering with incandescent crimson passion and raising high a glowing copper talisman of inflamed desire",
    "Beleth": "a towering, terrible king riding a pale warhorse, preceded by blasting brass trumpets that shake the earth and thrusting upward a silver talisman of compelled love",
    "Leraje": "a towering emerald-clad archer drawing a massive recurve bow that whistles with shearing winds, raising aloft a green-patinated copper talisman of gangrenous wounds",
    "Eligos": "a towering ghostly knight clad in shining spectral plate, bearing lance, standard, and serpent while hoisting high a polished copper talisman of martial victory",
    "Zepar": "a towering crimson-armored soldier whose curved broadsword radiates scorched heat-distortion, inflaming minds and brandishing aloft a blood-rusted iron talisman of sterile lust",
    "Botis": "a towering horned viper shifting into a dreadful warrior with great tusks and drawn sword, thrusting high an electrum talisman of reconciled enemies",
    "Bathin": "a towering muscular giant with a serpent's tail riding a pale steed through planar portals, raising aloft a pale brass talisman of instantaneous transit",
    "Sallos": "a towering gallant soldier in a ducal coronet riding a leviathan crocodile with serene dignity, raising high a gleaming silver talisman of peaceful love",
    "Purson": "a towering lion-faced king clutching a hissing viper, riding a monstrous bear through blaring trumpets and hoisting high a radiant gold talisman of hidden treasures",

    # 21 - 30
    "Marax": "a towering bull with the bearded face of an ancient sage, expounding astronomy and herbal virtues while brandishing aloft a heavy iron talisman of liberal sciences",
    "Ipos": "a towering, uncanny angel with a roaring lion's head, goose's feet, and a hare's tail, thrusting upward a bright brass talisman of wit and valor",
    "Aim": "a towering three-headed nobleman riding a viper with a blazing firebrand, setting citadels ablaze and raising high a smoldering bronze talisman of consuming intellect",
    "Naberius": "a towering three-headed black crane shrieking in a raspy voice, walking through chthonic vaults and raising aloft a dark copper talisman of restored dignity",
    "Glasya-Labolas": "a towering winged bloodhound with broad griffin wings, trailing a crimson mist of bloodshed and brandishing high a bloodstained talisman of slaughter and prophecy",
    "Bune": "a towering three-headed dragon speaking in high resonant tones from sepulchral vaults, hoisting high a lustrous copper talisman of necromantic wealth",
    "Ronove": "a towering shadow-cloaked monster leaning upon a reaper's staff, harvesting weary souls while raising high an iron talisman of persuasive rhetoric and languages",
    "Berith": "a towering red-and-gold soldier upon a crimson warhorse, wearing a crown of molten flame and thrusting upward a blazing gold talisman of transmuting metals",
    "Astaroth": "a towering sorrowful fallen angel riding an infernal dragon with a coiled viper in hand, brandishing aloft a poisonous copper talisman of past falls",
    "Forneus": "a towering abyssal sea monster gliding through cold oceanic depths, granting eloquence across tongues and raising high a silver talisman of oceanic rhetoric",

    # 31 - 40
    "Foras": "a towering muscular sage carved from ancient gnarled oak, teaching logic, ethics, and herbology while hoisting high an earthen brass talisman of wisdom and longevity",
    "Asmoday": "a towering three-headed emperor vomiting furnace flames from an infernal dragon, brandishing aloft a molten gold talisman of invincible dominion and wrath",
    "Gaap": "a towering majestic emperor guiding four cardinal kings through the subterranean void, transporting mortals swiftly and raising high an inscribed leaden talisman of philosophy",
    "Furfur": "a towering colossal hart with a blazing lightning-wreathed tail, summoning roaring thunderheads and thrusting upward a crackling copper talisman of tempests and love",
    "Marchosias": "a towering armored wolf with griffin wings and a serpent's tail, vomiting columns of fire and brandishing high a gleaming silver talisman of martial fidelity",
    "Stolas": "a towering crowned raven with infinitely deep starlit eyes, perched upon celestial astrolabes and raising high an etched silver talisman of astronomy and herbs",
    "Phenex": "a towering phoenix wreathed in black-and-crimson flames, singing poetic verses with a haunting child's voice and hoisting aloft a smoldering copper talisman of melody",
    "Halphas": "a towering dark iron stock-dove shrieking with a hoarse voice, raising impenetrable fortress walls and thrusting upward a heavy brass talisman of siege warfare",
    "Malphas": "a towering soot-black raven shifting into a grim architect who shatters enemy citadels, raising aloft a dense leaden talisman of masonry and deception",
    "Raum": "a towering monstrous crow shifting into an armored plunderer, toppling dynasties and destroying cities while brandishing high a fractured copper talisman of ruin and theft",

    # 41 - 50
    "Focalor": "a towering winged warrior commanding hurricane winds and churning ocean whirlpools to sink warships, raising high a verdigris copper talisman of maritime wrath",
    "Vepar": "a towering ferocious mermaid rising from foaming breakers, guiding dread armadas with sea-fogs and hoisting aloft a tarnished silver talisman of nautical pestilence",
    "Sabnock": "a towering lion-headed soldier in heavy armor riding a pale horse, striking foes with gangrenous wounds and thrusting upward an iron talisman of fortress architecture",
    "Shax": "a towering dark stork-demon shrieking in a raspy voice, plundering mortal sight, hearing, and understanding while brandishing aloft a black talisman of sensory theft",
    "Vine": "a towering crowned lion riding a jet-black warhorse, brandishing a viper while summoning howling squalls and raising high an opulent gold talisman of exposed sorcery",
    "Bifrons": "a towering shape-shifting shade lighting spectral grave-lanterns over crumbling tombs, thrusting upward a weathered copper talisman of necromantic geometry and astronomy",
    "Vual": "a towering desert dromedary shifting into a fierce warrior speaking ancient Egyptian mysteries, raising high a hammered copper talisman of enduring affection",
    "Haagenti": "a towering winged golden bull shifting into a man, transmuting lead to gold and water to wine while hoisting high a molten gold talisman of supreme alchemy",
    "Crocell": "a towering luminous fallen angel murmuring with the sound of rushing waters, revealing steaming mineral baths and brandishing aloft a geometric brass talisman of secret sciences",
    "Furcas": "a towering white-bearded sage riding a skeletal pale steed, wielding a razor-sharp barbed spear and thrusting upward a heavy lead talisman of logic and pyromancy",

    # 51 - 60
    "Balam": "a towering three-headed king with flaming eyes, riding a raging bear with a goshawk on his fist and raising high an engraved gold talisman of past and future",
    "Alloces": "a towering knight with a fiery lion's face and scarlet eyes, riding an armored warhorse with crushing momentum and brandishing aloft a copper talisman of astronomy",
    "Camio": "a towering singing thrush stepping from burning embers to become a swordsman answering in ash, hoisting high a silver talisman of animal omens",
    "Murmur": "a towering crowned ducal warrior riding a ferocious griffin, preceded by ghostly horn blasts and raising aloft an ancient copper talisman of necromantic philosophy",
    "Orobas": "a towering magnificent horse shifting into a noble man, declaring truths of divinity without deceit and thrusting upward a sacred silver talisman of virtuous favor",
    "Gremory": "a towering all-seeing duchess wearing a jeweled coronet, riding an imperial camel through velvet shadows and brandishing aloft an opulent copper talisman of hidden treasures",
    "Ose": "a towering spotted leopard shifting into a crowned illusionist who plunges minds into madness, raising high an etched silver talisman of mental transfiguration",
    "Amy": "a towering blazing column of roaring fire resolving into an astrological magus, thrusting upward a glowing brass talisman of guarded spirit-treasures",
    "Orias": "a towering lion riding a serpent-tailed stallion, holding two hissing vipers toward starry heavens and raising high a silver talisman of astrological wisdom",
    "Vapula": "a towering lion with broad griffin wings, guiding artisans in intricate mechanics and treatises while brandishing aloft a polished copper talisman of sacred craftsmanship",

    # 61 - 72
    "Zagan": "a towering bull with griffin wings, transmuting water to wine, blood to oil, and base metals into coin while hoisting high a radiant gold talisman of transmutation",
    "Valac": "a towering winged angelic youth with blazing eyes, riding a colossal two-headed dragon through cavernous fissures and thrusting upward a serpentine talisman of hidden treasures",
    "Andras": "a towering angelic warrior with the head of a black raven, riding a ravenous midnight wolf with drawn saber and raising aloft a gleaming steel talisman of discordant slaughter",
    "Flauros": "a towering terrifying leopard leaping into a flame-eyed warrior who incinerates foes with divine fury, brandishing high a smoldering copper talisman of consuming fire",
    "Andrealphus": "a towering peacock shrieking with earsplitting pride, shifting into an architect who measures the cosmos and hoisting high an engraved silver talisman of mensuration",
    "Kimaris": "a towering heroic warrior in blackened harness riding a colossal black charger across spirit realms, raising high a bronze talisman of hidden grammar and rhetoric",
    "Amdusias": "a towering unicorn bending great forest oaks as invisible orchestral trumpets resound, thrusting upward a polished copper talisman of symphonic command",
    "Belial": "a towering pair of radiant fallen angels riding a chariot of roaring fire, speaking with deceptive honeyed grace and raising aloft an imperial gold talisman of sovereign favor",
    "Decarabia": "a towering five-pointed pentacle-star shifting into an enchanter surrounded by flocks of singing birds, brandishing high a lustrous silver talisman of herbal lore",
    "Seere": "a towering breathtaking youth riding a winged pegasus, moving across the world in the blink of an eye and hoisting high an electrum talisman of instantaneous abundance",
    "Dantalion": "a towering nobleman of infinite shifting faces, holding an open grimoire of human desires and thrusting upward a dark copper talisman of telepathic sway",
    "Andromalius": "a towering stern judge holding a colossal writhing serpent, tracking down thieves and punishing the wicked while raising high an iron talisman of absolute retribution",
}

# Occurrence index per gate for angels whose name repeats in the roster
# (Lauviah appears at gates 11 and 17); parallel to GATES.
_ANGEL_SEEN = {}
ANGEL_OCCURRENCE = []
for _gg, _aa, _dd, _ee in GATES:
    ANGEL_OCCURRENCE.append(_ANGEL_SEEN.get(_aa, 0))
    _ANGEL_SEEN[_aa] = _ANGEL_SEEN.get(_aa, 0) + 1
del _ANGEL_SEEN, _gg, _aa, _dd, _ee


def angel_form(angel, occurrence=0):
    """Expanded traditional form for an angel; occurrence disambiguates repeats."""
    key = angel
    if occurrence:
        alts = [k for k in ANGEL_FORMS if k != angel and k.startswith(angel)]
        if alts:
            key = alts[occurrence - 1]
    return ANGEL_FORMS[key]


def demon_form(demon):
    """Expanded traditional Goetic form for a demon."""
    return DEMON_FORMS[demon]


# Physical gate architecture per element -- the thing he actually walks through.
THRESHOLDS = {
    "Fire":  "a towering arch of black volcanic glass wreathed in slow-burning cymatic flame",
    "Earth": "a moss-covered stone trilithon humming with a low resonant tone",
    "Air":   "a lattice of prismatic light refracting the sky into slow ribbons",
    "Water": "an arch of living coral and breathing bioluminescent membrane",
}

# Wheel tones: each act resolves to a pure tone a perfect fifth (1.5x) above
# the last, folded into the 40-80Hz sub range -- the circle of fifths walking
# the 12 signs. Every segment OPENS continuing the previous segment's tone and
# CLOSES in a calm moment at its tail tone -- the tail tone IS the handoff.
WHEEL_TONES = {
    1: 40, 2: 60, 3: 45, 4: 68, 5: 51, 6: 76,
    7: 57, 8: 43, 9: 64, 10: 48, 11: 72, 12: 54,
}

GLYPHS = {
    1:  ("Aries",       "the Ram's glyph, two bold curved horns"),
    2:  ("Taurus",      "the Bull's glyph, a horned circle"),
    3:  ("Gemini",      "the Twins' glyph, twin pillars joined by arcs"),
    4:  ("Cancer",      "the Crab's glyph, twin spiraling whorls"),
    5:  ("Leo",         "the Lion's glyph, a single sweeping mane-curve"),
    6:  ("Virgo",       "the Maiden's glyph, an intricate looping sigil"),
    7:  ("Libra",       "the Scales' glyph, a balanced horizon over a rising curve"),
    8:  ("Scorpio",     "the Scorpion's glyph, a barbed tail striking upward"),
    9:  ("Sagittarius", "the Archer's glyph, a drawn bow loosing an arrow skyward"),
    10: ("Capricorn",   "the Sea-Goat's glyph, a horned crown diving into a fishtail curl"),
    11: ("Aquarius",    "the Water-Bearer's glyph, twin zigzag waves"),
    12: ("Pisces",      "the Fishes' glyph, two mirrored arcs bound by a single cord"),
}

ACT_ENVS = {
    1:  "a suspended perfect circle of floating polished obsidian and volcanic glass shards erupting with cymatic fire fractals",
    2:  "colossal monolithic stone temples covered in ancient glowing moss-covered sigils",
    3:  "dual mirrored prisms floating on aggressive high-altitude air currents",
    4:  "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton",
    5:  "a blinding solar corona filled with spinning gold geometric projections",
    6:  "hyper-precise cold sacred geometry and infinite anatomical spine mappings",
    7:  "unfolding scales of light and absolute mirrored symmetry",
    8:  "a massive obsidian gateway pulsing with a bioluminescent serpentine nervous system",
    9:  "an arrow of pure light shooting through an expanding cosmic Akashic library",
    10: "an ancient mountain archive of deeply carved glowing seals",
    11: "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations",
    12: "an infinite borderless ocean of consciousness dissolving into starlight mist",
}

DISSOLVE = {
    "Fire":  "dying cymatic flames and cooling volcanic glass",
    "Earth": "crumbling stone dust and fading moss-light",
    "Air":   "thinning high-altitude currents and shattering prisms",
    "Water": "receding bioluminescent waters and dissolving mist",
}

GLYPH_FORMATION = {
    "Fire":  "forged from living flame and volcanic glass",
    "Earth": "carved from moss-covered living stone",
    "Air":   "woven from prismatic light and high wind",
    "Water": "condensed from bioluminescent water and living coral",
}

IGNITION = {
    "Fire":  "cymatic flames erupt outward, igniting",
    "Earth": "stone dust settles and monoliths rise, revealing",
    "Air":   "the air thins into high currents, revealing",
    "Water": "bioluminescent waters flood inward, revealing",
}

# ------------------------------------------------------------------ JSON builders
# Every Omni request is ONE JSON document: `meta` (standing rules + state
# variables, substituted from the environment at run time) + `segment` (every
# variable for this gate). The JSON text is the entire text part of the
# request -- no markdown, no mixed prose.


def build_meta():
    """Standing rules + run-time state variables. Identical in every request."""
    return {
        "schema": SCHEMA_VERSION,
        "state": {
            "model": MODEL_ID,
            "duration_s": GATE_SECONDS,
            "resolution": RESOLUTION,
            "draft_mode": DRAFT_MODE,
            "stride_cycle_s": 1.2,
            "tempo_bpm": 100,
            "forward_speed_m_s": 1.2,
            "camera": "continuous unbroken over-the-shoulder rear tracking dolly",
            "lens": "35mm T1.3 anamorphic",
        },
        "rules": {
            "continuity": "every segment begins on the last frame of the previous segment, ends at the seam to the next gate in mid-motion",
            "appearance": "the World Bridger's appearance is locked to the bridger_ref image -- reproduce his avatar in the new last feame reference rendering only his body from it exactly, never restyle or reinterpret it; airborne ash and embers are environment, never markings on his robes other than glowing golden runes and basalt staff",
            "behavior": "at the center of the scene 5s mark he should continue his pace as he looks at the massive deities ahead. He raises his staff his voice booms out the names of the angel and demon who immediately bow in submission having been defeated by his name magic",
            "behavior_order": "angel first, then demon -- never swapped, no text overlays and both names should be said within 1s of eachother at the halfway point",
            "pacing": "stride is metronome-locked at 100 BPM, 1.2 m/s, identical in every shot -- he never walks faster, never slows down, never hurries a gate crossing",
            "motion": "the dolly never stops its forward drift; the shot flows continuously first frame to last;",
            "heading": "he walks dead straight ahead at all times -- spine aligned to his forward axis, never drifting, leaning, or cocking; the naming beat is the only head motion in the traversal",
            "segment_boundary": "every segment ends mid-stride, mid-dolly, exactly as the next one begins in the center of the gates doorway, so the hard cut between them is invisible and audio should seam perfectly",
            "camera_lock": "locked over-the-shoulder rear tracking; forward motion only; no cuts, no angle changes, no unscripted camera moves",
            "text": "zero text overlays of any kind -- everything visible must exist physically inside the scene never any text",
            "audio": "full diegetic audio is rendered: the act's audio bed, the wheel-tone handoff, and the two guardian names spoken by the Bridger -- the ONLY words ever spoken in the traversal",
        },
    }


def build_audio(act_audio, tone, spoken_names):
    """Full-audio block: act bed + wheel-tone handoff + the two spoken names."""
    return {
        "mode": "full_diegetic_audio",
        "open": "already continuing the previous segment's tail tone -- no click, no jump",
        "bed": act_audio,
        "spoken": [
            {"at": "naming beat", "speaker": "the World Bridger", "line": name}
            for name in spoken_names
        ],
        "speech_rule": "the two guardian names are the ONLY words spoken in the entire traversal",
        "close": f"all sound settles into a calm pure {tone}Hz wheel tone, held steady through the final seconds -- this is the handoff; the next segment picks up exactly from this tone. Render the tail as a seamless continuation, never an ending.",
        "forbidden": ["narration", "voiceover", "host commentary", "music", "text", "non-diegetic sound effects"],
    }


def build_naming_beat(angel, demon):
    """The simplified naming beat. No face is ever mentioned."""
    return {
        "at": "the threshold, mid-segment",
        "actions": [
            {"actor": "bridger", "do": "raise_staff"},
            {"actor": "bridger", "do": "nod", "toward": "left", "speak": angel},
            {"actor": "bridger", "do": "nod", "toward": "right", "speak": demon},
            {"actor": angel, "do": "bow"},
            {"actor": demon, "do": "bow"},
            {"actor": "guardians", "do": "submit and acknowledge"},
        ],
    }


def gate_segment(i, gate, angel, demon, env, element, next_env):
    """JSON variable block for one gate segment."""
    act_num, act_name, _el, act_audio = act_of(gate)
    tone = WHEEL_TONES[act_num]
    mid = GATE_SECONDS // 2
    seg = {
        "id": f"gate_{gate:02d}",
        "kind": "gate",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": act_name,
            "element": element,
            "wheel_tone_hz": tone,
        },
        "guardians": {
            "left": {
                "name": angel,
                "choir": choir_of(gate),
                "form": angel_form(angel, ANGEL_OCCURRENCE[i]),
            },
            "right": {
                "name": demon,
                "order": "Ars Goetia",
                "form": demon_form(demon),
            },
        },
        "environment": env,
        "threshold": THRESHOLDS[element],
        "next_environment": next_env,
        "beats": {
            "approach": f"the threshold looms ahead -- {THRESHOLDS[element]} -- with {angel} gathering at its left pillar and {demon} at its right pillar",
            "naming": build_naming_beat(angel, demon),
            "naming_at_s": f"around {mid - 1}-{mid + 1}s",
        },
        "audio": build_audio(act_audio, tone, [angel, demon]),
    }
    if gate == 48:
        seg["beats"]["crossing"] = (
            "at the 28th spoke rim the Bridger strikes his staff down on the stone -- "
            "a golden 108-degree geodesic Zen Shield flares to life around him, "
            "shattering the matrix wall. Fully solid and opaque, he strides on as the "
            "camera turns 90 degrees left off the zodiac track into the zero-gravity "
            "Ophiuchus Void (the single scripted camera move of the traversal), and "
            "all audio drops into absolute dead silence."
        )
        # The breach ends in dead silence, not a wheel tone -- the Act 9
        # transition opens with "the sound rises from absolute silence".
        seg["audio"]["close"] = (
            "absolute dead silence -- the breach ends in silence, not a wheel tone. "
            "Render the tail as a seamless continuation into silence, never an ending."
        )
    elif gate == 72:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the final gate, fully solid and opaque -- he "
            "does not fade, dissolve, or vanish -- as the borderless ocean detonates "
            "into white-gold ignition: volcanic glass and cymatic fire fractals blooming "
            "outward. The 360-degree loop closes seamlessly back into the Gate 1 threshold."
        )
    else:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the gate plane on foot between the parted "
            "guardians -- fully solid and opaque, he does not fade, dissolve, or vanish "
            f"-- and the camera follows him through; beyond the far plane, {next_env} "
            "already resolving through the shimmer."
        )
    return seg


def transition_segment(act_num, is_first):
    """JSON variable block for one act-transition segment."""
    sign, glyph = GLYPHS[act_num]
    _n, _name, element, act_audio = act_of(act_num * 6 - 5)  # first gate of the act
    tone = WHEEL_TONES[act_num]
    seg = {
        "id": f"act_{act_num:02d}_transition",
        "kind": "act_transition",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": f"{sign} transition",
            "element": element,
            "wheel_tone_hz": tone,
        },
        "glyph": {
            "sign": sign,
            "description": glyph,
            "formation": f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}, hanging in the space and slowly rotating, lit from within",
        },
        "elemental_shift": {
            "outgoing": DISSOLVE[act_of((act_num - 1) * 6)[2]] if not is_first else "dying cymatic flames and cooling volcanic glass",
            "incoming": f"{IGNITION[element]} {ACT_ENVS[act_num]}",
        },
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken -- no guardians stand at this threshold",
    }
    if is_first:
        seg["scene"] = (
            "opening cold: the camera is already locked over the shoulder of the World "
            "Bridger as he strides forward through dying cymatic flames and cooling "
            "volcanic glass, the old element dying away. Golden wireframe 72-gate rings "
            "and glowing cuneiform code matrices float in zero gravity around him. Ahead, "
            f"the zodiac glyph of {sign} -- {glyph} -- materializes. The camera passes "
            f"beneath the glyph and {IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["open"] = "the loop closes here -- open continuing the calm 54Hz wheel tone the final Pisces gate left behind"
        seg["audio"]["spoken"] = []
    else:
        prev_element = act_of((act_num - 1) * 6)[2]
        prev_audio = act_of((act_num - 1) * 6)[3]
        prev_tone = WHEEL_TONES[act_num - 1]
        seg["scene"] = (
            "starting exactly from the first frame -- same camera, same forward drift, no "
            f"cut -- the old world falls behind. The camera drifts through {DISSOLVE[prev_element]} "
            f"as the old element dies away. Ahead, the zodiac glyph of {sign} -- {glyph} -- "
            f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}. The Bridger "
            f"walks on beneath it, staff in hand. As the camera passes beneath the glyph, "
            f"{IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["spoken"] = []
        if act_num == 9:
            # Act 9 follows gate 48's breach, which ends in dead silence, not a tone.
            seg["audio"]["open"] = "the sound rises from absolute silence"
        else:
            seg["audio"]["open"] = (
                f"gliding continuously from the fading {prev_tone}Hz wheel tone -- a smooth "
                f"pitch glide, no click, no jump -- as {prev_audio} dissolves into {act_audio}"
            )
    return seg


def cold_open_segment():
    """JSON variable block for the cold-open pre-step: black void, him walking.

    Generated FIRST; its last frame seeds segment 0. This is the loop point --
    gate 72's tail walks back into this frame.
    """
    return {
        "id": "cold_open",
        "kind": "cold_open",
        "shot": "single unbroken shot",
        "note": "the loop point and the film's first frame -- a deliberate frame, never an arbitrary fresh generation",
        "scene": (
            "pure infinite solid matte black background (#000000) -- absolute zero "
            "background geometry, no pillars, no fire, no ground textures, no horizon. "
            "Only the World Bridger exists in the frame: dead-center rear view, mid-stride, "
            "carved basalt staff in his RIGHT hand, walking dead straight ahead along the "
            "central axis at his metronomic 100 BPM pace. No guardians, no naming beat."
            "hood always up, black wool, crimson lining and glowing cuneiform glyphs and runes."
        ),
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken",
        "audio": {
            "mode": "full_diegetic_audio",
            "open": "continuing the calm 54Hz wheel tone the final Pisces gate leaves behind -- the loop closes here",
            "bed": "the near-silence of the void under a faint sub-bass",
            "spoken": [],
            "speech_rule": "no words are spoken in the cold open",
            "close": "steady and unresolved -- segment 0 continues it; the old fire world ignites out of the dark",
            "forbidden": ["narration", "voiceover", "host commentary", "music", "non-diegetic sound effects"],
        },
    }


def build_request(segment, image_names):
    """Assemble the full JSON request document for one generation."""
    sources = {}
    if "bridger_ref" in image_names:
        sources["bridger_ref"] = {
            "image": image_names["bridger_ref"],
            "role": "appearance reference for the World Bridger -- appearance ONLY; it does not define the environment, pose, lighting, or scene",
        }
    if "first_frame" in image_names:
        sources["first_frame"] = {
            "image": image_names["first_frame"],
            "role": "the last frame of the previous segment -- begin on this exact frame and continue its motion, audio and lighting, and scene seamlessly",
        }
    return {
        "request": "generate_video",
        "engine": "gemini_omni",
        "meta": build_meta(),
        "sources": sources,
        "segment": segment,
    }


SEGMENTS = []
_prev_act = None
for i, (g, a, d, env) in enumerate(GATES):
    act_num = act_of(g)[0]
    if act_num != _prev_act:
        SEGMENTS.append({
            "key": f"act{act_num:02d}_transition",
            "segment": transition_segment(act_num, is_first=(len(SEGMENTS) == 0)),
        })
        _prev_act = act_num
    SEGMENTS.append({
        "key": f"gate{g:02d}",
        "segment": gate_segment(i, g, a, d, env, act_of(g)[2],
                                GATES[i + 1][3] if i + 1 < len(GATES) else None),
    })

assert len(SEGMENTS) == 84

# ------------------------------------------------------------------ inputs & API
# Stills only: the bridger_ref still and the first-frame still. No video
# references in this build.


def to_bytes(data):
    if isinstance(data, str):
        return base64.b64decode(data)
    return bytes(data)


def build_inputs(segment, first_frame_b64):
    """Build the Omni input list: [bridger_ref still?, first_frame still?, JSON text].

    The text part is the ENTIRE prompt as one JSON document -- meta + sources
    + segment. Nothing is markdown, nothing is mixed prose.
    """
    contents = []
    image_names = {}
    n_img = 0

    def add_image(b64, mime="image/png"):
        nonlocal n_img
        n_img += 1
        contents.append({"type": "image", "data": b64, "mime_type": mime})
        return f"Image{n_img}"

    if BRIDGER_REF_B64:
        image_names["bridger_ref"] = add_image(BRIDGER_REF_B64, "image/jpeg")
    if first_frame_b64:
        image_names["first_frame"] = add_image(first_frame_b64)

    request = build_request(segment, image_names)
    text = json.dumps(request, indent=2)
    contents.append({"type": "text", "text": text})

    if len(contents) == 1:
        return text
    return contents


def get_output_video(interaction):
    """Prefer .output_video, fall back to manual step traversal."""
    out = getattr(interaction, "output_video", None)
    if out is not None and (getattr(out, "data", None) or getattr(out, "uri", None)):
        return out
    for step in reversed(getattr(interaction, "steps", []) or []):
        if getattr(step, "type", None) == "model_output" and getattr(step, "content", None):
            for part in reversed(step.content):
                if getattr(part, "type", None) == "video":
                    return part
    return None


def download_uri(uri, dest_path, timeout_s=1800, poll_s=10):
    """Poll the hosted file until ACTIVE, then download it."""
    file_id = uri.rstrip("/").split("/")[-1].split(":")[0].split("?")[0]
    name = f"files/{file_id}"
    deadline = time.time() + timeout_s
    t0 = time.time()
    vlog(f"    POLL {name} every {poll_s}s (timeout {timeout_s}s)")
    while time.time() < deadline:
        elapsed = time.time() - t0
        try:
            info = client.files.get(name=name)
            state = getattr(info.state, "name", str(info.state))
            if state == "ACTIVE":
                log(f"    poll +{elapsed:.0f}s: {state} -- downloading")
                break
            if state == "FAILED":
                raise RuntimeError("video generation failed (file state FAILED)")
            log(f"    poll +{elapsed:.0f}s: hosted file state: {state}")
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"    poll +{elapsed:.0f}s: waiting on hosted file ({exc.__class__.__name__}: {str(exc)[:120]})")
        time.sleep(poll_s)
    else:
        raise TimeoutError(f"timed out after {timeout_s}s waiting for hosted video file")
    data = client.files.download(file=uri)
    with open(dest_path, "wb") as f:
        f.write(to_bytes(data))
    size = os.path.getsize(dest_path) / 1e6
    log(f"    saved {dest_path} ({size:.1f} MB)")


def _norm(exc):
    # Normalize underscores so RATE_LIMIT_EXCEEDED / quota_exceeded style
    # status enums can't slip past the classifiers.
    return re.sub(r"[_\s]+", "", f"{type(exc).__name__} {exc}".lower())


def is_quota_error(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "429", "resourceexhausted", "ratelimitexceeded", "quotaexceeded",
        "exceededyourquota", "toomanyrequests", "dailylimitexceeded",
    ))


def is_auth_error(exc):
    t = _norm(exc)
    return (not is_quota_error(exc)) and any(k in t for k in (
        "401", "invalidapikey", "apikeyinvalid", "apikeynotvalid",
        "apikeyisinvalid", "unauthorized", "permissiondenied",
    ))


def is_content_refusal(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "prohibitedcontent", "prohibiteduse", "contentguidelines",
        "blockeddueto", "violatedgoogle",
        "likeness", "likenesses", "realperson", "realpeople",
        "contentblocked", "inputblocked",
    ))


def prompt_for_new_key(reason):
    print()
    print("=" * 64)
    print(f"  KEY ISSUE MID-RUN: {reason}")
    print("  Paste another Google AI Studio key to keep rendering,")
    print("  or type 'skip' to skip this segment, 'quit' to stop the run.")
    print("=" * 64)
    try:
        new_key = getpass.getpass("  New GEMINI_API_KEY (hidden input): ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise KeyboardInterrupt
    if not new_key:
        return None
    low = new_key.lower()
    if low in ("skip", "s"):
        return "skip"
    if low in ("quit", "q", "exit"):
        raise KeyboardInterrupt
    return new_key


def generate_segment(seg, first_frame_b64, dest_path):
    """Generate one Omni segment from its JSON request. Returns True/False."""
    mode = "chained" if first_frame_b64 else "fresh"
    t0 = time.time()
    log(f"[{seg['key']}] generating ({RESOLUTION}, {mode})...")
    inputs = build_inputs(seg["segment"], first_frame_b64)
    txt = inputs if isinstance(inputs, str) else inputs[-1]["text"]
    if isinstance(inputs, str):
        parts = [f"json:{len(inputs)} chars"]
    else:
        parts = []
        for p in inputs:
            if "text" in p:
                parts.append(f"json:{len(p['text'])} chars")
            elif p.get("type") == "image":
                parts.append(f"image:{len(p.get('data') or '')/1e6:.1f} MB b64 {p.get('mime_type', '?')}")
            else:
                parts.append(str(p.get("type")))
    vlog(f"    REQUEST -> model={MODEL_ID} resolution={RESOLUTION} delivery=uri")
    vlog(f"    REQUEST parts: {', '.join(parts)}")
    if VERBOSE:
        # In verbose mode show the JSON skeleton (keys only) -- full JSON is
        # in --dry-run output.
        req = json.loads(txt)
        vlog(f"    REQUEST json: meta.rules={len(req['meta']['rules'])} rules, "
             f"sources={list(req['sources'])} segment={req['segment']['id']}")
    log(f"    sending interactions.create ...")
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=inputs,
        response_format={"type": "video", "resolution": RESOLUTION, "delivery": "uri"},
    )
    log(f"    RESPONSE in {time.time()-t0:.1f}s | interaction id: {interaction.id}")
    video = get_output_video(interaction)
    if video is None:
        log("    done, but no video in response -- recording as failed.")
        return False
    if getattr(video, "uri", None):
        vlog(f"    RESPONSE video: uri={video.uri}")
        download_uri(video.uri, dest_path)
    elif getattr(video, "data", None):
        vlog(f"    RESPONSE video: inline data ({len(to_bytes(video.data))/1e6:.1f} MB)")
        with open(dest_path, "wb") as f:
            f.write(to_bytes(video.data))
        log(f"    saved {dest_path}")
    else:
        log("    done, but no video payload -- recording as failed.")
        return False
    log(f"    [{seg['key']}] done in {time.time()-t0:.1f}s")
    return True


def generate_with_retry(seg, first_frame_b64, dest_path):
    """Returns True (rendered), False (failed -- recorded, run continues).

    Tiered fallback on content refusal: chained (with first-frame still) ->
    fresh (no still). Quota/auth errors retry the same tier after a key swap.
    """
    tiers = []
    if first_frame_b64 is not None:
        tiers.append(first_frame_b64)
    tiers.append(None)
    tier_idx = 0
    while True:
        ff = tiers[tier_idx]
        tier_label = "chained" if ff else "fresh"
        vlog(f"    attempt tier {tier_idx+1}/{len(tiers)} ({tier_label})")
        try:
            return generate_segment(seg, ff, dest_path)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if is_content_refusal(exc):
                if tier_idx + 1 < len(tiers):
                    tier_idx += 1
                    print(f"    refused -- retrying [{seg['key']}] fresh (no first-frame still)...")
                    continue
                print(f"    content refused ({exc.__class__.__name__}) -- recording as failed.")
                print(f"    detail: {str(exc)[:200]}")
                return False
            if is_quota_error(exc):
                reason = f"quota/rate limit hit ({exc.__class__.__name__})"
            elif is_auth_error(exc):
                reason = f"API key rejected ({exc.__class__.__name__})"
            else:
                raise
            print(f"    {reason}: {str(exc)[:180]}")
            new_key = prompt_for_new_key(reason)
            if new_key == "skip":
                print(f"    [{seg['key']}] skipped by user choice.")
                return False
            if new_key:
                set_active_key(new_key, source="mid-run swap")
                print(f"    retrying [{seg['key']}] with the new key...")
                continue
            print("    no replacement key entered -- recording segment as failed.")
            return False

# ------------------------------------------------------------------ frames & stitch


def last_frame_of(mp4_path):
    """Extract the TRUE last frame of a segment as PNG bytes (cached).

    Decodes the tail of the file and keeps the final decoded frame -- exact,
    unlike `-sseof -0.5 -vframes 1`, which lands ~14 frames early and causes
    a visible jump at every chained transition.
    """
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_lastframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    tmpdir = tempfile.mkdtemp(prefix="tail_")
    try:
        pattern = os.path.join(tmpdir, "tail_%04d.png")
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-sseof", "-3", "-i", mp4_path,
                        "-frames:v", "400", pattern],
                       check=True, timeout=180, stdin=subprocess.DEVNULL)
        files = sorted(glob.glob(os.path.join(tmpdir, "tail_*.png")))
        if not files:
            print("    frame extract: no frames decoded; continuing without first frame")
            return None
        shutil.move(files[-1], out)
    except Exception as exc:
        print(f"    frame extract failed ({exc}); continuing without first frame")
        return None
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
    with open(out, "rb") as f:
        return f.read()


def first_frame_of(mp4_path):
    """Extract the FIRST frame of a segment as PNG bytes (cached) -- for gap-fill."""
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_firstframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    try:
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-i", mp4_path, "-vframes", "1", "-q:v", "2", out],
                       check=True, timeout=120, stdin=subprocess.DEVNULL)
    except Exception as exc:
        print(f"    first-frame extract failed ({exc}); no end-frame constraint")
        return None
    with open(out, "rb") as f:
        return f.read()


def probe_duration(ffmpeg, path):
    """Probe exact video stream duration; fall back to container duration."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        try:
            cmd = [
                ffprobe, "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ]
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
            if out and out != "N/A":
                return float(out)
        except Exception:
            pass
    p = subprocess.run([ffmpeg, "-i", path], capture_output=True, text=True,
                       timeout=60, stdin=subprocess.DEVNULL)
    m = re.search(r"Duration:\s*(\d+):(\d+):([\d.]+)", p.stderr)
    if not m:
        raise RuntimeError(f"could not probe duration of {path}")
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def stitch_xfade(ffmpeg, paths, out, preset="medium", timeout=1800,
                 normalize=True):
    """Join with short video+audio crossfades at every boundary."""
    f = XFADE_S
    durs = [probe_duration(ffmpeg, p) for p in paths]
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    fc = []
    # Normalize every input first: segments arrive with mixed timebases and
    # xfade refuses to link mismatched timebases -- one bad join kills the
    # whole chain.
    vlabels, alabels = [], []
    scale_dim = "640:360" if DRAFT_MODE else "1920:1080"
    for k in range(len(paths)):
        if normalize:
            # 1. Reset PTS to 0.
            # 2. Add tpad to clone the final frame if it runs short by a few milliseconds.
            fc.append(
                f"[{k}:v]scale={scale_dim},setsar=1,fps=24,setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5},format=yuv420p[v{k}n]"
            )
            # Trim audio to exact video stream duration to prevent audio/video length drift
            fc.append(
                f"[{k}:a]aresample=48000,aformat=channel_layouts=stereo,"
                f"asetpts=PTS-STARTPTS,atrim=0:{durs[k]:.3f}[a{k}n]"
            )
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")
        else:
            fc.append(
                f"[{k}:v]setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5}[v{k}n]"
            )
            fc.append(f"[{k}:a]asetpts=PTS-STARTPTS[a{k}n]")
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")

    vprev = vlabels[0]
    for k in range(1, len(paths)):
        off = sum(durs[:k]) - f * k
        vout = f"vx{k}"
        fc.append(f"[{vprev}][{vlabels[k]}]xfade=transition=fade:duration={f}:offset={off:.3f}[{vout}]")
        vprev = vout

    aprev = alabels[0]
    for k in range(1, len(paths)):
        aout = f"ax{k}"
        fc.append(f"[{aprev}][{alabels[k]}]acrossfade=d={f}[{aout}]")
        aprev = aout

    cmd += ["-filter_complex", ";".join(fc),
            "-map", f"[{vprev}]", "-map", f"[{aprev}]",
            "-c:v", "libx264", "-crf", "18", "-preset", preset,
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=timeout, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (crossfaded joins)")


def stitch(filenames, out="mirror_of_spheres_json.mp4"):
    """Join segments in order. Falls back to plain concat if xfade fails."""
    paths = [os.path.abspath(os.path.join(OUT_DIR, n)) for n in filenames]
    lst = os.path.join(OUT_DIR, "concat.txt")
    with open(lst, "w") as f:
        for p in paths:
            f.write(f"file '{p}'\n")
    ffmpeg = FFMPEG or shutil.which("ffmpeg")
    if ffmpeg is None:
        print("ffmpeg not found -- skipping auto-stitch. Run:")
        print(f"  ffmpeg -f concat -safe 0 -i {lst} -c copy {out}")
        return
    if XFADE_S > 0 and len(paths) > 1:
        try:
            # Two-level crossfade: one giant 84-input filtergraph exhausts
            # memory (84 simultaneous decoders + a 160+ filter chain), so
            # stitch small chunks first and then crossfade the chunks.
            CHUNK = 7
            if len(paths) > CHUNK:
                chunkdir = os.path.join(OUT_DIR, "chunks")
                os.makedirs(chunkdir, exist_ok=True)
                chunk_paths = []
                for ci in range(0, len(paths), CHUNK):
                    cpaths = paths[ci:ci + CHUNK]
                    if len(cpaths) > 1:
                        cfile = os.path.abspath(os.path.join(
                            chunkdir, f"chunk{ci // CHUNK:02d}.mp4"))
                        print(f"  stitching chunk {ci // CHUNK + 1} "
                              f"({len(cpaths)} segments)...")
                        stitch_xfade(ffmpeg, cpaths, cfile, normalize=True)
                        chunk_paths.append(cfile)
                    else:
                        chunk_paths.append(cpaths[0])
                print(f"  stitching {len(chunk_paths)} chunks into final...")
                stitch_xfade(ffmpeg, chunk_paths, out, timeout=3600,
                             normalize=True)
                shutil.rmtree(chunkdir, ignore_errors=True)
            else:
                stitch_xfade(ffmpeg, paths, out, normalize=True)
            return
        except Exception as exc:
            import traceback
            print(f"xfade stitch failed ({exc.__class__.__name__}: {exc}); falling back to concat filter")
            traceback.print_exc()
    # Fallback: concat filter with re-encode. Immune to the timestamp/DTS
    # quirks of stream-copy across mixed-mux segments.
    segs = "".join(f"[{k}:v][{k}:a]" for k in range(len(paths)))
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    cmd += ["-filter_complex", f"{segs}concat=n={len(paths)}:v=1:a=1[vcat][acat]",
            "-map", "[vcat]", "-map", "[acat]",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=3600, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (concat fallback)")


# ------------------------------------------------------------------ run

def cold_open_path():
    return os.path.join(OUT_DIR, "cold_open.mp4")


def run_cold_open():
    """Pre-step: generate the black-void loop-point frame, return its last
    frame as PNG bytes (the seed for segment 0). Reuses the cached file."""
    path = cold_open_path()
    if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
        print("[cold_open] already rendered -- resuming, skipping")
    else:
        print("[cold_open] generating the loop-point frame (black void, him walking)...")
        ok = generate_with_retry({"key": "cold_open", "segment": cold_open_segment()},
                                 None, path)
        if not ok:
            print("[cold_open] failed -- segment 0 will generate fresh.")
            return None
    raw = last_frame_of(path)
    if raw:
        print("[cold_open] last frame extracted -- seeding segment 0.")
    return raw


def dry_run():
    """Print every segment's full JSON request. No API calls, no key."""
    req0 = build_request(cold_open_segment(), {})
    print(f"# Mirror of Spheres JSON prompts -- {len(SEGMENTS)} segments + cold open (dry run)\n")
    print("## cold_open\n")
    print(json.dumps(req0, indent=2) + "\n")
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            break
        req = build_request(seg["segment"], {"bridger_ref": "Image1",
                                             "first_frame": "Image2"})
        print(f"## {i:02d}_{seg['key']} ({len(json.dumps(req))} chars)\n")
        print(json.dumps(req, indent=2) + "\n")


def seg_path(i):
    return os.path.join(OUT_DIR, f"{i:02d}_{SEGMENTS[i]['key']}.mp4")


def main():
    if DRY_RUN:
        dry_run()
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    rlog_init({
        "schema": SCHEMA_VERSION,
        "model": MODEL_ID,
        "resolution": RESOLUTION,
        "draft_mode": DRAFT_MODE,
        "gate_seconds": GATE_SECONDS,
        "xfade_s": XFADE_S,
        "max_segments": MAX_SEGMENTS,
        "cold_open": COLD_OPEN,
        "bridger_ref": bool(BRIDGER_REF_B64),
    })
    print(f"ffmpeg: {FFMPEG if FFMPEG else 'NOT FOUND -- chaining disabled, all segments will generate fresh!'}")
    print(f"segments: {len(SEGMENTS)} (12 act-transitions + 72 gates) | {RESOLUTION} | "
          f"xfade: {XFADE_S}s {'(hard cuts)' if XFADE_S == 0 else ''} | "
          f"{'DRAFT' if DRAFT_MODE else 'PRODUCTION'}")
    print("prompting: all-JSON (meta state variables + per-segment variables) | "
          "chaining: last-frame stills only, no video references")

    # Cold open first: its last frame seeds segment 0.
    cold_seed = run_cold_open() if COLD_OPEN else None

    made, failed = [], []
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            print(f"(stopping at MAX_SEGMENTS={MAX_SEGMENTS})")
            break
        path = seg_path(i)
        if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[{seg['key']}] already rendered -- resuming, skipping")
        else:
            first_frame = None
            if i == 0 and cold_seed:
                print("    seeding from cold-open last frame")
                first_frame = base64.b64encode(cold_seed).decode("ascii")
            elif i > 0 and os.path.exists(seg_path(i - 1)):
                print(f"    chaining from last frame of {os.path.basename(seg_path(i - 1))}")
                raw = last_frame_of(seg_path(i - 1))
                first_frame = base64.b64encode(raw).decode("ascii") if raw else None
                if first_frame is None:
                    print(f"    WARNING: last-frame extraction failed -- [{seg['key']}] will generate FRESH (expect a jump cut)")
            elif i > 0:
                print("    (no previous segment file; starting fresh)")
            # Gap-fill: if the NEXT segment already exists, constrain this
            # one's tail to the next segment's first frame.
            if i + 1 < len(SEGMENTS) and os.path.exists(seg_path(i + 1)):
                print(f"    (gap-fill tail note: {os.path.basename(seg_path(i + 1))} already exists)")
            t0 = time.time()
            try:
                ok = generate_with_retry(seg, first_frame, path)
            except KeyboardInterrupt:
                print("\nInterrupted. Re-run anytime -- finished segments are kept.")
                break
            rlog({"event": "segment_done", "key": seg["key"], "ok": ok,
                  "elapsed_s": round(time.time() - t0, 1),
                  "chained": first_frame is not None})
            if not ok:
                failed.append(seg["key"])
                continue
        if os.path.exists(path):
            made.append(os.path.basename(path))
    if failed:
        print(f"\nFailed segments: {failed}")
        print("Tweak / check quota, then re-run -- finished segments are skipped.")
    if made:
        stitch(made)


if __name__ == "__main__":
    main()

In [ ]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
The Mirror of Spheres -- 72-Gate Decan Traversal (Gemini Omni, all-JSON prompts).

84 chained segments: each of the 12 acts opens with a 10s act-transition --
the zodiac glyph materializing as a vast physical sigil while the elemental
shift occurs -- followed by its 6 gates (~12:00+ total at 10s gates).

WHAT'S DIFFERENT IN THIS BUILD
------------------------------
- ALL-JSON PROMPTS. Every request sent to Omni is one JSON document: a `meta`
  block (standing rules + state variables, substituted from the environment at
  run time) and a `segment` block (every variable for that gate: act, element,
  wheel tone, angel {name, choir, form}, demon {name, form}, environment,
  threshold, glyph, beats). No markdown, no mixed prose -- the model gets
  structure, the way it likes it.
- FULL AUDIO. Diegetic audio is rendered: the act's audio bed, the wheel-tone
  handoff (every segment opens continuing the previous tail tone and closes
  settled into its own), and the ONLY permitted spoken words -- the World
  Bridger speaking the two guardian names at the naming beat.
- STILL-FRAME CHAINING ONLY. Each segment takes a still of the TRUE last frame
  of the previous segment and patches it in as the first frame. No video
  references (no prev-segment attach, no hero clip) -- stills only.
- COLD OPEN. An optional pre-step generates the loop-point frame first: pure
  black void with him walking ("a black screen with him on it"). Its last
  frame seeds segment 0, so the film opens deliberately and gate 72's tail
  closes back into it -- the loop plays all day with no visible seam.
- SIMPLIFIED BEHAVIOR. The prompts never mention his face at all. At each
  gate: he raises his staff and nods toward the angel as he speaks its name,
  then nods toward the demon as he speaks its name. That's it.
- DRAFT/PROD TOGGLE. DRAFT_MODE=1 -> 360p into separate dirs at roughly a
  third of the cost. Unset -> full 1080p production.

    pip install -U "google-genai>=2.10.0" imageio-ffmpeg
    export GEMINI_API_KEY="your-key"      # from https://aistudio.google.com/
    python produce_video_mirror_json.py

Continuity: F_k,start = F_{k-1},end -- the previous segment's true last frame
(extracted with a bundled ffmpeg -- the exact final decoded frame, not an
estimate) is declared as the first-frame source of the next segment.

Gate roster: all 72 pairs match Paul's canonical blueprint verbatim.
Guardian iconography: 144 expanded historically-grounded descriptions
(ANGEL_FORMS / DEMON_FORMS, supplied by Paul 2026-10-08) -- every guardian
towering, elementally distinct, bearing its seal. Angel choirs derived from
gate number (1-8 Seraphim ... 65-72 Angels).

Quota handling: on rate-limit / quota errors (or a rejected key) the script
pauses and prompts for another Google AI Studio key (hidden input). Type
'skip' to skip the segment, 'quit' (or Ctrl+C) to stop.

NOTE: guardian names (Bael, Asmoday, Belial, Astaroth, ...) can trip the
content filter. The tiered fallback sheds the first-frame still on refusal
(chained -> fresh) and the run continues; re-run heals gaps via gap-fill.
"""

import base64
import getpass
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

try:
    from google import genai
except ImportError:  # --dry-run works without the SDK installed
    genai = None

# ------------------------------------------------------------------ config

SCHEMA_VERSION = "mirror-json/1.0"

# Paul's finding (2026-10-08): gemini-omni-flash-preview (the legacy preview
# path) encodes the same-quality renders at ~2-4MB vs ~9MB for 1.1-flash --
# same per-second price, far lighter segments and stitches. Default is the
# preview path he actually renders on; override with OMNI_MODEL=....
MODEL_ID = os.environ.get("OMNI_MODEL", "gemini-omni-flash-preview")

API_KEY = os.environ.get("GEMINI_API_KEY")
# --dry-run prints every segment's full JSON prompt without touching the API.
DRY_RUN = "--dry-run" in sys.argv
if not API_KEY and not DRY_RUN:
    raise SystemExit("Set GEMINI_API_KEY first: export GEMINI_API_KEY='your-key'")

client = None
API_KEY_POOL = []


def set_active_key(api_key, source=""):
    global client
    if genai is None:
        raise SystemExit("The google-genai SDK is not installed -- run via "
                         "run_mirror_json.sh or: pip install \"google-genai>=2.10.0\"")
    api_key = api_key.strip()
    os.environ["GEMINI_API_KEY"] = api_key
    if api_key not in API_KEY_POOL:
        API_KEY_POOL.append(api_key)
    client = genai.Client(api_key=api_key)
    print(f"    active API key set ({source or 'startup'}).")


if not DRY_RUN:
    set_active_key(API_KEY, source="environment")

# Draft <-> production toggle: DRAFT_MODE=1 renders 360p into isolated dirs.
DRAFT_MODE = os.environ.get("DRAFT_MODE", "").strip().lower() in ("1", "true", "yes")
OUT_DIR = "segments_mirror_json_draft" if DRAFT_MODE else "segments_mirror_json"
FRAMES_DIR = "frames_mirror_json_draft" if DRAFT_MODE else "frames_mirror_json"
RESOLUTION = "360p" if DRAFT_MODE else "1080p"

# Gate render length in seconds.
try:
    GATE_SECONDS = int(float(os.environ.get("GATE_SECONDS", "10")))
except ValueError:
    raise SystemExit(f"GATE_SECONDS must be a number, got {os.environ.get('GATE_SECONDS')!r}")
if GATE_SECONDS < 8:
    raise SystemExit(f"GATE_SECONDS must be >= 8, got {GATE_SECONDS}")

# Join style: 0 = direct hard cuts (the chained renders land so closely the
# cuts read as seamless). XFADE_S=0.5 for short dissolves, 1 for long ones.
XFADE_S = float(os.environ.get("XFADE_S", "0"))

# Cap a run to the first N segments (e.g. MAX_SEGMENTS=4 = cold open check +
# first gates). Unset or 0 = the full run.
try:
    MAX_SEGMENTS = int(os.environ.get("MAX_SEGMENTS", "0") or 0)
except ValueError:
    raise SystemExit("MAX_SEGMENTS must be an integer")

# Cold-open pre-step: generate the black-void loop-point frame first and seed
# segment 0 from its last frame. COLD_OPEN=0 skips it (segment 0 goes fresh).
COLD_OPEN = os.environ.get("COLD_OPEN", "1").strip().lower() not in ("0", "false", "no")

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # embedded / exec contexts without __file__
    _HERE = os.getcwd()

# Canonical behind-view still of the Bridger's look. Attached to every
# generation as an IMAGE (appearance only). This is a still, not a video --
# video references are out of this build. BRIDGER_REF="" disables it.
_BRIDGER_REF_PATH = os.environ.get("BRIDGER_REF", os.path.join(_HERE, "bridger_ref.jpg"))
BRIDGER_REF_B64 = None
if _BRIDGER_REF_PATH:
    try:
        with open(_BRIDGER_REF_PATH, "rb") as _rf:
            BRIDGER_REF_B64 = base64.b64encode(_rf.read()).decode("ascii")
        print(f"    bridger reference still: {_BRIDGER_REF_PATH}")
    except OSError:
        print(f"    bridger reference not found ({_BRIDGER_REF_PATH}) -- continuing without it")

RESUME_EXISTING = True
GENERATE_TIMEOUT_S = 1800
POLL_S = 15
VERBOSE = os.environ.get("VERBOSE", "1").strip() not in ("0", "false", "no")


def log(msg):
    """Timestamped log line (flush immediately so hangs are visible)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vlog(msg):
    """Verbose-only log line."""
    if VERBOSE:
        log(msg)


# ------------------------------------------------------------------ run log
# Developer-level run statistics: one JSON object per line, per run, under
# logs_mirror_json/. Captures what happened (timings, tiers, retries, sizes)
# -- never prompt text.
RUN_LOG_DIR = os.path.join(_HERE, "logs_mirror_json")
RUN_LOG_PATH = None


def rlog_init(config):
    """Open a new per-run JSONL log; writes the run_start event."""
    global RUN_LOG_PATH
    os.makedirs(RUN_LOG_DIR, exist_ok=True)
    RUN_LOG_PATH = os.path.join(RUN_LOG_DIR,
                                f"run_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
    rlog({"event": "run_start", "ts": time.time(), "config": config})
    print(f"    run log: {RUN_LOG_PATH}")


def rlog(event):
    """Append one event dict to the current run log (best-effort)."""
    if not RUN_LOG_PATH:
        return
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")
    except OSError:
        pass


def find_ffmpeg():
    try:
        from imageio_ffmpeg import get_ffmpeg_exe
        return get_ffmpeg_exe()
    except ImportError:
        return shutil.which("ffmpeg")


FFMPEG = find_ffmpeg()
if not FFMPEG:
    print("WARNING: no ffmpeg found (pip install imageio-ffmpeg). "
          "Segments will generate standalone without last-frame chaining.")

# ------------------------------------------------------------------ data tables
# (Paul's canonical data -- act/gate roster, environments, tones, iconography.)

# act: (name, element, gates lo-hi, audio profile)
ACTS = {
    1:  ("ARIES -- The Fire of Ignition",        "Fire",  (1, 6),   "heavy pulsating 40Hz sub-bass rhythm"),
    2:  ("TAURUS -- The Earth of Materiality",  "Earth", (7, 12),  "deep sustained earth drone with low stone-grinding friction"),
    3:  ("GEMINI -- The Air of Duality",        "Air",   (13, 18), "high-frequency rapidly panning binaural whispers"),
    4:  ("CANCER -- The Water of the Abyss",    "Water", (19, 24), "heavy filtered aqua reverb with sub-aquatic hydrostatic swells"),
    5:  ("LEO -- The Fire of Sovereignty",      "Fire",  (25, 30), "majestic brass swells underscored by visceral lion-roar sub-bass"),
    6:  ("VIRGO -- The Earth of Perfection",    "Earth", (31, 36), "ultra-clean exact sine tones oscillating in pure mathematical intervals"),
    7:  ("LIBRA -- The Air of Equilibrium",     "Air",   (37, 42), "wide-spectrum Perfect Fifth synth harmonics at 1.5 ratio"),
    8:  ("SCORPIO -- The Water of Transformation", "Water", (43, 48), "visceral slow 40Hz sub-bass heartbeat pulse"),
    9:  ("SAGITTARIUS -- The Fire of the Seeker", "Fire", (49, 54), "fast-tempo percussive drive with galloping rhythmic transients"),
    10: ("CAPRICORN -- The Earth of Architecture", "Earth", (55, 60), "massive acoustic stone cavern resonance and subterranean sub-drones"),
    11: ("AQUARIUS -- The Air of the Cosmos",   "Air",   (61, 66), "glitching bitcrushed modular synths and digital circuit feedback"),
    12: ("PISCES -- The Water of Dissolution",  "Water", (67, 72), "lush wordless 432Hz synth pad wash slowly pitching down into 40Hz sub-bass"),
}

# (gate, angel, demon, environment beat)
GATES = [
    # ACT 1 -- ARIES
    (1,  "Vehuiah",   "Bael",      "a suspended perfect circle of floating highly polished obsidian and volcanic glass shards erupting with cymatic fire fractals"),
    (2,  "Jeliel",    "Agares",    "twin ignition arcs spiral off the obsidian ring as the second fire node kindles"),
    (3,  "Sitael",    "Vassago",   "cymatic flames braid into triple-helix columns ringing the Bridger"),
    (4,  "Elemiah",   "Gamigin",   "volcanic glass shards ignite in sequence around the circle like a struck bell of fire"),
    (5,  "Mahasiah",  "Marbas",    "a roaring wall of flame parts down the middle as the guardians take their stations"),
    (6,  "Lelahel",   "Valefor",   "the full obsidian circle blazes white-gold as the sixth node locks the Aries fire circuit"),
    # ACT 2 -- TAURUS
    (7,  "Achaiah",   "Amon",      "colossal granite monoliths covered in glowing moss sigils forming an earthly circuit"),
    (8,  "Cahetel",   "Barbatos",  "glowing subterranean moss and deep stone fissures breathing faint green light"),
    (9,  "Haziel",    "Paimon",    "alabaster light channels cutting through bedrock echo vaults"),
    (10, "Aladiah",   "Buer",      "emerald ley-lines projected across slowly rotating basalt gears"),
    (11, "Lauviah",   "Gusion",    "sapphire starlight woven down into iron-ore foundations"),
    (12, "Hahaiah",   "Sitri",     "a marble sanctuary bound to smoldering red obsidian"),
    # ACT 3 -- GEMINI
    (13, "Iezalel",   "Beleth",    "dual mirrored prisms floating on aggressive high-altitude air currents"),
    (14, "Mebahel",   "Leraje",    "crystalline refractions sliced by whistling wind currents"),
    (15, "Hariel",    "Eligos",    "diamond-bright starlight colliding with sweeping iron dust vortices"),
    (16, "Hakamiah",  "Zepar",     "symmetrical light paths intersecting crimson atmospheric distortion"),
    (17, "Lauviah",   "Botis",     "soothing aura flares bound to razor-sharp shear winds"),
    (18, "Caliel",    "Bathin",    "geometric beam alignment meeting shadow-shrouded downdrafts"),
    # ACT 4 -- CANCER
    (19, "Leuviah",   "Sallos",    "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton forming moving seals"),
    (20, "Pahaliah",  "Purson",    "golden underwater arches veiled in drifting motes of light"),
    (21, "Nelchael",  "Marax",     "fluid geometry cutting through iron-infused silt clouds"),
    (22, "Yeiayel",   "Ipos",      "shimmering starlight piercing dark ocean undertows"),
    (23, "Melahel",   "Aim",       "pure aqua-energy distilling against undersea thermal vents"),
    (24, "Haheuiah",  "Naberius",  "a protective bioluminescent sphere anchored to ocean-floor vaults"),
    # ACT 5 -- LEO
    (25, "Nith-Haiah","Glasya-Labolas", "a blinding solar corona filled with spinning gold geometric projections"),
    (26, "Haaiah",    "Bune",      "gold architectural strictures anchoring into smoldering embers"),
    (27, "Yeratel",   "Ronove",    "corona flares locking into magnetic retention rings"),
    (28, "Seheiah",   "Berith",    "cooling golden aura balancing against molten royal flame"),
    (29, "Reiyel",    "Astaroth",  "pure solar sovereign light intersecting eclipsing copper shadow"),
    (30, "Omael",     "Forneus",   "vitalizing solar warmth grounding against cold solar vacuum"),
    # ACT 6 -- VIRGO
    (31, "Lecabel",   "Foras",     "hyper-precise cold sacred geometry and infinite anatomical spine mappings"),
    (32, "Vasariah",  "Asmoday",   "geometric symmetry carving along vertebrae entwined with chthonic horns"),
    (33, "Yehuiah",   "Gaap",      "starlight vertebrae seals binding to subterranean nerve roots"),
    (34, "Lehahiah",  "Furfur",    "cold spine plates aligning with bio-electric energy"),
    (35, "Chavakiah", "Marchosias","gold analytical geometry stabilizing inside skeletal frameworks"),
    (36, "Menadel",   "Stolas",    "a beam of mathematical order anchoring into starlight observatory roots"),
    # ACT 7 -- LIBRA
    (37, "Aniel",     "Phenex",    "unfolding scales of light and absolute mirrored symmetry on the 180-degree halfway axis"),
    (38, "Haamiah",   "Halphas",   "a symmetrical light sanctuary anchored into iron battlements"),
    (39, "Rehael",    "Malphas",   "silver threads of equilibrium meeting obsidian tower geometry"),
    (40, "Ieiazel",   "Raum",      "balanced air beams held inside absorbing shadow vortices"),
    (41, "Hahahel",   "Focalor",   "solar-winged scales balancing against heavy atmospheric pressure"),
    (42, "Mikael",    "Vepar",     "a central axis of pristine light mirroring deep sea-mist"),
    # ACT 8 -- SCORPIO
    (43, "Veuliah",   "Sabnock",   "massive wet obsidian vaults pulsing with a bioluminescent serpentine nervous system"),
    (44, "Yelahiah",  "Shax",      "bioluminescent aqua filaments pulsing through biological stone walls"),
    (45, "Sealiah",   "Vine",      "invigorating aqua currents anchoring into serpentine root systems"),
    (46, "Ariel",     "Bifrons",   "starlight streams balancing against tomb-stone shadows"),
    (47, "Asaliah",   "Vual",      "turquoise clarity illuminating the deep abyssal undertow"),
    (48, "Mihael",    "Haagenti",  "the 28th spoke rim at 240 degrees, wet obsidian giving way to void"),
    # ACT 9 -- SAGITTARIUS
    (49, "Vehuel",    "Crocell",   "an arrow of pure light shooting endlessly through an expanding cosmic Akashic library of flying parchment codices"),
    (50, "Daniel",    "Furcas",    "starlight illuminating ancient codices bound by iron quill marks"),
    (51, "Hahasiah",  "Balam",     "universal truths revealing in golden light through veils of mystery"),
    (52, "Imamiah",   "Alloces",   "forward trajectory of light accelerating through armor-plated momentum"),
    (53, "Nanael",    "Camio",     "celestial wisdom resonating across endless library halls"),
    (54, "Nithael",   "Murmur",    "a starlight throne bound to deep trumpet resonance"),
    # ACT 10 -- CAPRICORN
    (55, "Mebahiah",  "Orobas",    "an ancient mountain archive of deeply carved glowing seals"),
    (56, "Poiel",     "Gremory",   "gold light filling mountain vaults counterbalanced by velvet shadow chambers"),
    (57, "Nemamiah",  "Ose",       "crystalline spires raising toward mountain peaks"),
    (58, "Yeialel",   "Amy",       "iron beams aligning across stone archives bound by subterranean fire"),
    (59, "Harahel",   "Orias",     "cosmic knowledge vaults opening beside star-charting pillars"),
    (60, "Mitzrael",  "Vapula",    "architectural mountain integrity binding to mechanical craftsmanship"),
    # ACT 11 -- AQUARIUS
    (61, "Umabel",    "Zagan",     "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations"),
    (62, "Iah-Hel",   "Valac",     "high-speed data sanctuaries anchored into dragon-circuit foundations"),
    (63, "Anauel",    "Andras",    "collective thought-currents coordinating through static voltage spikes"),
    (64, "Mehiel",    "Flauros",   "protective light-scripts writing across sky firewalls"),
    (65, "Damabiah",  "Andrealphus","crystal-clear data streams transforming through prism matrices"),
    (66, "Manakel",   "Kimaris",   "a neural storm calming into steady logic above server vaults"),
    # ACT 12 -- PISCES
    (67, "Eyael",     "Amdusias",  "an infinite borderless ocean of consciousness dissolving into starlight mist"),
    (68, "Habuhiah",  "Belial",    "primordial waters of rebirth nurturing above seabed bedrock"),
    (69, "Rochel",    "Decarabia", "cosmic memories restoring in glowing tides and star-shadows"),
    (70, "Jabamiah",  "Seere",     "ocean energy transforming into spirit propelled by swift currents"),
    (71, "Haiaiel",   "Dantalion", "divine clarity arming the soul inside infinite mirrors of consciousness"),
    (72, "Mumiah",    "Andromalius","the final node where the borderless ocean ignites back into volcanic glass and cymatic fire fractals"),
]


def act_of(gate):
    for num, (name, el, lo_hi, audio) in ACTS.items():
        lo, hi = lo_hi
        if lo <= gate <= hi:
            return num, name, el, audio
    raise ValueError(gate)


def choir_of(gate):
    """Angel choir derived from gate number (gates 1-8 Seraphim ... 65-72 Angels)."""
    return {
        1: "Seraphim", 2: "Cherubim", 3: "Thrones", 4: "Dominations",
        5: "Powers", 6: "Virtues", 7: "Principalities", 8: "Archangels",
        9: "Angels",
    }[(gate - 1) // 8 + 1]

# ------------------------------------------------------------------ guardian iconography
# Historically-grounded forms: demons per Ars Goetia, angels per their 9 choirs.
# Every entity is towering, visually distinct, elemental, and dynamically raises its talisman.

ANGEL_FORMS = {
    # Choir 1: Seraphim (Fiery Divine Will & Primordial Light)
    "Vehuiah": "a towering, six-winged Seraph of blinding volcanic flame, igniting the primal void and thrusting high a blazing talisman of divine initiation",
    "Jeliel": "a towering, six-winged Seraph woven from soaring thermal firestorms, harmonizing celestial tides and raising aloft a radiant talisman of eternal union",
    "Sitael": "a towering, six-winged Seraph of piercing photon brilliance, laying cosmic foundation stones and brandishing high a burning talisman of architectural construction",
    "Elemiah": "a towering, six-winged Seraph expanding a golden corona of solar wrath, commanding the deep abyss and hoisting a blazing talisman of divine power",
    "Mahasiah": "a towering, six-winged Seraph radiating forge-hot alchemical heat, purifying cosmic discord and thrusting upward a fiery talisman of spiritual rectification",
    "Lelahel": "a towering, six-winged Seraph of blinding starlight and restorative solar flame, casting away darkness and raising high a luminous talisman of miraculous healing",
    "Achaiah": "a towering, six-winged Seraph of subterranean earth-fire, rooted in crystalline bedrock and brandishing aloft a heavy granite talisman of unyielding patience",
    "Cahetel": "a towering, six-winged Seraph wreathed in fertile golden auroras, commanding flourishing cosmic growth and hoisting high a jade-carved talisman of celestial harvest",

    # Choir 2: Cherubim (Wisdom, Four-Faced Tetramorphs & Whirling Wheels)
    "Haziel": "a towering, four-faced Cherub veined with molten gold and lapis lazuli, spinning crystalline wheels and raising aloft an earthen talisman of divine mercy",
    "Aladiah": "a towering, four-faced Cherub carved from adamantine granite, grinding crystalline wheels of light and thrusting high a diamond-bright talisman of restorative grace",
    "Lauviah": "a towering, four-faced Cherub crowned in sapphire starlight, turning heavy planetary wheels and brandishing aloft a meteoric iron talisman of prophetic revelation",
    "Hahaiah": "a towering, four-faced Cherub of translucent alabaster marble, shielding inner sanctuary fires and raising high a pearl-inlaid talisman of mystical refuge",
    "Iezalel": "a towering, four-faced Cherub of prismatic upper-atmosphere zephyrs, whirling gale-force crystalline wheels and hoisting aloft a radiant talisman of sacred fidelity",
    "Mebahel": "a towering, four-faced Cherub of blinding atmospheric lightning, severing untruth with spinning wheels and thrusting upward a crystalline talisman of absolute justice",
    "Hariel": "a towering, four-faced Cherub reflecting diamond-white astral dawns, chiming with infinite choir wheels and raising high an argent talisman of moral purification",
    "Hakamiah": "a towering, four-faced Cherub of roaring martial thunder, aligning celestial shields with turning wheels and brandishing aloft a bronze talisman of chivalric loyalty",

    # Choir 3: Thrones / Ophanim (Cosmic Foundations & Eye-Studded Beryl Wheels)
    "Lauviah (Throne)": "a towering Ophanim of revolving interlocking beryl wheels, burning with thousands of sleepless eyes and raising high an astral talisman of cosmic memory",
    "Caliel": "a towering Ophanim of whirling celestial chrysolite wheels, piercing atmospheric illusions with eye-studded rims and thrusting upward an electrum talisman of absolute truth",
    "Leuviah": "a towering Ophanim submerged in bioluminescent abyssal currents, turning eye-rimmed oceanic wheels and raising aloft a glowing coral talisman of expansive intellect",
    "Pahaliah": "a towering Ophanim burning with theological white flame beneath crushing deep-sea pressure, spinning sacred wheels and hoisting a golden talisman of fervent redemption",
    "Nelchael": "a towering Ophanim tracing astronomical geometry across the heavens, dripping liquid starlight from wheel-rims and brandishing aloft a brass talisman of higher learning",
    "Yeiayel": "a towering Ophanim navigating tempestuous celestial waters, guiding lost travelers with revolving eye-wheels and thrusting high a pearl-encrusted talisman of safe passage",
    "Melahel": "a towering Ophanim distilling pure restorative springwaters, blooming with medicinal flora along turning rims and raising high an emerald talisman of healing waters",
    "Haheuiah": "a towering Ophanim projecting an impenetrable oceanic dome, locking abyssal sanctuary wheels and brandishing aloft a heavy obsidian talisman of exile's protection",

    # Choir 4: Dominations (Sovereign Authority, Solar Crowns & Imperial Scepters)
    "Nith-Haiah": "a towering Domination draped in sovereign solar plasma, commanding secret occult forces with a glowing scepter and raising high a golden talisman of hidden wisdom",
    "Haaiah": "a towering Domination crowned in blinding coronal flares, orchestrating universal statecraft with an imperial staff and thrusting upward an electrum talisman of diplomacy",
    "Yeratel": "a towering Domination of victorious solar radiance, scattering encroaching shadows with sovereign authority and raising aloft a brilliant sunstone talisman of divine liberation",
    "Seheiah": "a towering Domination clad in golden dragon-scale armor, parrying ruinous cataclysms with an imperial scepter and hoisting high a mirror-polished talisman of longevity",
    "Reiyel": "a towering Domination blazing with sacred solar fury, shattering profane idols with royal decree and brandishing aloft a consecrated talisman of spiritual liberty",
    "Omael": "a towering Domination pulsing with generative solar warmth, multiplying the seeds of living creation and raising high a verdant chrysolite talisman of biological fertility",
    "Lecabel": "a towering Domination of laser-sharp geometric precision, dictating celestial vegetation cycles with an iron rod and thrusting upward an emerald talisman of exactitude",
    "Vasariah": "a towering Domination of stern imperial symmetry, weighing royal clemency with an ivory scepter and brandishing aloft a heavy marble talisman of righteous justice",

    # Choir 5: Powers (Martial Enforcers of Law, Chains & Restraining Aegis)
    "Yehuiah": "a towering Power clad in dark starlight plate, anchoring vertebrae of celestial law with glowing chains and raising high an iron talisman of cosmic subordination",
    "Lehahiah": "a towering Power encased in frost-gleaming adamantine armor, subduing rebellious spirits with celestial collars and thrusting upward a platinum talisman of divine obedience",
    "Chavakiah": "a towering Power armored in radiant solar gold, mending shattered ancestral pacts with bonds of light and hoisting aloft an engraved talisman of familial reconciliation",
    "Menadel": "a towering Power in heavy blackened-iron harness, liberating exhausted souls from oppressive toil and brandishing high an unyielding talisman of sanctified labor",
    "Aniel": "a towering Power balancing scales of kinetic celestial wind, shattering karmic chains of tyranny and raising aloft an argent talisman of broken cycles",
    "Haamiah": "a towering Power standing vigilant in ritual-scribed aegis armor, warding celestial sanctuaries with blazing runes and thrusting high a sacred talisman of spiritual protection",
    "Rehael": "a towering Power humming with harmonic frequencies of health, binding corrosive plagues with golden cords and hoisting aloft an amber talisman of filial healing",
    "Ieiazel": "a towering Power releasing concussive shockwaves of purified air, snapping dungeon iron with celestial chains and brandishing high a silver talisman of divine rescue",

    # Choir 6: Virtues (Miracle-Workers, Star Crowns & Elemental Catalysts)
    "Hahahel": "a towering Virtue crowned in blazing stellar constellations, channeling selfless apostolic sacrifice through an altar of light and raising high a consecrated talisman of holy priesthood",
    "Mikael": "a towering Virtue holding the central axis of sovereign harmony, steadying kingdoms amid shifting astral winds and thrusting upward a solar talisman of political order",
    "Veuliah": "a towering Virtue radiating turquoise subterranean luminescence, toppling tyrannical dynasties from the deep earth and raising aloft an orichalcum talisman of prosperous liberation",
    "Yelahiah": "a towering Virtue wreathed in bioluminescent martial wrath, guarding righteous warriors across dark currents and brandishing high a Damascus-steel talisman of karmic victory",
    "Sealiah": "a towering Virtue pulsing like a colossal cosmic heart, reigniting stagnant terrestrial currents with surge-force and hoisting aloft a verdant talisman of restored motion",
    "Ariel": "a towering Virtue parting abyssal ocean trenches, unearthing forgotten treasures of natural wisdom and thrusting high an iridescent mother-of-pearl talisman of secret revelation",
    "Asaliah": "a towering Virtue illuminating the midnight cosmos with contemplative brilliance, praising cosmic design and raising aloft a celestial prism talisman of divine cognition",
    "Mihael": "a towering Virtue radiating warm marine bioluminescence, intertwining divided souls into enduring unity and brandishing high a rose-gold talisman of conjugal harmony",

    # Choir 7: Principalities (Guardians of Nations, Celestial Architects & Scepters)
    "Vehuel": "a towering Principality crowned in galactic star-fire, releasing searing arrows of truth into the void and raising high a scintillating diamond talisman of divine exaltation",
    "Daniel": "a towering Principality crowned in shimmering rhetoric-flame, unfurling radiant parchment scrolls of cosmic law and thrusting upward an inscribed sapphire talisman of eloquent justice",
    "Hahasiah": "a towering Principality crowned in Hermetic alchemical fire, distilling universal panaceas from celestial vessels and raising aloft a smoldering cinnabar talisman of esoteric mysteries",
    "Imamiah": "a towering Principality crowned in incandescent plasma, trampling demonic pride beneath luminous boots and brandishing high a molten bronze talisman of spiritual expiation",
    "Nanael": "a towering Principality crowned in sacred philosophical radiance, contemplating planetary harmonics with celestial compasses and hoisting high an azure talisman of deep contemplation",
    "Nithael": "a towering Principality enthroned upon a pillar of starry marble, shielding emperors and rightful bloodlines while raising aloft an imperial gold talisman of legitimate succession",
    "Mebahiah": "a towering Principality crowned in carved mountain granite, anchoring righteous deeds into terrestrial foundations and thrusting high a lapis-inlaid talisman of lucid morality",
    "Poiel": "a towering Principality crowned in showering auric gold, opening infinite vaults of celestial abundance and brandishing high an opulent jewel-set talisman of boundless providence",

    # Choir 8: Archangels (Cosmic Heralds, Sacred Geometry & Divine Wisdom)
    "Nemamiah": "a towering Archangel forging crystalline spires of war, directing righteous commanders with a star-metal staff and raising high an adamantine talisman of righteous causes",
    "Yeialel": "a towering Archangel of deep resonant basalt, banishing sorrowful afflictions with an unbending rod and thrusting upward a cold meteoric iron talisman of endurance",
    "Harahel": "a towering Archangel unlocking limitless celestial archives, illuminating sacred texts with an ivory staff and hoisting high an inscribed golden talisman of intellectual wealth",
    "Mitzrael": "a towering Archangel mending the shattered geometry of the spirit, driving away internal madness and raising aloft a polished beryl talisman of psychological healing",
    "Umabel": "a towering Archangel weaving lines of astronomical affinity across the night sky, fostering enduring bonds and brandishing high an electrum talisman of sacred friendship",
    "Iah-Hel": "a towering Archangel enveloped in serene cloistered starlight, radiating transcendent wisdom through absolute silence and thrusting upward a crystal talisman of divine illumination",
    "Anauel": "a towering Archangel safeguarding terrestrial voyages and rightful commerce, shielding trade roads with a copper staff and raising high a coin-stamped talisman of material protection",
    "Mehiel": "a towering Archangel inking verses of celestial lightning into the clouds, inspiring grand orators and brandishing aloft a flaming quill-carved talisman of poetic inspiration",

    # Choir 9: Angels (Terrestrial Guardians, Living Waters & Rebirth)
    "Damabiah": "a towering Angel of gushing living headwaters, bestowing boundless oceanic wisdom upon searching souls and raising high a shimmering aquamarine talisman of the eternal spring",
    "Manakel": "a towering Angel diving through abyssal marine depths, interpreting prophetic dreams from dark waters and thrusting upward a polished pearl talisman of good and evil",
    "Eyael": "a towering Angel projecting incandescent mists of solar ascension, transmuting mundane sorrow into celestial joy and hoisting aloft an alchemical gold talisman of sublime transformation",
    "Habuhiah": "a towering Angel breathing fertile dew across barren orchards, healing blighted crops with radiant palms and raising high an emerald talisman of agricultural healing",
    "Rochel": "a towering Angel scouring the cosmic tides for lost treasures, returning stolen heritages to rightful heirs and brandishing aloft an argent talisman of complete restitution",
    "Jabamiah": "a towering Angel churning primordial oceans into celestial essence, resurrecting corrupted matter into perfection and thrusting upward a glowing chrysolite talisman of universal regeneration",
    "Haiaiel": "a towering Angel girded in blinding celestial armor, arming the pure of heart with flaming blades and raising high a luminous steel talisman of divine warfare and peace",
    "Mumiah": "a towering Angel of final oceanic dissolution, weaving twilight endings into triumphant rebirth and hoisting aloft an alpha-and-omega inscribed talisman of cosmic transformation",
}

DEMON_FORMS = {
    # 1 - 10
    "Bael": "a towering triple-headed king with heads of toad, cat, and crowned man on spider legs, rasping hoarsely and raising high a heavy iron talisman of invisibility",
    "Agares": "a towering, ancient nobleman riding an earth-shaking crocodile with a goshawk upon his fist, halting retreating armies and thrusting upward a smoldering copper talisman of language",
    "Vassago": "a towering infernal prince of radiant angelic countenance with burning sapphire eyes, declaring things past and future while raising aloft a crystalline talisman of prophecy",
    "Gamigin": "a towering spectral horse that shifts into an ashen phantom with a thunderous voice, summoning deceased souls and hoisting high a black-iron talisman of the dead",
    "Marbas": "a towering mechanical-maned lion with roaring furnace-fire eyes, answering truthfully of hidden secrets and brandishing high an iron-wrought talisman of mechanical craft and healing",
    "Valefor": "a towering prowling lion with the head of an ass-eared brigand, slipping through stealing shadows and raising high a tarnished silver talisman of thievery",
    "Amon": "a towering wolf with a serpent's tail vomiting dark obsidian flames, reconciling broken friendships and brandishing aloft a dense lead talisman of feuds and alliances",
    "Barbatos": "a towering horned hunter-lord accompanied by phantom kings and horn blasts, understanding animal tongues and raising high a moss-grown copper talisman of hidden treasure",
    "Paimon": "a towering, crowned monarch riding an imperial dromedary amid clashing cymbals and deafening trumpets, raising aloft an opulent gold talisman of spirit dominion",
    "Buer": "a towering wheel of five human legs radiating from a horned lion's head, spinning over bedrock and thrusting upward a bronze talisman of natural philosophy and healing",

    # 11 - 20
    "Gusion": "a towering cynocephalic ape-hound of weathered meteoric stone, discerning past, present, and future while brandishing aloft a heavy leaden talisman of restored honors",
    "Sitri": "a towering leopard with the wings of a massive griffin, smoldering with incandescent crimson passion and raising high a glowing copper talisman of inflamed desire",
    "Beleth": "a towering, terrible king riding a pale warhorse, preceded by blasting brass trumpets that shake the earth and thrusting upward a silver talisman of compelled love",
    "Leraje": "a towering emerald-clad archer drawing a massive recurve bow that whistles with shearing winds, raising aloft a green-patinated copper talisman of gangrenous wounds",
    "Eligos": "a towering ghostly knight clad in shining spectral plate, bearing lance, standard, and serpent while hoisting high a polished copper talisman of martial victory",
    "Zepar": "a towering crimson-armored soldier whose curved broadsword radiates scorched heat-distortion, inflaming minds and brandishing aloft a blood-rusted iron talisman of sterile lust",
    "Botis": "a towering horned viper shifting into a dreadful warrior with great tusks and drawn sword, thrusting high an electrum talisman of reconciled enemies",
    "Bathin": "a towering muscular giant with a serpent's tail riding a pale steed through planar portals, raising aloft a pale brass talisman of instantaneous transit",
    "Sallos": "a towering gallant soldier in a ducal coronet riding a leviathan crocodile with serene dignity, raising high a gleaming silver talisman of peaceful love",
    "Purson": "a towering lion-faced king clutching a hissing viper, riding a monstrous bear through blaring trumpets and hoisting high a radiant gold talisman of hidden treasures",

    # 21 - 30
    "Marax": "a towering bull with the bearded face of an ancient sage, expounding astronomy and herbal virtues while brandishing aloft a heavy iron talisman of liberal sciences",
    "Ipos": "a towering, uncanny angel with a roaring lion's head, goose's feet, and a hare's tail, thrusting upward a bright brass talisman of wit and valor",
    "Aim": "a towering three-headed nobleman riding a viper with a blazing firebrand, setting citadels ablaze and raising high a smoldering bronze talisman of consuming intellect",
    "Naberius": "a towering three-headed black crane shrieking in a raspy voice, walking through chthonic vaults and raising aloft a dark copper talisman of restored dignity",
    "Glasya-Labolas": "a towering winged bloodhound with broad griffin wings, trailing a crimson mist of bloodshed and brandishing high a bloodstained talisman of slaughter and prophecy",
    "Bune": "a towering three-headed dragon speaking in high resonant tones from sepulchral vaults, hoisting high a lustrous copper talisman of necromantic wealth",
    "Ronove": "a towering shadow-cloaked monster leaning upon a reaper's staff, harvesting weary souls while raising high an iron talisman of persuasive rhetoric and languages",
    "Berith": "a towering red-and-gold soldier upon a crimson warhorse, wearing a crown of molten flame and thrusting upward a blazing gold talisman of transmuting metals",
    "Astaroth": "a towering sorrowful fallen angel riding an infernal dragon with a coiled viper in hand, brandishing aloft a poisonous copper talisman of past falls",
    "Forneus": "a towering abyssal sea monster gliding through cold oceanic depths, granting eloquence across tongues and raising high a silver talisman of oceanic rhetoric",

    # 31 - 40
    "Foras": "a towering muscular sage carved from ancient gnarled oak, teaching logic, ethics, and herbology while hoisting high an earthen brass talisman of wisdom and longevity",
    "Asmoday": "a towering three-headed emperor vomiting furnace flames from an infernal dragon, brandishing aloft a molten gold talisman of invincible dominion and wrath",
    "Gaap": "a towering majestic emperor guiding four cardinal kings through the subterranean void, transporting mortals swiftly and raising high an inscribed leaden talisman of philosophy",
    "Furfur": "a towering colossal hart with a blazing lightning-wreathed tail, summoning roaring thunderheads and thrusting upward a crackling copper talisman of tempests and love",
    "Marchosias": "a towering armored wolf with griffin wings and a serpent's tail, vomiting columns of fire and brandishing high a gleaming silver talisman of martial fidelity",
    "Stolas": "a towering crowned raven with infinitely deep starlit eyes, perched upon celestial astrolabes and raising high an etched silver talisman of astronomy and herbs",
    "Phenex": "a towering phoenix wreathed in black-and-crimson flames, singing poetic verses with a haunting child's voice and hoisting aloft a smoldering copper talisman of melody",
    "Halphas": "a towering dark iron stock-dove shrieking with a hoarse voice, raising impenetrable fortress walls and thrusting upward a heavy brass talisman of siege warfare",
    "Malphas": "a towering soot-black raven shifting into a grim architect who shatters enemy citadels, raising aloft a dense leaden talisman of masonry and deception",
    "Raum": "a towering monstrous crow shifting into an armored plunderer, toppling dynasties and destroying cities while brandishing high a fractured copper talisman of ruin and theft",

    # 41 - 50
    "Focalor": "a towering winged warrior commanding hurricane winds and churning ocean whirlpools to sink warships, raising high a verdigris copper talisman of maritime wrath",
    "Vepar": "a towering ferocious mermaid rising from foaming breakers, guiding dread armadas with sea-fogs and hoisting aloft a tarnished silver talisman of nautical pestilence",
    "Sabnock": "a towering lion-headed soldier in heavy armor riding a pale horse, striking foes with gangrenous wounds and thrusting upward an iron talisman of fortress architecture",
    "Shax": "a towering dark stork-demon shrieking in a raspy voice, plundering mortal sight, hearing, and understanding while brandishing aloft a black talisman of sensory theft",
    "Vine": "a towering crowned lion riding a jet-black warhorse, brandishing a viper while summoning howling squalls and raising high an opulent gold talisman of exposed sorcery",
    "Bifrons": "a towering shape-shifting shade lighting spectral grave-lanterns over crumbling tombs, thrusting upward a weathered copper talisman of necromantic geometry and astronomy",
    "Vual": "a towering desert dromedary shifting into a fierce warrior speaking ancient Egyptian mysteries, raising high a hammered copper talisman of enduring affection",
    "Haagenti": "a towering winged golden bull shifting into a man, transmuting lead to gold and water to wine while hoisting high a molten gold talisman of supreme alchemy",
    "Crocell": "a towering luminous fallen angel murmuring with the sound of rushing waters, revealing steaming mineral baths and brandishing aloft a geometric brass talisman of secret sciences",
    "Furcas": "a towering white-bearded sage riding a skeletal pale steed, wielding a razor-sharp barbed spear and thrusting upward a heavy lead talisman of logic and pyromancy",

    # 51 - 60
    "Balam": "a towering three-headed king with flaming eyes, riding a raging bear with a goshawk on his fist and raising high an engraved gold talisman of past and future",
    "Alloces": "a towering knight with a fiery lion's face and scarlet eyes, riding an armored warhorse with crushing momentum and brandishing aloft a copper talisman of astronomy",
    "Camio": "a towering singing thrush stepping from burning embers to become a swordsman answering in ash, hoisting high a silver talisman of animal omens",
    "Murmur": "a towering crowned ducal warrior riding a ferocious griffin, preceded by ghostly horn blasts and raising aloft an ancient copper talisman of necromantic philosophy",
    "Orobas": "a towering magnificent horse shifting into a noble man, declaring truths of divinity without deceit and thrusting upward a sacred silver talisman of virtuous favor",
    "Gremory": "a towering all-seeing duchess wearing a jeweled coronet, riding an imperial camel through velvet shadows and brandishing aloft an opulent copper talisman of hidden treasures",
    "Ose": "a towering spotted leopard shifting into a crowned illusionist who plunges minds into madness, raising high an etched silver talisman of mental transfiguration",
    "Amy": "a towering blazing column of roaring fire resolving into an astrological magus, thrusting upward a glowing brass talisman of guarded spirit-treasures",
    "Orias": "a towering lion riding a serpent-tailed stallion, holding two hissing vipers toward starry heavens and raising high a silver talisman of astrological wisdom",
    "Vapula": "a towering lion with broad griffin wings, guiding artisans in intricate mechanics and treatises while brandishing aloft a polished copper talisman of sacred craftsmanship",

    # 61 - 72
    "Zagan": "a towering bull with griffin wings, transmuting water to wine, blood to oil, and base metals into coin while hoisting high a radiant gold talisman of transmutation",
    "Valac": "a towering winged angelic youth with blazing eyes, riding a colossal two-headed dragon through cavernous fissures and thrusting upward a serpentine talisman of hidden treasures",
    "Andras": "a towering angelic warrior with the head of a black raven, riding a ravenous midnight wolf with drawn saber and raising aloft a gleaming steel talisman of discordant slaughter",
    "Flauros": "a towering terrifying leopard leaping into a flame-eyed warrior who incinerates foes with divine fury, brandishing high a smoldering copper talisman of consuming fire",
    "Andrealphus": "a towering peacock shrieking with earsplitting pride, shifting into an architect who measures the cosmos and hoisting high an engraved silver talisman of mensuration",
    "Kimaris": "a towering heroic warrior in blackened harness riding a colossal black charger across spirit realms, raising high a bronze talisman of hidden grammar and rhetoric",
    "Amdusias": "a towering unicorn bending great forest oaks as invisible orchestral trumpets resound, thrusting upward a polished copper talisman of symphonic command",
    "Belial": "a towering pair of radiant fallen angels riding a chariot of roaring fire, speaking with deceptive honeyed grace and raising aloft an imperial gold talisman of sovereign favor",
    "Decarabia": "a towering five-pointed pentacle-star shifting into an enchanter surrounded by flocks of singing birds, brandishing high a lustrous silver talisman of herbal lore",
    "Seere": "a towering breathtaking youth riding a winged pegasus, moving across the world in the blink of an eye and hoisting high an electrum talisman of instantaneous abundance",
    "Dantalion": "a towering nobleman of infinite shifting faces, holding an open grimoire of human desires and thrusting upward a dark copper talisman of telepathic sway",
    "Andromalius": "a towering stern judge holding a colossal writhing serpent, tracking down thieves and punishing the wicked while raising high an iron talisman of absolute retribution",
}

# Occurrence index per gate for angels whose name repeats in the roster
# (Lauviah appears at gates 11 and 17); parallel to GATES.
_ANGEL_SEEN = {}
ANGEL_OCCURRENCE = []
for _gg, _aa, _dd, _ee in GATES:
    ANGEL_OCCURRENCE.append(_ANGEL_SEEN.get(_aa, 0))
    _ANGEL_SEEN[_aa] = _ANGEL_SEEN.get(_aa, 0) + 1
del _ANGEL_SEEN, _gg, _aa, _dd, _ee


def angel_form(angel, occurrence=0):
    """Expanded traditional form for an angel; occurrence disambiguates repeats."""
    key = angel
    if occurrence:
        alts = [k for k in ANGEL_FORMS if k != angel and k.startswith(angel)]
        if alts:
            key = alts[occurrence - 1]
    return ANGEL_FORMS[key]


def demon_form(demon):
    """Expanded traditional Goetic form for a demon."""
    return DEMON_FORMS[demon]


# Physical gate architecture per element -- the thing he actually walks through.
THRESHOLDS = {
    "Fire":  "a towering arch of black volcanic glass wreathed in slow-burning cymatic flame",
    "Earth": "a moss-covered stone trilithon humming with a low resonant tone",
    "Air":   "a lattice of prismatic light refracting the sky into slow ribbons",
    "Water": "an arch of living coral and breathing bioluminescent membrane",
}

# Wheel tones: each act resolves to a pure tone a perfect fifth (1.5x) above
# the last, folded into the 40-80Hz sub range -- the circle of fifths walking
# the 12 signs. Every segment OPENS continuing the previous segment's tone and
# CLOSES in a calm moment at its tail tone -- the tail tone IS the handoff.
WHEEL_TONES = {
    1: 40, 2: 60, 3: 45, 4: 68, 5: 51, 6: 76,
    7: 57, 8: 43, 9: 64, 10: 48, 11: 72, 12: 54,
}

GLYPHS = {
    1:  ("Aries",       "the Ram's glyph, two bold curved horns"),
    2:  ("Taurus",      "the Bull's glyph, a horned circle"),
    3:  ("Gemini",      "the Twins' glyph, twin pillars joined by arcs"),
    4:  ("Cancer",      "the Crab's glyph, twin spiraling whorls"),
    5:  ("Leo",         "the Lion's glyph, a single sweeping mane-curve"),
    6:  ("Virgo",       "the Maiden's glyph, an intricate looping sigil"),
    7:  ("Libra",       "the Scales' glyph, a balanced horizon over a rising curve"),
    8:  ("Scorpio",     "the Scorpion's glyph, a barbed tail striking upward"),
    9:  ("Sagittarius", "the Archer's glyph, a drawn bow loosing an arrow skyward"),
    10: ("Capricorn",   "the Sea-Goat's glyph, a horned crown diving into a fishtail curl"),
    11: ("Aquarius",    "the Water-Bearer's glyph, twin zigzag waves"),
    12: ("Pisces",      "the Fishes' glyph, two mirrored arcs bound by a single cord"),
}

ACT_ENVS = {
    1:  "a suspended perfect circle of floating polished obsidian and volcanic glass shards erupting with cymatic fire fractals",
    2:  "colossal monolithic stone temples covered in ancient glowing moss-covered sigils",
    3:  "dual mirrored prisms floating on aggressive high-altitude air currents",
    4:  "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton",
    5:  "a blinding solar corona filled with spinning gold geometric projections",
    6:  "hyper-precise cold sacred geometry and infinite anatomical spine mappings",
    7:  "unfolding scales of light and absolute mirrored symmetry",
    8:  "a massive obsidian gateway pulsing with a bioluminescent serpentine nervous system",
    9:  "an arrow of pure light shooting through an expanding cosmic Akashic library",
    10: "an ancient mountain archive of deeply carved glowing seals",
    11: "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations",
    12: "an infinite borderless ocean of consciousness dissolving into starlight mist",
}

DISSOLVE = {
    "Fire":  "dying cymatic flames and cooling volcanic glass",
    "Earth": "crumbling stone dust and fading moss-light",
    "Air":   "thinning high-altitude currents and shattering prisms",
    "Water": "receding bioluminescent waters and dissolving mist",
}

GLYPH_FORMATION = {
    "Fire":  "forged from living flame and volcanic glass",
    "Earth": "carved from moss-covered living stone",
    "Air":   "woven from prismatic light and high wind",
    "Water": "condensed from bioluminescent water and living coral",
}

IGNITION = {
    "Fire":  "cymatic flames erupt outward, igniting",
    "Earth": "stone dust settles and monoliths rise, revealing",
    "Air":   "the air thins into high currents, revealing",
    "Water": "bioluminescent waters flood inward, revealing",
}

# ------------------------------------------------------------------ JSON builders
# Every Omni request is ONE JSON document: `meta` (standing rules + state
# variables, substituted from the environment at run time) + `segment` (every
# variable for this gate). The JSON text is the entire text part of the
# request -- no markdown, no mixed prose.


def build_meta():
    """Standing rules + run-time state variables. Identical in every request."""
    return {
        "schema": SCHEMA_VERSION,
        "state": {
            "model": MODEL_ID,
            "duration_s": GATE_SECONDS,
            "resolution": RESOLUTION,
            "draft_mode": DRAFT_MODE,
            "stride_cycle_s": 1.2,
            "tempo_bpm": 100,
            "forward_speed_m_s": 1.2,
            "camera": "continuous unbroken over-the-shoulder rear tracking dolly",
            "lens": "35mm T1.3 anamorphic",
        },
        "rules": {
            "continuity": "every segment begins on the last frame of the previous segment, ends at the seam to the next gate in mid-motion",
            "appearance": "the World Bridger's appearance is locked to the bridger_ref image -- reproduce his avatar in the new last feame reference rendering only his body from it exactly, never restyle or reinterpret it; airborne ash and embers are environment, never markings on his robes other than glowing golden runes and basalt staff",
            "behavior": "at the center of the scene 5s mark he should continue his pace as he looks at the massive deities ahead. He raises his staff his voice booms out the names of the angel and demon who immediately bow in submission having been defeated by his name magic",
            "behavior_order": "angel first, then demon -- never swapped, no text overlays and both names should be said within 1s of eachother at the halfway point",
            "pacing": "stride is metronome-locked at 100 BPM, 1.2 m/s, identical in every shot -- he never walks faster, never slows down, never hurries a gate crossing",
            "motion": "the dolly never stops its forward drift; the shot flows continuously first frame to last;",
            "heading": "he walks dead straight ahead at all times -- spine aligned to his forward axis, never drifting, leaning, or cocking; the naming beat is the only head motion in the traversal",
            "segment_boundary": "every segment ends mid-stride, mid-dolly, exactly as the next one begins in the center of the gates doorway, so the hard cut between them is invisible and audio should seam perfectly",
            "camera_lock": "locked over-the-shoulder rear tracking; forward motion only; no cuts, no angle changes, no unscripted camera moves",
            "text": "zero text overlays of any kind -- everything visible must exist physically inside the scene never any text",
            "audio": "full diegetic audio is rendered: the act's audio bed, the wheel-tone handoff, and the two guardian names spoken by the Bridger -- the ONLY words ever spoken in the traversal",
        },
    }


def build_audio(act_audio, tone, spoken_names):
    """Full-audio block: act bed + wheel-tone handoff + the two spoken names."""
    return {
        "mode": "full_diegetic_audio",
        "open": "already continuing the previous segment's tail tone -- no click, no jump",
        "bed": act_audio,
        "spoken": [
            {"at": "naming beat", "speaker": "the World Bridger", "line": name}
            for name in spoken_names
        ],
        "speech_rule": "the two guardian names are the ONLY words spoken in the entire traversal",
        "close": f"all sound settles into a calm pure {tone}Hz wheel tone, held steady through the final seconds -- this is the handoff; the next segment picks up exactly from this tone. Render the tail as a seamless continuation, never an ending.",
        "forbidden": ["narration", "voiceover", "host commentary", "music", "text", "non-diegetic sound effects"],
    }


def build_naming_beat(angel, demon):
    """The simplified naming beat. No face is ever mentioned."""
    return {
        "at": "the threshold, mid-segment",
        "actions": [
            {"actor": "bridger", "do": "raise_staff"},
            {"actor": "bridger", "do": "nod", "toward": "left", "speak": angel},
            {"actor": "bridger", "do": "nod", "toward": "right", "speak": demon},
            {"actor": angel, "do": "bow"},
            {"actor": demon, "do": "bow"},
            {"actor": "guardians", "do": "submit and acknowledge"},
        ],
    }


def gate_segment(i, gate, angel, demon, env, element, next_env):
    """JSON variable block for one gate segment."""
    act_num, act_name, _el, act_audio = act_of(gate)
    tone = WHEEL_TONES[act_num]
    mid = GATE_SECONDS // 2
    seg = {
        "id": f"gate_{gate:02d}",
        "kind": "gate",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": act_name,
            "element": element,
            "wheel_tone_hz": tone,
        },
        "guardians": {
            "left": {
                "name": angel,
                "choir": choir_of(gate),
                "form": angel_form(angel, ANGEL_OCCURRENCE[i]),
            },
            "right": {
                "name": demon,
                "order": "Ars Goetia",
                "form": demon_form(demon),
            },
        },
        "environment": env,
        "threshold": THRESHOLDS[element],
        "next_environment": next_env,
        "beats": {
            "approach": f"the threshold looms ahead -- {THRESHOLDS[element]} -- with {angel} gathering at its left pillar and {demon} at its right pillar",
            "naming": build_naming_beat(angel, demon),
            "naming_at_s": f"around {mid - 1}-{mid + 1}s",
        },
        "audio": build_audio(act_audio, tone, [angel, demon]),
    }
    if gate == 48:
        seg["beats"]["crossing"] = (
            "at the 28th spoke rim the Bridger strikes his staff down on the stone -- "
            "a golden 108-degree geodesic Zen Shield flares to life around him, "
            "shattering the matrix wall. Fully solid and opaque, he strides on as the "
            "camera turns 90 degrees left off the zodiac track into the zero-gravity "
            "Ophiuchus Void (the single scripted camera move of the traversal), and "
            "all audio drops into absolute dead silence."
        )
        # The breach ends in dead silence, not a wheel tone -- the Act 9
        # transition opens with "the sound rises from absolute silence".
        seg["audio"]["close"] = (
            "absolute dead silence -- the breach ends in silence, not a wheel tone. "
            "Render the tail as a seamless continuation into silence, never an ending."
        )
    elif gate == 72:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the final gate, fully solid and opaque -- he "
            "does not fade, dissolve, or vanish -- as the borderless ocean detonates "
            "into white-gold ignition: volcanic glass and cymatic fire fractals blooming "
            "outward. The 360-degree loop closes seamlessly back into the Gate 1 threshold."
        )
    else:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the gate plane on foot between the parted "
            "guardians -- fully solid and opaque, he does not fade, dissolve, or vanish "
            f"-- and the camera follows him through; beyond the far plane, {next_env} "
            "already resolving through the shimmer."
        )
    return seg


def transition_segment(act_num, is_first):
    """JSON variable block for one act-transition segment."""
    sign, glyph = GLYPHS[act_num]
    _n, _name, element, act_audio = act_of(act_num * 6 - 5)  # first gate of the act
    tone = WHEEL_TONES[act_num]
    seg = {
        "id": f"act_{act_num:02d}_transition",
        "kind": "act_transition",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": f"{sign} transition",
            "element": element,
            "wheel_tone_hz": tone,
        },
        "glyph": {
            "sign": sign,
            "description": glyph,
            "formation": f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}, hanging in the space and slowly rotating, lit from within",
        },
        "elemental_shift": {
            "outgoing": DISSOLVE[act_of((act_num - 1) * 6)[2]] if not is_first else "dying cymatic flames and cooling volcanic glass",
            "incoming": f"{IGNITION[element]} {ACT_ENVS[act_num]}",
        },
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken -- no guardians stand at this threshold",
    }
    if is_first:
        seg["scene"] = (
            "opening cold: the camera is already locked over the shoulder of the World "
            "Bridger as he strides forward through dying cymatic flames and cooling "
            "volcanic glass, the old element dying away. Golden wireframe 72-gate rings "
            "and glowing cuneiform code matrices float in zero gravity around him. Ahead, "
            f"the zodiac glyph of {sign} -- {glyph} -- materializes. The camera passes "
            f"beneath the glyph and {IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["open"] = "the loop closes here -- open continuing the calm 54Hz wheel tone the final Pisces gate left behind"
        seg["audio"]["spoken"] = []
    else:
        prev_element = act_of((act_num - 1) * 6)[2]
        prev_audio = act_of((act_num - 1) * 6)[3]
        prev_tone = WHEEL_TONES[act_num - 1]
        seg["scene"] = (
            "starting exactly from the first frame -- same camera, same forward drift, no "
            f"cut -- the old world falls behind. The camera drifts through {DISSOLVE[prev_element]} "
            f"as the old element dies away. Ahead, the zodiac glyph of {sign} -- {glyph} -- "
            f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}. The Bridger "
            f"walks on beneath it, staff in hand. As the camera passes beneath the glyph, "
            f"{IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["spoken"] = []
        if act_num == 9:
            # Act 9 follows gate 48's breach, which ends in dead silence, not a tone.
            seg["audio"]["open"] = "the sound rises from absolute silence"
        else:
            seg["audio"]["open"] = (
                f"gliding continuously from the fading {prev_tone}Hz wheel tone -- a smooth "
                f"pitch glide, no click, no jump -- as {prev_audio} dissolves into {act_audio}"
            )
    return seg


def cold_open_segment():
    """JSON variable block for the cold-open pre-step: black void, him walking.

    Generated FIRST; its last frame seeds segment 0. This is the loop point --
    gate 72's tail walks back into this frame.
    """
    return {
        "id": "cold_open",
        "kind": "cold_open",
        "shot": "single unbroken shot",
        "note": "the loop point and the film's first frame -- a deliberate frame, never an arbitrary fresh generation",
        "scene": (
            "pure infinite solid matte black background (#000000) -- absolute zero "
            "background geometry, no pillars, no fire, no ground textures, no horizon. "
            "Only the World Bridger exists in the frame: dead-center rear view, mid-stride, "
            "carved basalt staff in his RIGHT hand, walking dead straight ahead along the "
            "central axis at his metronomic 100 BPM pace. No guardians, no naming beat."
            "hood always up, black wool, crimson lining and glowing cuneiform glyphs and runes."
        ),
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken",
        "audio": {
            "mode": "full_diegetic_audio",
            "open": "continuing the calm 54Hz wheel tone the final Pisces gate leaves behind -- the loop closes here",
            "bed": "the near-silence of the void under a faint sub-bass",
            "spoken": [],
            "speech_rule": "no words are spoken in the cold open",
            "close": "steady and unresolved -- segment 0 continues it; the old fire world ignites out of the dark",
            "forbidden": ["narration", "voiceover", "host commentary", "music", "non-diegetic sound effects"],
        },
    }


def build_request(segment, image_names):
    """Assemble the full JSON request document for one generation."""
    sources = {}
    if "bridger_ref" in image_names:
        sources["bridger_ref"] = {
            "image": image_names["bridger_ref"],
            "role": "appearance reference for the World Bridger -- appearance ONLY; it does not define the environment, pose, lighting, or scene",
        }
    if "first_frame" in image_names:
        sources["first_frame"] = {
            "image": image_names["first_frame"],
            "role": "the last frame of the previous segment -- begin on this exact frame and continue its motion, audio and lighting, and scene seamlessly",
        }
    return {
        "request": "generate_video",
        "engine": "gemini_omni",
        "meta": build_meta(),
        "sources": sources,
        "segment": segment,
    }


SEGMENTS = []
_prev_act = None
for i, (g, a, d, env) in enumerate(GATES):
    act_num = act_of(g)[0]
    if act_num != _prev_act:
        SEGMENTS.append({
            "key": f"act{act_num:02d}_transition",
            "segment": transition_segment(act_num, is_first=(len(SEGMENTS) == 0)),
        })
        _prev_act = act_num
    SEGMENTS.append({
        "key": f"gate{g:02d}",
        "segment": gate_segment(i, g, a, d, env, act_of(g)[2],
                                GATES[i + 1][3] if i + 1 < len(GATES) else None),
    })

assert len(SEGMENTS) == 84

# ------------------------------------------------------------------ inputs & API
# Stills only: the bridger_ref still and the first-frame still. No video
# references in this build.


def to_bytes(data):
    if isinstance(data, str):
        return base64.b64decode(data)
    return bytes(data)


def build_inputs(segment, first_frame_b64):
    """Build the Omni input list: [bridger_ref still?, first_frame still?, JSON text].

    The text part is the ENTIRE prompt as one JSON document -- meta + sources
    + segment. Nothing is markdown, nothing is mixed prose.
    """
    contents = []
    image_names = {}
    n_img = 0

    def add_image(b64, mime="image/png"):
        nonlocal n_img
        n_img += 1
        contents.append({"type": "image", "data": b64, "mime_type": mime})
        return f"Image{n_img}"

    if BRIDGER_REF_B64:
        image_names["bridger_ref"] = add_image(BRIDGER_REF_B64, "image/jpeg")
    if first_frame_b64:
        image_names["first_frame"] = add_image(first_frame_b64)

    request = build_request(segment, image_names)
    text = json.dumps(request, indent=2)
    contents.append({"type": "text", "text": text})

    if len(contents) == 1:
        return text
    return contents


def get_output_video(interaction):
    """Prefer .output_video, fall back to manual step traversal."""
    out = getattr(interaction, "output_video", None)
    if out is not None and (getattr(out, "data", None) or getattr(out, "uri", None)):
        return out
    for step in reversed(getattr(interaction, "steps", []) or []):
        if getattr(step, "type", None) == "model_output" and getattr(step, "content", None):
            for part in reversed(step.content):
                if getattr(part, "type", None) == "video":
                    return part
    return None


def download_uri(uri, dest_path, timeout_s=1800, poll_s=10):
    """Poll the hosted file until ACTIVE, then download it."""
    file_id = uri.rstrip("/").split("/")[-1].split(":")[0].split("?")[0]
    name = f"files/{file_id}"
    deadline = time.time() + timeout_s
    t0 = time.time()
    vlog(f"    POLL {name} every {poll_s}s (timeout {timeout_s}s)")
    while time.time() < deadline:
        elapsed = time.time() - t0
        try:
            info = client.files.get(name=name)
            state = getattr(info.state, "name", str(info.state))
            if state == "ACTIVE":
                log(f"    poll +{elapsed:.0f}s: {state} -- downloading")
                break
            if state == "FAILED":
                raise RuntimeError("video generation failed (file state FAILED)")
            log(f"    poll +{elapsed:.0f}s: hosted file state: {state}")
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"    poll +{elapsed:.0f}s: waiting on hosted file ({exc.__class__.__name__}: {str(exc)[:120]})")
        time.sleep(poll_s)
    else:
        raise TimeoutError(f"timed out after {timeout_s}s waiting for hosted video file")
    data = client.files.download(file=uri)
    with open(dest_path, "wb") as f:
        f.write(to_bytes(data))
    size = os.path.getsize(dest_path) / 1e6
    log(f"    saved {dest_path} ({size:.1f} MB)")


def _norm(exc):
    # Normalize underscores so RATE_LIMIT_EXCEEDED / quota_exceeded style
    # status enums can't slip past the classifiers.
    return re.sub(r"[_\s]+", "", f"{type(exc).__name__} {exc}".lower())


def is_quota_error(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "429", "resourceexhausted", "ratelimitexceeded", "quotaexceeded",
        "exceededyourquota", "toomanyrequests", "dailylimitexceeded",
    ))


def is_auth_error(exc):
    t = _norm(exc)
    return (not is_quota_error(exc)) and any(k in t for k in (
        "401", "invalidapikey", "apikeyinvalid", "apikeynotvalid",
        "apikeyisinvalid", "unauthorized", "permissiondenied",
    ))


def is_content_refusal(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "prohibitedcontent", "prohibiteduse", "contentguidelines",
        "blockeddueto", "violatedgoogle",
        "likeness", "likenesses", "realperson", "realpeople",
        "contentblocked", "inputblocked",
    ))


def prompt_for_new_key(reason):
    print()
    print("=" * 64)
    print(f"  KEY ISSUE MID-RUN: {reason}")
    print("  Paste another Google AI Studio key to keep rendering,")
    print("  or type 'skip' to skip this segment, 'quit' to stop the run.")
    print("=" * 64)
    try:
        new_key = getpass.getpass("  New GEMINI_API_KEY (hidden input): ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise KeyboardInterrupt
    if not new_key:
        return None
    low = new_key.lower()
    if low in ("skip", "s"):
        return "skip"
    if low in ("quit", "q", "exit"):
        raise KeyboardInterrupt
    return new_key


def generate_segment(seg, first_frame_b64, dest_path):
    """Generate one Omni segment from its JSON request. Returns True/False."""
    mode = "chained" if first_frame_b64 else "fresh"
    t0 = time.time()
    log(f"[{seg['key']}] generating ({RESOLUTION}, {mode})...")
    inputs = build_inputs(seg["segment"], first_frame_b64)
    txt = inputs if isinstance(inputs, str) else inputs[-1]["text"]
    if isinstance(inputs, str):
        parts = [f"json:{len(inputs)} chars"]
    else:
        parts = []
        for p in inputs:
            if "text" in p:
                parts.append(f"json:{len(p['text'])} chars")
            elif p.get("type") == "image":
                parts.append(f"image:{len(p.get('data') or '')/1e6:.1f} MB b64 {p.get('mime_type', '?')}")
            else:
                parts.append(str(p.get("type")))
    vlog(f"    REQUEST -> model={MODEL_ID} resolution={RESOLUTION} delivery=uri")
    vlog(f"    REQUEST parts: {', '.join(parts)}")
    if VERBOSE:
        # In verbose mode show the JSON skeleton (keys only) -- full JSON is
        # in --dry-run output.
        req = json.loads(txt)
        vlog(f"    REQUEST json: meta.rules={len(req['meta']['rules'])} rules, "
             f"sources={list(req['sources'])} segment={req['segment']['id']}")
    log(f"    sending interactions.create ...")
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=inputs,
        response_format={"type": "video", "resolution": RESOLUTION, "delivery": "uri"},
    )
    log(f"    RESPONSE in {time.time()-t0:.1f}s | interaction id: {interaction.id}")
    video = get_output_video(interaction)
    if video is None:
        log("    done, but no video in response -- recording as failed.")
        return False
    if getattr(video, "uri", None):
        vlog(f"    RESPONSE video: uri={video.uri}")
        download_uri(video.uri, dest_path)
    elif getattr(video, "data", None):
        vlog(f"    RESPONSE video: inline data ({len(to_bytes(video.data))/1e6:.1f} MB)")
        with open(dest_path, "wb") as f:
            f.write(to_bytes(video.data))
        log(f"    saved {dest_path}")
    else:
        log("    done, but no video payload -- recording as failed.")
        return False
    log(f"    [{seg['key']}] done in {time.time()-t0:.1f}s")
    return True


def generate_with_retry(seg, first_frame_b64, dest_path):
    """Returns True (rendered), False (failed -- recorded, run continues).

    Tiered fallback on content refusal: chained (with first-frame still) ->
    fresh (no still). Quota/auth errors retry the same tier after a key swap.
    """
    tiers = []
    if first_frame_b64 is not None:
        tiers.append(first_frame_b64)
    tiers.append(None)
    tier_idx = 0
    while True:
        ff = tiers[tier_idx]
        tier_label = "chained" if ff else "fresh"
        vlog(f"    attempt tier {tier_idx+1}/{len(tiers)} ({tier_label})")
        try:
            return generate_segment(seg, ff, dest_path)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if is_content_refusal(exc):
                if tier_idx + 1 < len(tiers):
                    tier_idx += 1
                    print(f"    refused -- retrying [{seg['key']}] fresh (no first-frame still)...")
                    continue
                print(f"    content refused ({exc.__class__.__name__}) -- recording as failed.")
                print(f"    detail: {str(exc)[:200]}")
                return False
            if is_quota_error(exc):
                reason = f"quota/rate limit hit ({exc.__class__.__name__})"
            elif is_auth_error(exc):
                reason = f"API key rejected ({exc.__class__.__name__})"
            else:
                raise
            print(f"    {reason}: {str(exc)[:180]}")
            new_key = prompt_for_new_key(reason)
            if new_key == "skip":
                print(f"    [{seg['key']}] skipped by user choice.")
                return False
            if new_key:
                set_active_key(new_key, source="mid-run swap")
                print(f"    retrying [{seg['key']}] with the new key...")
                continue
            print("    no replacement key entered -- recording segment as failed.")
            return False

# ------------------------------------------------------------------ frames & stitch


def last_frame_of(mp4_path):
    """Extract the TRUE last frame of a segment as PNG bytes (cached).

    Decodes the tail of the file and keeps the final decoded frame -- exact,
    unlike `-sseof -0.5 -vframes 1`, which lands ~14 frames early and causes
    a visible jump at every chained transition.
    """
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_lastframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    tmpdir = tempfile.mkdtemp(prefix="tail_")
    try:
        pattern = os.path.join(tmpdir, "tail_%04d.png")
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-sseof", "-3", "-i", mp4_path,
                        "-frames:v", "400", pattern],
                       check=True, timeout=180, stdin=subprocess.DEVNULL)
        files = sorted(glob.glob(os.path.join(tmpdir, "tail_*.png")))
        if not files:
            print("    frame extract: no frames decoded; continuing without first frame")
            return None
        shutil.move(files[-1], out)
    except Exception as exc:
        print(f"    frame extract failed ({exc}); continuing without first frame")
        return None
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
    with open(out, "rb") as f:
        return f.read()


def first_frame_of(mp4_path):
    """Extract the FIRST frame of a segment as PNG bytes (cached) -- for gap-fill."""
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_firstframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    try:
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-i", mp4_path, "-vframes", "1", "-q:v", "2", out],
                       check=True, timeout=120, stdin=subprocess.DEVNULL)
    except Exception as exc:
        print(f"    first-frame extract failed ({exc}); no end-frame constraint")
        return None
    with open(out, "rb") as f:
        return f.read()


def probe_duration(ffmpeg, path):
    """Probe exact video stream duration; fall back to container duration."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        try:
            cmd = [
                ffprobe, "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ]
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
            if out and out != "N/A":
                return float(out)
        except Exception:
            pass
    p = subprocess.run([ffmpeg, "-i", path], capture_output=True, text=True,
                       timeout=60, stdin=subprocess.DEVNULL)
    m = re.search(r"Duration:\s*(\d+):(\d+):([\d.]+)", p.stderr)
    if not m:
        raise RuntimeError(f"could not probe duration of {path}")
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def stitch_xfade(ffmpeg, paths, out, preset="medium", timeout=1800,
                 normalize=True):
    """Join with short video+audio crossfades at every boundary."""
    f = XFADE_S
    durs = [probe_duration(ffmpeg, p) for p in paths]
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    fc = []
    # Normalize every input first: segments arrive with mixed timebases and
    # xfade refuses to link mismatched timebases -- one bad join kills the
    # whole chain.
    vlabels, alabels = [], []
    scale_dim = "640:360" if DRAFT_MODE else "1920:1080"
    for k in range(len(paths)):
        if normalize:
            # 1. Reset PTS to 0.
            # 2. Add tpad to clone the final frame if it runs short by a few milliseconds.
            fc.append(
                f"[{k}:v]scale={scale_dim},setsar=1,fps=24,setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5},format=yuv420p[v{k}n]"
            )
            # Trim audio to exact video stream duration to prevent audio/video length drift
            fc.append(
                f"[{k}:a]aresample=48000,aformat=channel_layouts=stereo,"
                f"asetpts=PTS-STARTPTS,atrim=0:{durs[k]:.3f}[a{k}n]"
            )
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")
        else:
            fc.append(
                f"[{k}:v]setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5}[v{k}n]"
            )
            fc.append(f"[{k}:a]asetpts=PTS-STARTPTS[a{k}n]")
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")

    vprev = vlabels[0]
    for k in range(1, len(paths)):
        off = sum(durs[:k]) - f * k
        vout = f"vx{k}"
        fc.append(f"[{vprev}][{vlabels[k]}]xfade=transition=fade:duration={f}:offset={off:.3f}[{vout}]")
        vprev = vout

    aprev = alabels[0]
    for k in range(1, len(paths)):
        aout = f"ax{k}"
        fc.append(f"[{aprev}][{alabels[k]}]acrossfade=d={f}[{aout}]")
        aprev = aout

    cmd += ["-filter_complex", ";".join(fc),
            "-map", f"[{vprev}]", "-map", f"[{aprev}]",
            "-c:v", "libx264", "-crf", "18", "-preset", preset,
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=timeout, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (crossfaded joins)")


def stitch(filenames, out="mirror_of_spheres_json.mp4"):
    """Join segments in order. Falls back to plain concat if xfade fails."""
    paths = [os.path.abspath(os.path.join(OUT_DIR, n)) for n in filenames]
    lst = os.path.join(OUT_DIR, "concat.txt")
    with open(lst, "w") as f:
        for p in paths:
            f.write(f"file '{p}'\n")
    ffmpeg = FFMPEG or shutil.which("ffmpeg")
    if ffmpeg is None:
        print("ffmpeg not found -- skipping auto-stitch. Run:")
        print(f"  ffmpeg -f concat -safe 0 -i {lst} -c copy {out}")
        return
    if XFADE_S > 0 and len(paths) > 1:
        try:
            # Two-level crossfade: one giant 84-input filtergraph exhausts
            # memory (84 simultaneous decoders + a 160+ filter chain), so
            # stitch small chunks first and then crossfade the chunks.
            CHUNK = 7
            if len(paths) > CHUNK:
                chunkdir = os.path.join(OUT_DIR, "chunks")
                os.makedirs(chunkdir, exist_ok=True)
                chunk_paths = []
                for ci in range(0, len(paths), CHUNK):
                    cpaths = paths[ci:ci + CHUNK]
                    if len(cpaths) > 1:
                        cfile = os.path.abspath(os.path.join(
                            chunkdir, f"chunk{ci // CHUNK:02d}.mp4"))
                        print(f"  stitching chunk {ci // CHUNK + 1} "
                              f"({len(cpaths)} segments)...")
                        stitch_xfade(ffmpeg, cpaths, cfile, normalize=True)
                        chunk_paths.append(cfile)
                    else:
                        chunk_paths.append(cpaths[0])
                print(f"  stitching {len(chunk_paths)} chunks into final...")
                stitch_xfade(ffmpeg, chunk_paths, out, timeout=3600,
                             normalize=True)
                shutil.rmtree(chunkdir, ignore_errors=True)
            else:
                stitch_xfade(ffmpeg, paths, out, normalize=True)
            return
        except Exception as exc:
            import traceback
            print(f"xfade stitch failed ({exc.__class__.__name__}: {exc}); falling back to concat filter")
            traceback.print_exc()
    # Fallback: concat filter with re-encode. Immune to the timestamp/DTS
    # quirks of stream-copy across mixed-mux segments.
    segs = "".join(f"[{k}:v][{k}:a]" for k in range(len(paths)))
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    cmd += ["-filter_complex", f"{segs}concat=n={len(paths)}:v=1:a=1[vcat][acat]",
            "-map", "[vcat]", "-map", "[acat]",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=3600, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (concat fallback)")


# ------------------------------------------------------------------ run

def cold_open_path():
    return os.path.join(OUT_DIR, "cold_open.mp4")


def run_cold_open():
    """Pre-step: generate the black-void loop-point frame, return its last
    frame as PNG bytes (the seed for segment 0). Reuses the cached file."""
    path = cold_open_path()
    if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
        print("[cold_open] already rendered -- resuming, skipping")
    else:
        print("[cold_open] generating the loop-point frame (black void, him walking)...")
        ok = generate_with_retry({"key": "cold_open", "segment": cold_open_segment()},
                                 None, path)
        if not ok:
            print("[cold_open] failed -- segment 0 will generate fresh.")
            return None
    raw = last_frame_of(path)
    if raw:
        print("[cold_open] last frame extracted -- seeding segment 0.")
    return raw


def dry_run():
    """Print every segment's full JSON request. No API calls, no key."""
    req0 = build_request(cold_open_segment(), {})
    print(f"# Mirror of Spheres JSON prompts -- {len(SEGMENTS)} segments + cold open (dry run)\n")
    print("## cold_open\n")
    print(json.dumps(req0, indent=2) + "\n")
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            break
        req = build_request(seg["segment"], {"bridger_ref": "Image1",
                                             "first_frame": "Image2"})
        print(f"## {i:02d}_{seg['key']} ({len(json.dumps(req))} chars)\n")
        print(json.dumps(req, indent=2) + "\n")


def seg_path(i):
    return os.path.join(OUT_DIR, f"{i:02d}_{SEGMENTS[i]['key']}.mp4")


def main():
    if DRY_RUN:
        dry_run()
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    rlog_init({
        "schema": SCHEMA_VERSION,
        "model": MODEL_ID,
        "resolution": RESOLUTION,
        "draft_mode": DRAFT_MODE,
        "gate_seconds": GATE_SECONDS,
        "xfade_s": XFADE_S,
        "max_segments": MAX_SEGMENTS,
        "cold_open": COLD_OPEN,
        "bridger_ref": bool(BRIDGER_REF_B64),
    })
    print(f"ffmpeg: {FFMPEG if FFMPEG else 'NOT FOUND -- chaining disabled, all segments will generate fresh!'}")
    print(f"segments: {len(SEGMENTS)} (12 act-transitions + 72 gates) | {RESOLUTION} | "
          f"xfade: {XFADE_S}s {'(hard cuts)' if XFADE_S == 0 else ''} | "
          f"{'DRAFT' if DRAFT_MODE else 'PRODUCTION'}")
    print("prompting: all-JSON (meta state variables + per-segment variables) | "
          "chaining: last-frame stills only, no video references")

    # Cold open first: its last frame seeds segment 0.
    cold_seed = run_cold_open() if COLD_OPEN else None

    made, failed = [], []
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            print(f"(stopping at MAX_SEGMENTS={MAX_SEGMENTS})")
            break
        path = seg_path(i)
        if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[{seg['key']}] already rendered -- resuming, skipping")
        else:
            first_frame = None
            if i == 0 and cold_seed:
                print("    seeding from cold-open last frame")
                first_frame = base64.b64encode(cold_seed).decode("ascii")
            elif i > 0 and os.path.exists(seg_path(i - 1)):
                print(f"    chaining from last frame of {os.path.basename(seg_path(i - 1))}")
                raw = last_frame_of(seg_path(i - 1))
                first_frame = base64.b64encode(raw).decode("ascii") if raw else None
                if first_frame is None:
                    print(f"    WARNING: last-frame extraction failed -- [{seg['key']}] will generate FRESH (expect a jump cut)")
            elif i > 0:
                print("    (no previous segment file; starting fresh)")
            # Gap-fill: if the NEXT segment already exists, constrain this
            # one's tail to the next segment's first frame.
            if i + 1 < len(SEGMENTS) and os.path.exists(seg_path(i + 1)):
                print(f"    (gap-fill tail note: {os.path.basename(seg_path(i + 1))} already exists)")
            t0 = time.time()
            try:
                ok = generate_with_retry(seg, first_frame, path)
            except KeyboardInterrupt:
                print("\nInterrupted. Re-run anytime -- finished segments are kept.")
                break
            rlog({"event": "segment_done", "key": seg["key"], "ok": ok,
                  "elapsed_s": round(time.time() - t0, 1),
                  "chained": first_frame is not None})
            if not ok:
                failed.append(seg["key"])
                continue
        if os.path.exists(path):
            made.append(os.path.basename(path))
    if failed:
        print(f"\nFailed segments: {failed}")
        print("Tweak / check quota, then re-run -- finished segments are skipped.")
    if made:
        stitch(made)


if __name__ == "__main__":
    main()

Here is your complete script with the FFmpeg fixes integrated directly into `probe_duration`, `stitch_xfade`, and `stitch`:

### Summary of Changes Integrated

1. **Accurate Stream Duration (`probe_duration`)**: Automatically queries `ffprobe` for the exact video stream duration (`stream=duration`) to avoid audio/container padding inflation, falling back gracefully to FFmpeg banner parsing if `ffprobe` is absent.
2. **End-of-Stream Frame Padding (`tpad`)**: Appends `tpad=stop_mode=clone:stop_duration={f + 0.5}` to every video input so that rounding or 24 FPS quantization differences clone the final frame instead of truncating early.
3. **Audio Duration Trimming (`atrim`)**: Trims audio streams to exact video duration (`atrim=0:{durs[k]:.3f}`) and resets timestamps (`asetpts=PTS-STARTPTS`) to prevent audio and video offset drift.
4. **Timestamp Zeroing (`setpts=PTS-STARTPTS`)**: Cleans input start times so `xfade` offsets align identically with presentation timestamps.
5. **Enforced Normalization on Chunk Stitch (`normalize=True`)**: Preserves stream normalization during the final chunk stitch pass so chunk timebases, color formats, and sample rates stay locked.

---

In [ ]:
# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
The Mirror of Spheres -- 72-Gate Decan Traversal (Gemini Omni, all-JSON prompts).

84 chained segments: each of the 12 acts opens with a 10s act-transition --
the zodiac glyph materializing as a vast physical sigil while the elemental
shift occurs -- followed by its 6 gates (~12:00+ total at 10s gates).

WHAT'S DIFFERENT IN THIS BUILD
------------------------------
- ALL-JSON PROMPTS. Every request sent to Omni is one JSON document: a `meta`
  block (standing rules + state variables, substituted from the environment at
  run time) and a `segment` block (every variable for that gate: act, element,
  wheel tone, angel {name, choir, form}, demon {name, form}, environment,
  threshold, glyph, beats). No markdown, no mixed prose -- the model gets
  structure, the way it likes it.
- FULL AUDIO. Diegetic audio is rendered: the act's audio bed, the wheel-tone
  handoff (every segment opens continuing the previous tail tone and closes
  settled into its own), and the ONLY permitted spoken words -- the World
  Bridger speaking the two guardian names at the naming beat.
- STILL-FRAME CHAINING ONLY. Each segment takes a still of the TRUE last frame
  of the previous segment and patches it in as the first frame. No video
  references (no prev-segment attach, no hero clip) -- stills only.
- COLD OPEN. An optional pre-step generates the loop-point frame first: pure
  black void with him walking ("a black screen with him on it"). Its last
  frame seeds segment 0, so the film opens deliberately and gate 72's tail
  closes back into it -- the loop plays all day with no visible seam.
- SIMPLIFIED BEHAVIOR. The prompts never mention his face at all. At each
  gate: he raises his staff and nods toward the angel as he speaks its name,
  then nods toward the demon as he speaks its name. That's it.
- DRAFT/PROD TOGGLE. DRAFT_MODE=1 -> 360p into separate dirs at roughly a
  third of the cost. Unset -> full 1080p production.

    pip install -U "google-genai>=2.10.0" imageio-ffmpeg
    export GEMINI_API_KEY="your-key"      # from https://aistudio.google.com/
    python produce_video_mirror_json.py

Continuity: F_k,start = F_{k-1},end -- the previous segment's true last frame
(extracted with a bundled ffmpeg -- the exact final decoded frame, not an
estimate) is declared as the first-frame source of the next segment.

Gate roster: all 72 pairs match Paul's canonical blueprint verbatim.
Guardian iconography: 144 expanded historically-grounded descriptions
(ANGEL_FORMS / DEMON_FORMS, supplied by Paul 2026-10-08) -- every guardian
towering, elementally distinct, bearing its seal. Angel choirs derived from
gate number (1-8 Seraphim ... 65-72 Angels).

Quota handling: on rate-limit / quota errors (or a rejected key) the script
pauses and prompts for another Google AI Studio key (hidden input). Type
'skip' to skip the segment, 'quit' (or Ctrl+C) to stop.

NOTE: guardian names (Bael, Asmoday, Belial, Astaroth, ...) can trip the
content filter. The tiered fallback sheds the first-frame still on refusal
(chained -> fresh) and the run continues; re-run heals gaps via gap-fill.
"""

import base64
import getpass
import glob
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time

try:
    from google import genai
except ImportError:  # --dry-run works without the SDK installed
    genai = None

# ------------------------------------------------------------------ config

SCHEMA_VERSION = "mirror-json/1.0"

# Paul's finding (2026-10-08): gemini-omni-flash-preview (the legacy preview
# path) encodes the same-quality renders at ~2-4MB vs ~9MB for 1.1-flash --
# same per-second price, far lighter segments and stitches. Default is the
# preview path he actually renders on; override with OMNI_MODEL=....
MODEL_ID = os.environ.get("OMNI_MODEL", "gemini-omni-flash-preview")

API_KEY = os.environ.get("GEMINI_API_KEY")
# --dry-run prints every segment's full JSON prompt without touching the API.
DRY_RUN = "--dry-run" in sys.argv
if not API_KEY and not DRY_RUN:
    raise SystemExit("Set GEMINI_API_KEY first: export GEMINI_API_KEY='your-key'")

client = None
API_KEY_POOL = []


def set_active_key(api_key, source=""):
    global client
    if genai is None:
        raise SystemExit("The google-genai SDK is not installed -- run via "
                         "run_mirror_json.sh or: pip install \"google-genai>=2.10.0\"")
    api_key = api_key.strip()
    os.environ["GEMINI_API_KEY"] = api_key
    if api_key not in API_KEY_POOL:
        API_KEY_POOL.append(api_key)
    client = genai.Client(api_key=api_key)
    print(f"    active API key set ({source or 'startup'}).")


if not DRY_RUN:
    set_active_key(API_KEY, source="environment")

# Draft <-> production toggle: DRAFT_MODE=1 renders 360p into isolated dirs.
DRAFT_MODE = os.environ.get("DRAFT_MODE", "").strip().lower() in ("1", "true", "yes")
OUT_DIR = "segments_mirror_json_draft" if DRAFT_MODE else "segments_mirror_json"
FRAMES_DIR = "frames_mirror_json_draft" if DRAFT_MODE else "frames_mirror_json"
RESOLUTION = "360p" if DRAFT_MODE else "1080p"

# Gate render length in seconds.
try:
    GATE_SECONDS = int(float(os.environ.get("GATE_SECONDS", "10")))
except ValueError:
    raise SystemExit(f"GATE_SECONDS must be a number, got {os.environ.get('GATE_SECONDS')!r}")
if GATE_SECONDS < 8:
    raise SystemExit(f"GATE_SECONDS must be >= 8, got {GATE_SECONDS}")

# Join style: 0 = direct hard cuts (the chained renders land so closely the
# cuts read as seamless). XFADE_S=0.5 for short dissolves, 1 for long ones.
XFADE_S = float(os.environ.get("XFADE_S", "0"))

# Cap a run to the first N segments (e.g. MAX_SEGMENTS=4 = cold open check +
# first gates). Unset or 0 = the full run.
try:
    MAX_SEGMENTS = int(os.environ.get("MAX_SEGMENTS", "0") or 0)
except ValueError:
    raise SystemExit("MAX_SEGMENTS must be an integer")

# Cold-open pre-step: generate the black-void loop-point frame first and seed
# segment 0 from its last frame. COLD_OPEN=0 skips it (segment 0 goes fresh).
COLD_OPEN = os.environ.get("COLD_OPEN", "1").strip().lower() not in ("0", "false", "no")

try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # embedded / exec contexts without __file__
    _HERE = os.getcwd()

# Canonical behind-view still of the Bridger's look. Attached to every
# generation as an IMAGE (appearance only). This is a still, not a video --
# video references are out of this build. BRIDGER_REF="" disables it.
_BRIDGER_REF_PATH = os.environ.get("BRIDGER_REF", os.path.join(_HERE, "bridger_ref.jpg"))
BRIDGER_REF_B64 = None
if _BRIDGER_REF_PATH:
    try:
        with open(_BRIDGER_REF_PATH, "rb") as _rf:
            BRIDGER_REF_B64 = base64.b64encode(_rf.read()).decode("ascii")
        print(f"    bridger reference still: {_BRIDGER_REF_PATH}")
    except OSError:
        print(f"    bridger reference not found ({_BRIDGER_REF_PATH}) -- continuing without it")

RESUME_EXISTING = True
GENERATE_TIMEOUT_S = 1800
POLL_S = 15
VERBOSE = os.environ.get("VERBOSE", "1").strip() not in ("0", "false", "no")


def log(msg):
    """Timestamped log line (flush immediately so hangs are visible)."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def vlog(msg):
    """Verbose-only log line."""
    if VERBOSE:
        log(msg)


# ------------------------------------------------------------------ run log
# Developer-level run statistics: one JSON object per line, per run, under
# logs_mirror_json/. Captures what happened (timings, tiers, retries, sizes)
# -- never prompt text.
RUN_LOG_DIR = os.path.join(_HERE, "logs_mirror_json")
RUN_LOG_PATH = None


def rlog_init(config):
    """Open a new per-run JSONL log; writes the run_start event."""
    global RUN_LOG_PATH
    os.makedirs(RUN_LOG_DIR, exist_ok=True)
    RUN_LOG_PATH = os.path.join(RUN_LOG_DIR,
                                f"run_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
    rlog({"event": "run_start", "ts": time.time(), "config": config})
    print(f"    run log: {RUN_LOG_PATH}")


def rlog(event):
    """Append one event dict to the current run log (best-effort)."""
    if not RUN_LOG_PATH:
        return
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, default=str) + "\n")
    except OSError:
        pass


def find_ffmpeg():
    try:
        from imageio_ffmpeg import get_ffmpeg_exe
        return get_ffmpeg_exe()
    except ImportError:
        return shutil.which("ffmpeg")


FFMPEG = find_ffmpeg()
if not FFMPEG:
    print("WARNING: no ffmpeg found (pip install imageio-ffmpeg). "
          "Segments will generate standalone without last-frame chaining.")

# ------------------------------------------------------------------ data tables
# (Paul's canonical data -- act/gate roster, environments, tones, iconography.)

# act: (name, element, gates lo-hi, audio profile)
ACTS = {
    1:  ("ARIES -- The Fire of Ignition",        "Fire",  (1, 6),   "heavy pulsating 40Hz sub-bass rhythm"),
    2:  ("TAURUS -- The Earth of Materiality",  "Earth", (7, 12),  "deep sustained earth drone with low stone-grinding friction"),
    3:  ("GEMINI -- The Air of Duality",        "Air",   (13, 18), "high-frequency rapidly panning binaural whispers"),
    4:  ("CANCER -- The Water of the Abyss",    "Water", (19, 24), "heavy filtered aqua reverb with sub-aquatic hydrostatic swells"),
    5:  ("LEO -- The Fire of Sovereignty",      "Fire",  (25, 30), "majestic brass swells underscored by visceral lion-roar sub-bass"),
    6:  ("VIRGO -- The Earth of Perfection",    "Earth", (31, 36), "ultra-clean exact sine tones oscillating in pure mathematical intervals"),
    7:  ("LIBRA -- The Air of Equilibrium",     "Air",   (37, 42), "wide-spectrum Perfect Fifth synth harmonics at 1.5 ratio"),
    8:  ("SCORPIO -- The Water of Transformation", "Water", (43, 48), "visceral slow 40Hz sub-bass heartbeat pulse"),
    9:  ("SAGITTARIUS -- The Fire of the Seeker", "Fire", (49, 54), "fast-tempo percussive drive with galloping rhythmic transients"),
    10: ("CAPRICORN -- The Earth of Architecture", "Earth", (55, 60), "massive acoustic stone cavern resonance and subterranean sub-drones"),
    11: ("AQUARIUS -- The Air of the Cosmos",   "Air",   (61, 66), "glitching bitcrushed modular synths and digital circuit feedback"),
    12: ("PISCES -- The Water of Dissolution",  "Water", (67, 72), "lush wordless 432Hz synth pad wash slowly pitching down into 40Hz sub-bass"),
}

# (gate, angel, demon, environment beat)
GATES = [
    # ACT 1 -- ARIES
    (1,  "Vehuiah",   "Bael",      "a suspended perfect circle of floating highly polished obsidian and volcanic glass shards erupting with cymatic fire fractals"),
    (2,  "Jeliel",    "Agares",    "twin ignition arcs spiral off the obsidian ring as the second fire node kindles"),
    (3,  "Sitael",    "Vassago",   "cymatic flames braid into triple-helix columns ringing the Bridger"),
    (4,  "Elemiah",   "Gamigin",   "volcanic glass shards ignite in sequence around the circle like a struck bell of fire"),
    (5,  "Mahasiah",  "Marbas",    "a roaring wall of flame parts down the middle as the guardians take their stations"),
    (6,  "Lelahel",   "Valefor",   "the full obsidian circle blazes white-gold as the sixth node locks the Aries fire circuit"),
    # ACT 2 -- TAURUS
    (7,  "Achaiah",   "Amon",      "colossal granite monoliths covered in glowing moss sigils forming an earthly circuit"),
    (8,  "Cahetel",   "Barbatos",  "glowing subterranean moss and deep stone fissures breathing faint green light"),
    (9,  "Haziel",    "Paimon",    "alabaster light channels cutting through bedrock echo vaults"),
    (10, "Aladiah",   "Buer",      "emerald ley-lines projected across slowly rotating basalt gears"),
    (11, "Lauviah",   "Gusion",    "sapphire starlight woven down into iron-ore foundations"),
    (12, "Hahaiah",   "Sitri",     "a marble sanctuary bound to smoldering red obsidian"),
    # ACT 3 -- GEMINI
    (13, "Iezalel",   "Beleth",    "dual mirrored prisms floating on aggressive high-altitude air currents"),
    (14, "Mebahel",   "Leraje",    "crystalline refractions sliced by whistling wind currents"),
    (15, "Hariel",    "Eligos",    "diamond-bright starlight colliding with sweeping iron dust vortices"),
    (16, "Hakamiah",  "Zepar",     "symmetrical light paths intersecting crimson atmospheric distortion"),
    (17, "Lauviah",   "Botis",     "soothing aura flares bound to razor-sharp shear winds"),
    (18, "Caliel",    "Bathin",    "geometric beam alignment meeting shadow-shrouded downdrafts"),
    # ACT 4 -- CANCER
    (19, "Leuviah",   "Sallos",    "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton forming moving seals"),
    (20, "Pahaliah",  "Purson",    "golden underwater arches veiled in drifting motes of light"),
    (21, "Nelchael",  "Marax",     "fluid geometry cutting through iron-infused silt clouds"),
    (22, "Yeiayel",   "Ipos",      "shimmering starlight piercing dark ocean undertows"),
    (23, "Melahel",   "Aim",       "pure aqua-energy distilling against undersea thermal vents"),
    (24, "Haheuiah",  "Naberius",  "a protective bioluminescent sphere anchored to ocean-floor vaults"),
    # ACT 5 -- LEO
    (25, "Nith-Haiah","Glasya-Labolas", "a blinding solar corona filled with spinning gold geometric projections"),
    (26, "Haaiah",    "Bune",      "gold architectural strictures anchoring into smoldering embers"),
    (27, "Yeratel",   "Ronove",    "corona flares locking into magnetic retention rings"),
    (28, "Seheiah",   "Berith",    "cooling golden aura balancing against molten royal flame"),
    (29, "Reiyel",    "Astaroth",  "pure solar sovereign light intersecting eclipsing copper shadow"),
    (30, "Omael",     "Forneus",   "vitalizing solar warmth grounding against cold solar vacuum"),
    # ACT 6 -- VIRGO
    (31, "Lecabel",   "Foras",     "hyper-precise cold sacred geometry and infinite anatomical spine mappings"),
    (32, "Vasariah",  "Asmoday",   "geometric symmetry carving along vertebrae entwined with chthonic horns"),
    (33, "Yehuiah",   "Gaap",      "starlight vertebrae seals binding to subterranean nerve roots"),
    (34, "Lehahiah",  "Furfur",    "cold spine plates aligning with bio-electric energy"),
    (35, "Chavakiah", "Marchosias","gold analytical geometry stabilizing inside skeletal frameworks"),
    (36, "Menadel",   "Stolas",    "a beam of mathematical order anchoring into starlight observatory roots"),
    # ACT 7 -- LIBRA
    (37, "Aniel",     "Phenex",    "unfolding scales of light and absolute mirrored symmetry on the 180-degree halfway axis"),
    (38, "Haamiah",   "Halphas",   "a symmetrical light sanctuary anchored into iron battlements"),
    (39, "Rehael",    "Malphas",   "silver threads of equilibrium meeting obsidian tower geometry"),
    (40, "Ieiazel",   "Raum",      "balanced air beams held inside absorbing shadow vortices"),
    (41, "Hahahel",   "Focalor",   "solar-winged scales balancing against heavy atmospheric pressure"),
    (42, "Mikael",    "Vepar",     "a central axis of pristine light mirroring deep sea-mist"),
    # ACT 8 -- SCORPIO
    (43, "Veuliah",   "Sabnock",   "massive wet obsidian vaults pulsing with a bioluminescent serpentine nervous system"),
    (44, "Yelahiah",  "Shax",      "bioluminescent aqua filaments pulsing through biological stone walls"),
    (45, "Sealiah",   "Vine",      "invigorating aqua currents anchoring into serpentine root systems"),
    (46, "Ariel",     "Bifrons",   "starlight streams balancing against tomb-stone shadows"),
    (47, "Asaliah",   "Vual",      "turquoise clarity illuminating the deep abyssal undertow"),
    (48, "Mihael",    "Haagenti",  "the 28th spoke rim at 240 degrees, wet obsidian giving way to void"),
    # ACT 9 -- SAGITTARIUS
    (49, "Vehuel",    "Crocell",   "an arrow of pure light shooting endlessly through an expanding cosmic Akashic library of flying parchment codices"),
    (50, "Daniel",    "Furcas",    "starlight illuminating ancient codices bound by iron quill marks"),
    (51, "Hahasiah",  "Balam",     "universal truths revealing in golden light through veils of mystery"),
    (52, "Imamiah",   "Alloces",   "forward trajectory of light accelerating through armor-plated momentum"),
    (53, "Nanael",    "Camio",     "celestial wisdom resonating across endless library halls"),
    (54, "Nithael",   "Murmur",    "a starlight throne bound to deep trumpet resonance"),
    # ACT 10 -- CAPRICORN
    (55, "Mebahiah",  "Orobas",    "an ancient mountain archive of deeply carved glowing seals"),
    (56, "Poiel",     "Gremory",   "gold light filling mountain vaults counterbalanced by velvet shadow chambers"),
    (57, "Nemamiah",  "Ose",       "crystalline spires raising toward mountain peaks"),
    (58, "Yeialel",   "Amy",       "iron beams aligning across stone archives bound by subterranean fire"),
    (59, "Harahel",   "Orias",     "cosmic knowledge vaults opening beside star-charting pillars"),
    (60, "Mitzrael",  "Vapula",    "architectural mountain integrity binding to mechanical craftsmanship"),
    # ACT 11 -- AQUARIUS
    (61, "Umabel",    "Zagan",     "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations"),
    (62, "Iah-Hel",   "Valac",     "high-speed data sanctuaries anchored into dragon-circuit foundations"),
    (63, "Anauel",    "Andras",    "collective thought-currents coordinating through static voltage spikes"),
    (64, "Mehiel",    "Flauros",   "protective light-scripts writing across sky firewalls"),
    (65, "Damabiah",  "Andrealphus","crystal-clear data streams transforming through prism matrices"),
    (66, "Manakel",   "Kimaris",   "a neural storm calming into steady logic above server vaults"),
    # ACT 12 -- PISCES
    (67, "Eyael",     "Amdusias",  "an infinite borderless ocean of consciousness dissolving into starlight mist"),
    (68, "Habuhiah",  "Belial",    "primordial waters of rebirth nurturing above seabed bedrock"),
    (69, "Rochel",    "Decarabia", "cosmic memories restoring in glowing tides and star-shadows"),
    (70, "Jabamiah",  "Seere",     "ocean energy transforming into spirit propelled by swift currents"),
    (71, "Haiaiel",   "Dantalion", "divine clarity arming the soul inside infinite mirrors of consciousness"),
    (72, "Mumiah",    "Andromalius","the final node where the borderless ocean ignites back into volcanic glass and cymatic fire fractals"),
]


def act_of(gate):
    for num, (name, el, lo_hi, audio) in ACTS.items():
        lo, hi = lo_hi
        if lo <= gate <= hi:
            return num, name, el, audio
    raise ValueError(gate)


def choir_of(gate):
    """Angel choir derived from gate number (gates 1-8 Seraphim ... 65-72 Angels)."""
    return {
        1: "Seraphim", 2: "Cherubim", 3: "Thrones", 4: "Dominations",
        5: "Powers", 6: "Virtues", 7: "Principalities", 8: "Archangels",
        9: "Angels",
    }[(gate - 1) // 8 + 1]

# ------------------------------------------------------------------ guardian iconography
# Historically-grounded forms: demons per Ars Goetia, angels per their 9 choirs.
# Every entity is towering, visually distinct, elemental, and dynamically raises its talisman.

ANGEL_FORMS = {
    # Choir 1: Seraphim (Fiery Divine Will & Primordial Light)
    "Vehuiah": "a towering, six-winged Seraph of blinding volcanic flame, igniting the primal void and thrusting high a blazing talisman of divine initiation",
    "Jeliel": "a towering, six-winged Seraph woven from soaring thermal firestorms, harmonizing celestial tides and raising aloft a radiant talisman of eternal union",
    "Sitael": "a towering, six-winged Seraph of piercing photon brilliance, laying cosmic foundation stones and brandishing high a burning talisman of architectural construction",
    "Elemiah": "a towering, six-winged Seraph expanding a golden corona of solar wrath, commanding the deep abyss and hoisting a blazing talisman of divine power",
    "Mahasiah": "a towering, six-winged Seraph radiating forge-hot alchemical heat, purifying cosmic discord and thrusting upward a fiery talisman of spiritual rectification",
    "Lelahel": "a towering, six-winged Seraph of blinding starlight and restorative solar flame, casting away darkness and raising high a luminous talisman of miraculous healing",
    "Achaiah": "a towering, six-winged Seraph of subterranean earth-fire, rooted in crystalline bedrock and brandishing aloft a heavy granite talisman of unyielding patience",
    "Cahetel": "a towering, six-winged Seraph wreathed in fertile golden auroras, commanding flourishing cosmic growth and hoisting high a jade-carved talisman of celestial harvest",

    # Choir 2: Cherubim (Wisdom, Four-Faced Tetramorphs & Whirling Wheels)
    "Haziel": "a towering, four-faced Cherub veined with molten gold and lapis lazuli, spinning crystalline wheels and raising aloft an earthen talisman of divine mercy",
    "Aladiah": "a towering, four-faced Cherub carved from adamantine granite, grinding crystalline wheels of light and thrusting high a diamond-bright talisman of restorative grace",
    "Lauviah": "a towering, four-faced Cherub crowned in sapphire starlight, turning heavy planetary wheels and brandishing aloft a meteoric iron talisman of prophetic revelation",
    "Hahaiah": "a towering, four-faced Cherub of translucent alabaster marble, shielding inner sanctuary fires and raising high a pearl-inlaid talisman of mystical refuge",
    "Iezalel": "a towering, four-faced Cherub of prismatic upper-atmosphere zephyrs, whirling gale-force crystalline wheels and hoisting aloft a radiant talisman of sacred fidelity",
    "Mebahel": "a towering, four-faced Cherub of blinding atmospheric lightning, severing untruth with spinning wheels and thrusting upward a crystalline talisman of absolute justice",
    "Hariel": "a towering, four-faced Cherub reflecting diamond-white astral dawns, chiming with infinite choir wheels and raising high an argent talisman of moral purification",
    "Hakamiah": "a towering, four-faced Cherub of roaring martial thunder, aligning celestial shields with turning wheels and brandishing aloft a bronze talisman of chivalric loyalty",

    # Choir 3: Thrones / Ophanim (Cosmic Foundations & Eye-Studded Beryl Wheels)
    "Lauviah (Throne)": "a towering Ophanim of revolving interlocking beryl wheels, burning with thousands of sleepless eyes and raising high an astral talisman of cosmic memory",
    "Caliel": "a towering Ophanim of whirling celestial chrysolite wheels, piercing atmospheric illusions with eye-studded rims and thrusting upward an electrum talisman of absolute truth",
    "Leuviah": "a towering Ophanim submerged in bioluminescent abyssal currents, turning eye-rimmed oceanic wheels and raising aloft a glowing coral talisman of expansive intellect",
    "Pahaliah": "a towering Ophanim burning with theological white flame beneath crushing deep-sea pressure, spinning sacred wheels and hoisting a golden talisman of fervent redemption",
    "Nelchael": "a towering Ophanim tracing astronomical geometry across the heavens, dripping liquid starlight from wheel-rims and brandishing aloft a brass talisman of higher learning",
    "Yeiayel": "a towering Ophanim navigating tempestuous celestial waters, guiding lost travelers with revolving eye-wheels and thrusting high a pearl-encrusted talisman of safe passage",
    "Melahel": "a towering Ophanim distilling pure restorative springwaters, blooming with medicinal flora along turning rims and raising high an emerald talisman of healing waters",
    "Haheuiah": "a towering Ophanim projecting an impenetrable oceanic dome, locking abyssal sanctuary wheels and brandishing aloft a heavy obsidian talisman of exile's protection",

    # Choir 4: Dominations (Sovereign Authority, Solar Crowns & Imperial Scepters)
    "Nith-Haiah": "a towering Domination draped in sovereign solar plasma, commanding secret occult forces with a glowing scepter and raising high a golden talisman of hidden wisdom",
    "Haaiah": "a towering Domination crowned in blinding coronal flares, orchestrating universal statecraft with an imperial staff and thrusting upward an electrum talisman of diplomacy",
    "Yeratel": "a towering Domination of victorious solar radiance, scattering encroaching shadows with sovereign authority and raising aloft a brilliant sunstone talisman of divine liberation",
    "Seheiah": "a towering Domination clad in golden dragon-scale armor, parrying ruinous cataclysms with an imperial scepter and hoisting high a mirror-polished talisman of longevity",
    "Reiyel": "a towering Domination blazing with sacred solar fury, shattering profane idols with royal decree and brandishing aloft a consecrated talisman of spiritual liberty",
    "Omael": "a towering Domination pulsing with generative solar warmth, multiplying the seeds of living creation and raising high a verdant chrysolite talisman of biological fertility",
    "Lecabel": "a towering Domination of laser-sharp geometric precision, dictating celestial vegetation cycles with an iron rod and thrusting upward an emerald talisman of exactitude",
    "Vasariah": "a towering Domination of stern imperial symmetry, weighing royal clemency with an ivory scepter and brandishing aloft a heavy marble talisman of righteous justice",

    # Choir 5: Powers (Martial Enforcers of Law, Chains & Restraining Aegis)
    "Yehuiah": "a towering Power clad in dark starlight plate, anchoring vertebrae of celestial law with glowing chains and raising high an iron talisman of cosmic subordination",
    "Lehahiah": "a towering Power encased in frost-gleaming adamantine armor, subduing rebellious spirits with celestial collars and thrusting upward a platinum talisman of divine obedience",
    "Chavakiah": "a towering Power armored in radiant solar gold, mending shattered ancestral pacts with bonds of light and hoisting aloft an engraved talisman of familial reconciliation",
    "Menadel": "a towering Power in heavy blackened-iron harness, liberating exhausted souls from oppressive toil and brandishing high an unyielding talisman of sanctified labor",
    "Aniel": "a towering Power balancing scales of kinetic celestial wind, shattering karmic chains of tyranny and raising aloft an argent talisman of broken cycles",
    "Haamiah": "a towering Power standing vigilant in ritual-scribed aegis armor, warding celestial sanctuaries with blazing runes and thrusting high a sacred talisman of spiritual protection",
    "Rehael": "a towering Power humming with harmonic frequencies of health, binding corrosive plagues with golden cords and hoisting aloft an amber talisman of filial healing",
    "Ieiazel": "a towering Power releasing concussive shockwaves of purified air, snapping dungeon iron with celestial chains and brandishing high a silver talisman of divine rescue",

    # Choir 6: Virtues (Miracle-Workers, Star Crowns & Elemental Catalysts)
    "Hahahel": "a towering Virtue crowned in blazing stellar constellations, channeling selfless apostolic sacrifice through an altar of light and raising high a consecrated talisman of holy priesthood",
    "Mikael": "a towering Virtue holding the central axis of sovereign harmony, steadying kingdoms amid shifting astral winds and thrusting upward a solar talisman of political order",
    "Veuliah": "a towering Virtue radiating turquoise subterranean luminescence, toppling tyrannical dynasties from the deep earth and raising aloft an orichalcum talisman of prosperous liberation",
    "Yelahiah": "a towering Virtue wreathed in bioluminescent martial wrath, guarding righteous warriors across dark currents and brandishing high a Damascus-steel talisman of karmic victory",
    "Sealiah": "a towering Virtue pulsing like a colossal cosmic heart, reigniting stagnant terrestrial currents with surge-force and hoisting aloft a verdant talisman of restored motion",
    "Ariel": "a towering Virtue parting abyssal ocean trenches, unearthing forgotten treasures of natural wisdom and thrusting high an iridescent mother-of-pearl talisman of secret revelation",
    "Asaliah": "a towering Virtue illuminating the midnight cosmos with contemplative brilliance, praising cosmic design and raising aloft a celestial prism talisman of divine cognition",
    "Mihael": "a towering Virtue radiating warm marine bioluminescence, intertwining divided souls into enduring unity and brandishing high a rose-gold talisman of conjugal harmony",

    # Choir 7: Principalities (Guardians of Nations, Celestial Architects & Scepters)
    "Vehuel": "a towering Principality crowned in galactic star-fire, releasing searing arrows of truth into the void and raising high a scintillating diamond talisman of divine exaltation",
    "Daniel": "a towering Principality crowned in shimmering rhetoric-flame, unfurling radiant parchment scrolls of cosmic law and thrusting upward an inscribed sapphire talisman of eloquent justice",
    "Hahasiah": "a towering Principality crowned in Hermetic alchemical fire, distilling universal panaceas from celestial vessels and raising aloft a smoldering cinnabar talisman of esoteric mysteries",
    "Imamiah": "a towering Principality crowned in incandescent plasma, trampling demonic pride beneath luminous boots and brandishing high a molten bronze talisman of spiritual expiation",
    "Nanael": "a towering Principality crowned in sacred philosophical radiance, contemplating planetary harmonics with celestial compasses and hoisting high an azure talisman of deep contemplation",
    "Nithael": "a towering Principality enthroned upon a pillar of starry marble, shielding emperors and rightful bloodlines while raising aloft an imperial gold talisman of legitimate succession",
    "Mebahiah": "a towering Principality crowned in carved mountain granite, anchoring righteous deeds into terrestrial foundations and thrusting high a lapis-inlaid talisman of lucid morality",
    "Poiel": "a towering Principality crowned in showering auric gold, opening infinite vaults of celestial abundance and brandishing high an opulent jewel-set talisman of boundless providence",

    # Choir 8: Archangels (Cosmic Heralds, Sacred Geometry & Divine Wisdom)
    "Nemamiah": "a towering Archangel forging crystalline spires of war, directing righteous commanders with a star-metal staff and raising high an adamantine talisman of righteous causes",
    "Yeialel": "a towering Archangel of deep resonant basalt, banishing sorrowful afflictions with an unbending rod and thrusting upward a cold meteoric iron talisman of endurance",
    "Harahel": "a towering Archangel unlocking limitless celestial archives, illuminating sacred texts with an ivory staff and hoisting high an inscribed golden talisman of intellectual wealth",
    "Mitzrael": "a towering Archangel mending the shattered geometry of the spirit, driving away internal madness and raising aloft a polished beryl talisman of psychological healing",
    "Umabel": "a towering Archangel weaving lines of astronomical affinity across the night sky, fostering enduring bonds and brandishing high an electrum talisman of sacred friendship",
    "Iah-Hel": "a towering Archangel enveloped in serene cloistered starlight, radiating transcendent wisdom through absolute silence and thrusting upward a crystal talisman of divine illumination",
    "Anauel": "a towering Archangel safeguarding terrestrial voyages and rightful commerce, shielding trade roads with a copper staff and raising high a coin-stamped talisman of material protection",
    "Mehiel": "a towering Archangel inking verses of celestial lightning into the clouds, inspiring grand orators and brandishing aloft a flaming quill-carved talisman of poetic inspiration",

    # Choir 9: Angels (Terrestrial Guardians, Living Waters & Rebirth)
    "Damabiah": "a towering Angel of gushing living headwaters, bestowing boundless oceanic wisdom upon searching souls and raising high a shimmering aquamarine talisman of the eternal spring",
    "Manakel": "a towering Angel diving through abyssal marine depths, interpreting prophetic dreams from dark waters and thrusting upward a polished pearl talisman of good and evil",
    "Eyael": "a towering Angel projecting incandescent mists of solar ascension, transmuting mundane sorrow into celestial joy and hoisting aloft an alchemical gold talisman of sublime transformation",
    "Habuhiah": "a towering Angel breathing fertile dew across barren orchards, healing blighted crops with radiant palms and raising high an emerald talisman of agricultural healing",
    "Rochel": "a towering Angel scouring the cosmic tides for lost treasures, returning stolen heritages to rightful heirs and brandishing aloft an argent talisman of complete restitution",
    "Jabamiah": "a towering Angel churning primordial oceans into celestial essence, resurrecting corrupted matter into perfection and thrusting upward a glowing chrysolite talisman of universal regeneration",
    "Haiaiel": "a towering Angel girded in blinding celestial armor, arming the pure of heart with flaming blades and raising high a luminous steel talisman of divine warfare and peace",
    "Mumiah": "a towering Angel of final oceanic dissolution, weaving twilight endings into triumphant rebirth and hoisting aloft an alpha-and-omega inscribed talisman of cosmic transformation",
}

DEMON_FORMS = {
    # 1 - 10
    "Bael": "a towering triple-headed king with heads of toad, cat, and crowned man on spider legs, rasping hoarsely and raising high a heavy iron talisman of invisibility",
    "Agares": "a towering, ancient nobleman riding an earth-shaking crocodile with a goshawk upon his fist, halting retreating armies and thrusting upward a smoldering copper talisman of language",
    "Vassago": "a towering infernal prince of radiant angelic countenance with burning sapphire eyes, declaring things past and future while raising aloft a crystalline talisman of prophecy",
    "Gamigin": "a towering spectral horse that shifts into an ashen phantom with a thunderous voice, summoning deceased souls and hoisting high a black-iron talisman of the dead",
    "Marbas": "a towering mechanical-maned lion with roaring furnace-fire eyes, answering truthfully of hidden secrets and brandishing high an iron-wrought talisman of mechanical craft and healing",
    "Valefor": "a towering prowling lion with the head of an ass-eared brigand, slipping through stealing shadows and raising high a tarnished silver talisman of thievery",
    "Amon": "a towering wolf with a serpent's tail vomiting dark obsidian flames, reconciling broken friendships and brandishing aloft a dense lead talisman of feuds and alliances",
    "Barbatos": "a towering horned hunter-lord accompanied by phantom kings and horn blasts, understanding animal tongues and raising high a moss-grown copper talisman of hidden treasure",
    "Paimon": "a towering, crowned monarch riding an imperial dromedary amid clashing cymbals and deafening trumpets, raising aloft an opulent gold talisman of spirit dominion",
    "Buer": "a towering wheel of five human legs radiating from a horned lion's head, spinning over bedrock and thrusting upward a bronze talisman of natural philosophy and healing",

    # 11 - 20
    "Gusion": "a towering cynocephalic ape-hound of weathered meteoric stone, discerning past, present, and future while brandishing aloft a heavy leaden talisman of restored honors",
    "Sitri": "a towering leopard with the wings of a massive griffin, smoldering with incandescent crimson passion and raising high a glowing copper talisman of inflamed desire",
    "Beleth": "a towering, terrible king riding a pale warhorse, preceded by blasting brass trumpets that shake the earth and thrusting upward a silver talisman of compelled love",
    "Leraje": "a towering emerald-clad archer drawing a massive recurve bow that whistles with shearing winds, raising aloft a green-patinated copper talisman of gangrenous wounds",
    "Eligos": "a towering ghostly knight clad in shining spectral plate, bearing lance, standard, and serpent while hoisting high a polished copper talisman of martial victory",
    "Zepar": "a towering crimson-armored soldier whose curved broadsword radiates scorched heat-distortion, inflaming minds and brandishing aloft a blood-rusted iron talisman of sterile lust",
    "Botis": "a towering horned viper shifting into a dreadful warrior with great tusks and drawn sword, thrusting high an electrum talisman of reconciled enemies",
    "Bathin": "a towering muscular giant with a serpent's tail riding a pale steed through planar portals, raising aloft a pale brass talisman of instantaneous transit",
    "Sallos": "a towering gallant soldier in a ducal coronet riding a leviathan crocodile with serene dignity, raising high a gleaming silver talisman of peaceful love",
    "Purson": "a towering lion-faced king clutching a hissing viper, riding a monstrous bear through blaring trumpets and hoisting high a radiant gold talisman of hidden treasures",

    # 21 - 30
    "Marax": "a towering bull with the bearded face of an ancient sage, expounding astronomy and herbal virtues while brandishing aloft a heavy iron talisman of liberal sciences",
    "Ipos": "a towering, uncanny angel with a roaring lion's head, goose's feet, and a hare's tail, thrusting upward a bright brass talisman of wit and valor",
    "Aim": "a towering three-headed nobleman riding a viper with a blazing firebrand, setting citadels ablaze and raising high a smoldering bronze talisman of consuming intellect",
    "Naberius": "a towering three-headed black crane shrieking in a raspy voice, walking through chthonic vaults and raising aloft a dark copper talisman of restored dignity",
    "Glasya-Labolas": "a towering winged bloodhound with broad griffin wings, trailing a crimson mist of bloodshed and brandishing high a bloodstained talisman of slaughter and prophecy",
    "Bune": "a towering three-headed dragon speaking in high resonant tones from sepulchral vaults, hoisting high a lustrous copper talisman of necromantic wealth",
    "Ronove": "a towering shadow-cloaked monster leaning upon a reaper's staff, harvesting weary souls while raising high an iron talisman of persuasive rhetoric and languages",
    "Berith": "a towering red-and-gold soldier upon a crimson warhorse, wearing a crown of molten flame and thrusting upward a blazing gold talisman of transmuting metals",
    "Astaroth": "a towering sorrowful fallen angel riding an infernal dragon with a coiled viper in hand, brandishing aloft a poisonous copper talisman of past falls",
    "Forneus": "a towering abyssal sea monster gliding through cold oceanic depths, granting eloquence across tongues and raising high a silver talisman of oceanic rhetoric",

    # 31 - 40
    "Foras": "a towering muscular sage carved from ancient gnarled oak, teaching logic, ethics, and herbology while hoisting high an earthen brass talisman of wisdom and longevity",
    "Asmoday": "a towering three-headed emperor vomiting furnace flames from an infernal dragon, brandishing aloft a molten gold talisman of invincible dominion and wrath",
    "Gaap": "a towering majestic emperor guiding four cardinal kings through the subterranean void, transporting mortals swiftly and raising high an inscribed leaden talisman of philosophy",
    "Furfur": "a towering colossal hart with a blazing lightning-wreathed tail, summoning roaring thunderheads and thrusting upward a crackling copper talisman of tempests and love",
    "Marchosias": "a towering armored wolf with griffin wings and a serpent's tail, vomiting columns of fire and brandishing high a gleaming silver talisman of martial fidelity",
    "Stolas": "a towering crowned raven with infinitely deep starlit eyes, perched upon celestial astrolabes and raising high an etched silver talisman of astronomy and herbs",
    "Phenex": "a towering phoenix wreathed in black-and-crimson flames, singing poetic verses with a haunting child's voice and hoisting aloft a smoldering copper talisman of melody",
    "Halphas": "a towering dark iron stock-dove shrieking with a hoarse voice, raising impenetrable fortress walls and thrusting upward a heavy brass talisman of siege warfare",
    "Malphas": "a towering soot-black raven shifting into a grim architect who shatters enemy citadels, raising aloft a dense leaden talisman of masonry and deception",
    "Raum": "a towering monstrous crow shifting into an armored plunderer, toppling dynasties and destroying cities while brandishing high a fractured copper talisman of ruin and theft",

    # 41 - 50
    "Focalor": "a towering winged warrior commanding hurricane winds and churning ocean whirlpools to sink warships, raising high a verdigris copper talisman of maritime wrath",
    "Vepar": "a towering ferocious mermaid rising from foaming breakers, guiding dread armadas with sea-fogs and hoisting aloft a tarnished silver talisman of nautical pestilence",
    "Sabnock": "a towering lion-headed soldier in heavy armor riding a pale horse, striking foes with gangrenous wounds and thrusting upward an iron talisman of fortress architecture",
    "Shax": "a towering dark stork-demon shrieking in a raspy voice, plundering mortal sight, hearing, and understanding while brandishing aloft a black talisman of sensory theft",
    "Vine": "a towering crowned lion riding a jet-black warhorse, brandishing a viper while summoning howling squalls and raising high an opulent gold talisman of exposed sorcery",
    "Bifrons": "a towering shape-shifting shade lighting spectral grave-lanterns over crumbling tombs, thrusting upward a weathered copper talisman of necromantic geometry and astronomy",
    "Vual": "a towering desert dromedary shifting into a fierce warrior speaking ancient Egyptian mysteries, raising high a hammered copper talisman of enduring affection",
    "Haagenti": "a towering winged golden bull shifting into a man, transmuting lead to gold and water to wine while hoisting high a molten gold talisman of supreme alchemy",
    "Crocell": "a towering luminous fallen angel murmuring with the sound of rushing waters, revealing steaming mineral baths and brandishing aloft a geometric brass talisman of secret sciences",
    "Furcas": "a towering white-bearded sage riding a skeletal pale steed, wielding a razor-sharp barbed spear and thrusting upward a heavy lead talisman of logic and pyromancy",

    # 51 - 60
    "Balam": "a towering three-headed king with flaming eyes, riding a raging bear with a goshawk on his fist and raising high an engraved gold talisman of past and future",
    "Alloces": "a towering knight with a fiery lion's face and scarlet eyes, riding an armored warhorse with crushing momentum and brandishing aloft a copper talisman of astronomy",
    "Camio": "a towering singing thrush stepping from burning embers to become a swordsman answering in ash, hoisting high a silver talisman of animal omens",
    "Murmur": "a towering crowned ducal warrior riding a ferocious griffin, preceded by ghostly horn blasts and raising aloft an ancient copper talisman of necromantic philosophy",
    "Orobas": "a towering magnificent horse shifting into a noble man, declaring truths of divinity without deceit and thrusting upward a sacred silver talisman of virtuous favor",
    "Gremory": "a towering all-seeing duchess wearing a jeweled coronet, riding an imperial camel through velvet shadows and brandishing aloft an opulent copper talisman of hidden treasures",
    "Ose": "a towering spotted leopard shifting into a crowned illusionist who plunges minds into madness, raising high an etched silver talisman of mental transfiguration",
    "Amy": "a towering blazing column of roaring fire resolving into an astrological magus, thrusting upward a glowing brass talisman of guarded spirit-treasures",
    "Orias": "a towering lion riding a serpent-tailed stallion, holding two hissing vipers toward starry heavens and raising high a silver talisman of astrological wisdom",
    "Vapula": "a towering lion with broad griffin wings, guiding artisans in intricate mechanics and treatises while brandishing aloft a polished copper talisman of sacred craftsmanship",

    # 61 - 72
    "Zagan": "a towering bull with griffin wings, transmuting water to wine, blood to oil, and base metals into coin while hoisting high a radiant gold talisman of transmutation",
    "Valac": "a towering winged angelic youth with blazing eyes, riding a colossal two-headed dragon through cavernous fissures and thrusting upward a serpentine talisman of hidden treasures",
    "Andras": "a towering angelic warrior with the head of a black raven, riding a ravenous midnight wolf with drawn saber and raising aloft a gleaming steel talisman of discordant slaughter",
    "Flauros": "a towering terrifying leopard leaping into a flame-eyed warrior who incinerates foes with divine fury, brandishing high a smoldering copper talisman of consuming fire",
    "Andrealphus": "a towering peacock shrieking with earsplitting pride, shifting into an architect who measures the cosmos and hoisting high an engraved silver talisman of mensuration",
    "Kimaris": "a towering heroic warrior in blackened harness riding a colossal black charger across spirit realms, raising high a bronze talisman of hidden grammar and rhetoric",
    "Amdusias": "a towering unicorn bending great forest oaks as invisible orchestral trumpets resound, thrusting upward a polished copper talisman of symphonic command",
    "Belial": "a towering pair of radiant fallen angels riding a chariot of roaring fire, speaking with deceptive honeyed grace and raising aloft an imperial gold talisman of sovereign favor",
    "Decarabia": "a towering five-pointed pentacle-star shifting into an enchanter surrounded by flocks of singing birds, brandishing high a lustrous silver talisman of herbal lore",
    "Seere": "a towering breathtaking youth riding a winged pegasus, moving across the world in the blink of an eye and hoisting high an electrum talisman of instantaneous abundance",
    "Dantalion": "a towering nobleman of infinite shifting faces, holding an open grimoire of human desires and thrusting upward a dark copper talisman of telepathic sway",
    "Andromalius": "a towering stern judge holding a colossal writhing serpent, tracking down thieves and punishing the wicked while raising high an iron talisman of absolute retribution",
}

# Occurrence index per gate for angels whose name repeats in the roster
# (Lauviah appears at gates 11 and 17); parallel to GATES.
_ANGEL_SEEN = {}
ANGEL_OCCURRENCE = []
for _gg, _aa, _dd, _ee in GATES:
    ANGEL_OCCURRENCE.append(_ANGEL_SEEN.get(_aa, 0))
    _ANGEL_SEEN[_aa] = _ANGEL_SEEN.get(_aa, 0) + 1
del _ANGEL_SEEN, _gg, _aa, _dd, _ee


def angel_form(angel, occurrence=0):
    """Expanded traditional form for an angel; occurrence disambiguates repeats."""
    key = angel
    if occurrence:
        alts = [k for k in ANGEL_FORMS if k != angel and k.startswith(angel)]
        if alts:
            key = alts[occurrence - 1]
    return ANGEL_FORMS[key]


def demon_form(demon):
    """Expanded traditional Goetic form for a demon."""
    return DEMON_FORMS[demon]


# Physical gate architecture per element -- the thing he actually walks through.
THRESHOLDS = {
    "Fire":  "a towering arch of black volcanic glass wreathed in slow-burning cymatic flame",
    "Earth": "a moss-covered stone trilithon humming with a low resonant tone",
    "Air":   "a lattice of prismatic light refracting the sky into slow ribbons",
    "Water": "an arch of living coral and breathing bioluminescent membrane",
}

# Wheel tones: each act resolves to a pure tone a perfect fifth (1.5x) above
# the last, folded into the 40-80Hz sub range -- the circle of fifths walking
# the 12 signs. Every segment OPENS continuing the previous segment's tone and
# CLOSES in a calm moment at its tail tone -- the tail tone IS the handoff.
WHEEL_TONES = {
    1: 40, 2: 60, 3: 45, 4: 68, 5: 51, 6: 76,
    7: 57, 8: 43, 9: 64, 10: 48, 11: 72, 12: 54,
}

GLYPHS = {
    1:  ("Aries",       "the Ram's glyph, two bold curved horns"),
    2:  ("Taurus",      "the Bull's glyph, a horned circle"),
    3:  ("Gemini",      "the Twins' glyph, twin pillars joined by arcs"),
    4:  ("Cancer",      "the Crab's glyph, twin spiraling whorls"),
    5:  ("Leo",         "the Lion's glyph, a single sweeping mane-curve"),
    6:  ("Virgo",       "the Maiden's glyph, an intricate looping sigil"),
    7:  ("Libra",       "the Scales' glyph, a balanced horizon over a rising curve"),
    8:  ("Scorpio",     "the Scorpion's glyph, a barbed tail striking upward"),
    9:  ("Sagittarius", "the Archer's glyph, a drawn bow loosing an arrow skyward"),
    10: ("Capricorn",   "the Sea-Goat's glyph, a horned crown diving into a fishtail curl"),
    11: ("Aquarius",    "the Water-Bearer's glyph, twin zigzag waves"),
    12: ("Pisces",      "the Fishes' glyph, two mirrored arcs bound by a single cord"),
}

ACT_ENVS = {
    1:  "a suspended perfect circle of floating polished obsidian and volcanic glass shards erupting with cymatic fire fractals",
    2:  "colossal monolithic stone temples covered in ancient glowing moss-covered sigils",
    3:  "dual mirrored prisms floating on aggressive high-altitude air currents",
    4:  "a sprawling submerged cathedral of coral and stone lit by bioluminescent plankton",
    5:  "a blinding solar corona filled with spinning gold geometric projections",
    6:  "hyper-precise cold sacred geometry and infinite anatomical spine mappings",
    7:  "unfolding scales of light and absolute mirrored symmetry",
    8:  "a massive obsidian gateway pulsing with a bioluminescent serpentine nervous system",
    9:  "an arrow of pure light shooting through an expanding cosmic Akashic library",
    10: "an ancient mountain archive of deeply carved glowing seals",
    11: "a sprawling hyper-advanced neural network sky of shimmering circuitboard constellations",
    12: "an infinite borderless ocean of consciousness dissolving into starlight mist",
}

DISSOLVE = {
    "Fire":  "dying cymatic flames and cooling volcanic glass",
    "Earth": "crumbling stone dust and fading moss-light",
    "Air":   "thinning high-altitude currents and shattering prisms",
    "Water": "receding bioluminescent waters and dissolving mist",
}

GLYPH_FORMATION = {
    "Fire":  "forged from living flame and volcanic glass",
    "Earth": "carved from moss-covered living stone",
    "Air":   "woven from prismatic light and high wind",
    "Water": "condensed from bioluminescent water and living coral",
}

IGNITION = {
    "Fire":  "cymatic flames erupt outward, igniting",
    "Earth": "stone dust settles and monoliths rise, revealing",
    "Air":   "the air thins into high currents, revealing",
    "Water": "bioluminescent waters flood inward, revealing",
}

# ------------------------------------------------------------------ JSON builders
# Every Omni request is ONE JSON document: `meta` (standing rules + state
# variables, substituted from the environment at run time) + `segment` (every
# variable for this gate). The JSON text is the entire text part of the
# request -- no markdown, no mixed prose.


def build_meta():
    """Standing rules + run-time state variables. Identical in every request."""
    return {
        "schema": SCHEMA_VERSION,
        "state": {
            "model": MODEL_ID,
            "duration_s": GATE_SECONDS,
            "resolution": RESOLUTION,
            "draft_mode": DRAFT_MODE,
            "stride_cycle_s": 1.2,
            "tempo_bpm": 100,
            "forward_speed_m_s": 1.2,
            "camera": "continuous unbroken over-the-shoulder rear tracking dolly",
            "lens": "35mm T1.3 anamorphic",
        },
        "rules": {
            "continuity": "every segment begins on the last frame of the previous segment, ends at the seam to the next gate in mid-motion",
            "appearance": "the World Bridger's appearance is locked to the bridger_ref image -- reproduce his avatar in the new last feame reference rendering only his body from it exactly, never restyle or reinterpret it; airborne ash and embers are environment, never markings on his robes other than glowing golden runes and basalt staff",
            "behavior": "at the center of the scene 5s mark he should continue his pace as he looks at the massive deities ahead. He raises his staff his voice booms out the names of the angel and demon who immediately bow in submission having been defeated by his name magic",
            "behavior_order": "angel first, then demon -- never swapped, no text overlays and both names should be said within 1s of eachother at the halfway point",
            "pacing": "stride is metronome-locked at 100 BPM, 1.2 m/s, identical in every shot -- he never walks faster, never slows down, never hurries a gate crossing",
            "motion": "the dolly never stops its forward drift; the shot flows continuously first frame to last;",
            "heading": "he walks dead straight ahead at all times -- spine aligned to his forward axis, never drifting, leaning, or cocking; the naming beat is the only head motion in the traversal",
            "segment_boundary": "every segment ends mid-stride, mid-dolly, exactly as the next one begins in the center of the gates doorway, so the hard cut between them is invisible and audio should seam perfectly",
            "camera_lock": "locked over-the-shoulder rear tracking; forward motion only; no cuts, no angle changes, no unscripted camera moves",
            "text": "zero text overlays of any kind -- everything visible must exist physically inside the scene never any text",
            "audio": "full diegetic audio is rendered: the act's audio bed, the wheel-tone handoff, and the two guardian names spoken by the Bridger -- the ONLY words ever spoken in the traversal",
        },
    }


def build_audio(act_audio, tone, spoken_names):
    """Full-audio block: act bed + wheel-tone handoff + the two spoken names."""
    return {
        "mode": "full_diegetic_audio",
        "open": "already continuing the previous segment's tail tone -- no click, no jump",
        "bed": act_audio,
        "spoken": [
            {"at": "naming beat", "speaker": "the World Bridger", "line": name}
            for name in spoken_names
        ],
        "speech_rule": "the two guardian names are the ONLY words spoken in the entire traversal",
        "close": f"all sound settles into a calm pure {tone}Hz wheel tone, held steady through the final seconds -- this is the handoff; the next segment picks up exactly from this tone. Render the tail as a seamless continuation, never an ending.",
        "forbidden": ["narration", "voiceover", "host commentary", "music", "text", "non-diegetic sound effects"],
    }


def build_naming_beat(angel, demon):
    """The simplified naming beat. No face is ever mentioned."""
    return {
        "at": "the threshold, mid-segment",
        "actions": [
            {"actor": "bridger", "do": "raise_staff"},
            {"actor": "bridger", "do": "nod", "toward": "left", "speak": angel},
            {"actor": "bridger", "do": "nod", "toward": "right", "speak": demon},
            {"actor": angel, "do": "bow"},
            {"actor": demon, "do": "bow"},
            {"actor": "guardians", "do": "submit and acknowledge"},
        ],
    }


def gate_segment(i, gate, angel, demon, env, element, next_env):
    """JSON variable block for one gate segment."""
    act_num, act_name, _el, act_audio = act_of(gate)
    tone = WHEEL_TONES[act_num]
    mid = GATE_SECONDS // 2
    seg = {
        "id": f"gate_{gate:02d}",
        "kind": "gate",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": act_name,
            "element": element,
            "wheel_tone_hz": tone,
        },
        "guardians": {
            "left": {
                "name": angel,
                "choir": choir_of(gate),
                "form": angel_form(angel, ANGEL_OCCURRENCE[i]),
            },
            "right": {
                "name": demon,
                "order": "Ars Goetia",
                "form": demon_form(demon),
            },
        },
        "environment": env,
        "threshold": THRESHOLDS[element],
        "next_environment": next_env,
        "beats": {
            "approach": f"the threshold looms ahead -- {THRESHOLDS[element]} -- with {angel} gathering at its left pillar and {demon} at its right pillar",
            "naming": build_naming_beat(angel, demon),
            "naming_at_s": f"around {mid - 1}-{mid + 1}s",
        },
        "audio": build_audio(act_audio, tone, [angel, demon]),
    }
    if gate == 48:
        seg["beats"]["crossing"] = (
            "at the 28th spoke rim the Bridger strikes his staff down on the stone -- "
            "a golden 108-degree geodesic Zen Shield flares to life around him, "
            "shattering the matrix wall. Fully solid and opaque, he strides on as the "
            "camera turns 90 degrees left off the zodiac track into the zero-gravity "
            "Ophiuchus Void (the single scripted camera move of the traversal), and "
            "all audio drops into absolute dead silence."
        )
        # The breach ends in dead silence, not a wheel tone -- the Act 9
        # transition opens with "the sound rises from absolute silence".
        seg["audio"]["close"] = (
            "absolute dead silence -- the breach ends in silence, not a wheel tone. "
            "Render the tail as a seamless continuation into silence, never an ending."
        )
    elif gate == 72:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the final gate, fully solid and opaque -- he "
            "does not fade, dissolve, or vanish -- as the borderless ocean detonates "
            "into white-gold ignition: volcanic glass and cymatic fire fractals blooming "
            "outward. The 360-degree loop closes seamlessly back into the Gate 1 threshold."
        )
    else:
        seg["beats"]["crossing"] = (
            "the Bridger strides through the gate plane on foot between the parted "
            "guardians -- fully solid and opaque, he does not fade, dissolve, or vanish "
            f"-- and the camera follows him through; beyond the far plane, {next_env} "
            "already resolving through the shimmer."
        )
    return seg


def transition_segment(act_num, is_first):
    """JSON variable block for one act-transition segment."""
    sign, glyph = GLYPHS[act_num]
    _n, _name, element, act_audio = act_of(act_num * 6 - 5)  # first gate of the act
    tone = WHEEL_TONES[act_num]
    seg = {
        "id": f"act_{act_num:02d}_transition",
        "kind": "act_transition",
        "shot": "single unbroken shot",
        "act": {
            "number": act_num,
            "name": f"{sign} transition",
            "element": element,
            "wheel_tone_hz": tone,
        },
        "glyph": {
            "sign": sign,
            "description": glyph,
            "formation": f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}, hanging in the space and slowly rotating, lit from within",
        },
        "elemental_shift": {
            "outgoing": DISSOLVE[act_of((act_num - 1) * 6)[2]] if not is_first else "dying cymatic flames and cooling volcanic glass",
            "incoming": f"{IGNITION[element]} {ACT_ENVS[act_num]}",
        },
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken -- no guardians stand at this threshold",
    }
    if is_first:
        seg["scene"] = (
            "opening cold: the camera is already locked over the shoulder of the World "
            "Bridger as he strides forward through dying cymatic flames and cooling "
            "volcanic glass, the old element dying away. Golden wireframe 72-gate rings "
            "and glowing cuneiform code matrices float in zero gravity around him. Ahead, "
            f"the zodiac glyph of {sign} -- {glyph} -- materializes. The camera passes "
            f"beneath the glyph and {IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["open"] = "the loop closes here -- open continuing the calm 54Hz wheel tone the final Pisces gate left behind"
        seg["audio"]["spoken"] = []
    else:
        prev_element = act_of((act_num - 1) * 6)[2]
        prev_audio = act_of((act_num - 1) * 6)[3]
        prev_tone = WHEEL_TONES[act_num - 1]
        seg["scene"] = (
            "starting exactly from the first frame -- same camera, same forward drift, no "
            f"cut -- the old world falls behind. The camera drifts through {DISSOLVE[prev_element]} "
            f"as the old element dies away. Ahead, the zodiac glyph of {sign} -- {glyph} -- "
            f"materializes as a vast physical sigil {GLYPH_FORMATION[element]}. The Bridger "
            f"walks on beneath it, staff in hand. As the camera passes beneath the glyph, "
            f"{IGNITION[element]} {ACT_ENVS[act_num]}."
        )
        seg["audio"] = build_audio(act_audio, tone, [])
        seg["audio"]["spoken"] = []
        if act_num == 9:
            # Act 9 follows gate 48's breach, which ends in dead silence, not a tone.
            seg["audio"]["open"] = "the sound rises from absolute silence"
        else:
            seg["audio"]["open"] = (
                f"gliding continuously from the fading {prev_tone}Hz wheel tone -- a smooth "
                f"pitch glide, no click, no jump -- as {prev_audio} dissolves into {act_audio}"
            )
    return seg


def cold_open_segment():
    """JSON variable block for the cold-open pre-step: black void, him walking.

    Generated FIRST; its last frame seeds segment 0. This is the loop point --
    gate 72's tail walks back into this frame.
    """
    return {
        "id": "cold_open",
        "kind": "cold_open",
        "shot": "single unbroken shot",
        "note": "the loop point and the film's first frame -- a deliberate frame, never an arbitrary fresh generation",
        "scene": (
            "pure infinite solid matte black background (#000000) -- absolute zero "
            "background geometry, no pillars, no fire, no ground textures, no horizon. "
            "Only the World Bridger exists in the frame: dead-center rear view, mid-stride, "
            "carved basalt staff in his RIGHT hand, walking dead straight ahead along the "
            "central axis at his metronomic 100 BPM pace. No guardians, no naming beat."
            "hood always up, black wool, crimson lining and glowing cuneiform glyphs and runes."
        ),
        "guardians": None,
        "naming_beat": None,
        "speech": "no words are spoken",
        "audio": {
            "mode": "full_diegetic_audio",
            "open": "continuing the calm 54Hz wheel tone the final Pisces gate leaves behind -- the loop closes here",
            "bed": "the near-silence of the void under a faint sub-bass",
            "spoken": [],
            "speech_rule": "no words are spoken in the cold open",
            "close": "steady and unresolved -- segment 0 continues it; the old fire world ignites out of the dark",
            "forbidden": ["narration", "voiceover", "host commentary", "music", "non-diegetic sound effects"],
        },
    }


def build_request(segment, image_names):
    """Assemble the full JSON request document for one generation."""
    sources = {}
    if "bridger_ref" in image_names:
        sources["bridger_ref"] = {
            "image": image_names["bridger_ref"],
            "role": "appearance reference for the World Bridger -- appearance ONLY; it does not define the environment, pose, lighting, or scene",
        }
    if "first_frame" in image_names:
        sources["first_frame"] = {
            "image": image_names["first_frame"],
            "role": "the last frame of the previous segment -- begin on this exact frame and continue its motion, audio and lighting, and scene seamlessly",
        }
    return {
        "request": "generate_video",
        "engine": "gemini_omni",
        "meta": build_meta(),
        "sources": sources,
        "segment": segment,
    }


SEGMENTS = []
_prev_act = None
for i, (g, a, d, env) in enumerate(GATES):
    act_num = act_of(g)[0]
    if act_num != _prev_act:
        SEGMENTS.append({
            "key": f"act{act_num:02d}_transition",
            "segment": transition_segment(act_num, is_first=(len(SEGMENTS) == 0)),
        })
        _prev_act = act_num
    SEGMENTS.append({
        "key": f"gate{g:02d}",
        "segment": gate_segment(i, g, a, d, env, act_of(g)[2],
                                GATES[i + 1][3] if i + 1 < len(GATES) else None),
    })

assert len(SEGMENTS) == 84

# ------------------------------------------------------------------ inputs & API
# Stills only: the bridger_ref still and the first-frame still. No video
# references in this build.


def to_bytes(data):
    if isinstance(data, str):
        return base64.b64decode(data)
    return bytes(data)


def build_inputs(segment, first_frame_b64):
    """Build the Omni input list: [bridger_ref still?, first_frame still?, JSON text].

    The text part is the ENTIRE prompt as one JSON document -- meta + sources
    + segment. Nothing is markdown, nothing is mixed prose.
    """
    contents = []
    image_names = {}
    n_img = 0

    def add_image(b64, mime="image/png"):
        nonlocal n_img
        n_img += 1
        contents.append({"type": "image", "data": b64, "mime_type": mime})
        return f"Image{n_img}"

    if BRIDGER_REF_B64:
        image_names["bridger_ref"] = add_image(BRIDGER_REF_B64, "image/jpeg")
    if first_frame_b64:
        image_names["first_frame"] = add_image(first_frame_b64)

    request = build_request(segment, image_names)
    text = json.dumps(request, indent=2)
    contents.append({"type": "text", "text": text})

    if len(contents) == 1:
        return text
    return contents


def get_output_video(interaction):
    """Prefer .output_video, fall back to manual step traversal."""
    out = getattr(interaction, "output_video", None)
    if out is not None and (getattr(out, "data", None) or getattr(out, "uri", None)):
        return out
    for step in reversed(getattr(interaction, "steps", []) or []):
        if getattr(step, "type", None) == "model_output" and getattr(step, "content", None):
            for part in reversed(step.content):
                if getattr(part, "type", None) == "video":
                    return part
    return None


def download_uri(uri, dest_path, timeout_s=1800, poll_s=10):
    """Poll the hosted file until ACTIVE, then download it."""
    file_id = uri.rstrip("/").split("/")[-1].split(":")[0].split("?")[0]
    name = f"files/{file_id}"
    deadline = time.time() + timeout_s
    t0 = time.time()
    vlog(f"    POLL {name} every {poll_s}s (timeout {timeout_s}s)")
    while time.time() < deadline:
        elapsed = time.time() - t0
        try:
            info = client.files.get(name=name)
            state = getattr(info.state, "name", str(info.state))
            if state == "ACTIVE":
                log(f"    poll +{elapsed:.0f}s: {state} -- downloading")
                break
            if state == "FAILED":
                raise RuntimeError("video generation failed (file state FAILED)")
            log(f"    poll +{elapsed:.0f}s: hosted file state: {state}")
        except RuntimeError:
            raise
        except Exception as exc:
            log(f"    poll +{elapsed:.0f}s: waiting on hosted file ({exc.__class__.__name__}: {str(exc)[:120]})")
        time.sleep(poll_s)
    else:
        raise TimeoutError(f"timed out after {timeout_s}s waiting for hosted video file")
    data = client.files.download(file=uri)
    with open(dest_path, "wb") as f:
        f.write(to_bytes(data))
    size = os.path.getsize(dest_path) / 1e6
    log(f"    saved {dest_path} ({size:.1f} MB)")


def _norm(exc):
    # Normalize underscores so RATE_LIMIT_EXCEEDED / quota_exceeded style
    # status enums can't slip past the classifiers.
    return re.sub(r"[_\s]+", "", f"{type(exc).__name__} {exc}".lower())


def is_quota_error(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "429", "resourceexhausted", "ratelimitexceeded", "quotaexceeded",
        "exceededyourquota", "toomanyrequests", "dailylimitexceeded",
    ))


def is_auth_error(exc):
    t = _norm(exc)
    return (not is_quota_error(exc)) and any(k in t for k in (
        "401", "invalidapikey", "apikeyinvalid", "apikeynotvalid",
        "apikeyisinvalid", "unauthorized", "permissiondenied",
    ))


def is_content_refusal(exc):
    t = _norm(exc)
    return any(k in t for k in (
        "prohibitedcontent", "prohibiteduse", "contentguidelines",
        "blockeddueto", "violatedgoogle",
        "likeness", "likenesses", "realperson", "realpeople",
        "contentblocked", "inputblocked",
    ))


def prompt_for_new_key(reason):
    print()
    print("=" * 64)
    print(f"  KEY ISSUE MID-RUN: {reason}")
    print("  Paste another Google AI Studio key to keep rendering,")
    print("  or type 'skip' to skip this segment, 'quit' to stop the run.")
    print("=" * 64)
    try:
        new_key = getpass.getpass("  New GEMINI_API_KEY (hidden input): ").strip()
    except (EOFError, KeyboardInterrupt):
        print()
        raise KeyboardInterrupt
    if not new_key:
        return None
    low = new_key.lower()
    if low in ("skip", "s"):
        return "skip"
    if low in ("quit", "q", "exit"):
        raise KeyboardInterrupt
    return new_key


def generate_segment(seg, first_frame_b64, dest_path):
    """Generate one Omni segment from its JSON request. Returns True/False."""
    mode = "chained" if first_frame_b64 else "fresh"
    t0 = time.time()
    log(f"[{seg['key']}] generating ({RESOLUTION}, {mode})...")
    inputs = build_inputs(seg["segment"], first_frame_b64)
    txt = inputs if isinstance(inputs, str) else inputs[-1]["text"]
    if isinstance(inputs, str):
        parts = [f"json:{len(inputs)} chars"]
    else:
        parts = []
        for p in inputs:
            if "text" in p:
                parts.append(f"json:{len(p['text'])} chars")
            elif p.get("type") == "image":
                parts.append(f"image:{len(p.get('data') or '')/1e6:.1f} MB b64 {p.get('mime_type', '?')}")
            else:
                parts.append(str(p.get("type")))
    vlog(f"    REQUEST -> model={MODEL_ID} resolution={RESOLUTION} delivery=uri")
    vlog(f"    REQUEST parts: {', '.join(parts)}")
    if VERBOSE:
        # In verbose mode show the JSON skeleton (keys only) -- full JSON is
        # in --dry-run output.
        req = json.loads(txt)
        vlog(f"    REQUEST json: meta.rules={len(req['meta']['rules'])} rules, "
             f"sources={list(req['sources'])} segment={req['segment']['id']}")
    log(f"    sending interactions.create ...")
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=inputs,
        response_format={"type": "video", "resolution": RESOLUTION, "delivery": "uri"},
    )
    log(f"    RESPONSE in {time.time()-t0:.1f}s | interaction id: {interaction.id}")
    video = get_output_video(interaction)
    if video is None:
        log("    done, but no video in response -- recording as failed.")
        return False
    if getattr(video, "uri", None):
        vlog(f"    RESPONSE video: uri={video.uri}")
        download_uri(video.uri, dest_path)
    elif getattr(video, "data", None):
        vlog(f"    RESPONSE video: inline data ({len(to_bytes(video.data))/1e6:.1f} MB)")
        with open(dest_path, "wb") as f:
            f.write(to_bytes(video.data))
        log(f"    saved {dest_path}")
    else:
        log("    done, but no video payload -- recording as failed.")
        return False
    log(f"    [{seg['key']}] done in {time.time()-t0:.1f}s")
    return True


def generate_with_retry(seg, first_frame_b64, dest_path):
    """Returns True (rendered), False (failed -- recorded, run continues).

    Tiered fallback on content refusal: chained (with first-frame still) ->
    fresh (no still). Quota/auth errors retry the same tier after a key swap.
    """
    tiers = []
    if first_frame_b64 is not None:
        tiers.append(first_frame_b64)
    tiers.append(None)
    tier_idx = 0
    while True:
        ff = tiers[tier_idx]
        tier_label = "chained" if ff else "fresh"
        vlog(f"    attempt tier {tier_idx+1}/{len(tiers)} ({tier_label})")
        try:
            return generate_segment(seg, ff, dest_path)
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            if is_content_refusal(exc):
                if tier_idx + 1 < len(tiers):
                    tier_idx += 1
                    print(f"    refused -- retrying [{seg['key']}] fresh (no first-frame still)...")
                    continue
                print(f"    content refused ({exc.__class__.__name__}) -- recording as failed.")
                print(f"    detail: {str(exc)[:200]}")
                return False
            if is_quota_error(exc):
                reason = f"quota/rate limit hit ({exc.__class__.__name__})"
            elif is_auth_error(exc):
                reason = f"API key rejected ({exc.__class__.__name__})"
            else:
                raise
            print(f"    {reason}: {str(exc)[:180]}")
            new_key = prompt_for_new_key(reason)
            if new_key == "skip":
                print(f"    [{seg['key']}] skipped by user choice.")
                return False
            if new_key:
                set_active_key(new_key, source="mid-run swap")
                print(f"    retrying [{seg['key']}] with the new key...")
                continue
            print("    no replacement key entered -- recording segment as failed.")
            return False

# ------------------------------------------------------------------ frames & stitch


def last_frame_of(mp4_path):
    """Extract the TRUE last frame of a segment as PNG bytes (cached).

    Decodes the tail of the file and keeps the final decoded frame -- exact,
    unlike `-sseof -0.5 -vframes 1`, which lands ~14 frames early and causes
    a visible jump at every chained transition.
    """
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_lastframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    tmpdir = tempfile.mkdtemp(prefix="tail_")
    try:
        pattern = os.path.join(tmpdir, "tail_%04d.png")
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-sseof", "-3", "-i", mp4_path,
                        "-frames:v", "400", pattern],
                       check=True, timeout=180, stdin=subprocess.DEVNULL)
        files = sorted(glob.glob(os.path.join(tmpdir, "tail_*.png")))
        if not files:
            print("    frame extract: no frames decoded; continuing without first frame")
            return None
        shutil.move(files[-1], out)
    except Exception as exc:
        print(f"    frame extract failed ({exc}); continuing without first frame")
        return None
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)
    with open(out, "rb") as f:
        return f.read()


def first_frame_of(mp4_path):
    """Extract the FIRST frame of a segment as PNG bytes (cached) -- for gap-fill."""
    if not FFMPEG:
        return None
    os.makedirs(FRAMES_DIR, exist_ok=True)
    key = os.path.splitext(os.path.basename(mp4_path))[0]
    out = os.path.join(FRAMES_DIR, key + "_firstframe.png")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        if os.path.getmtime(out) >= os.path.getmtime(mp4_path):
            with open(out, "rb") as f:
                return f.read()
    try:
        subprocess.run([FFMPEG, "-y", "-v", "error",
                        "-i", mp4_path, "-vframes", "1", "-q:v", "2", out],
                       check=True, timeout=120, stdin=subprocess.DEVNULL)
    except Exception as exc:
        print(f"    first-frame extract failed ({exc}); no end-frame constraint")
        return None
    with open(out, "rb") as f:
        return f.read()


def probe_duration(ffmpeg, path):
    """Probe exact video stream duration; fall back to container duration."""
    ffprobe = shutil.which("ffprobe")
    if ffprobe:
        try:
            cmd = [
                ffprobe, "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ]
            out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
            if out and out != "N/A":
                return float(out)
        except Exception:
            pass
    p = subprocess.run([ffmpeg, "-i", path], capture_output=True, text=True,
                       timeout=60, stdin=subprocess.DEVNULL)
    m = re.search(r"Duration:\s*(\d+):(\d+):([\d.]+)", p.stderr)
    if not m:
        raise RuntimeError(f"could not probe duration of {path}")
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def stitch_xfade(ffmpeg, paths, out, preset="medium", timeout=1800,
                 normalize=True):
    """Join with short video+audio crossfades at every boundary."""
    f = XFADE_S
    durs = [probe_duration(ffmpeg, p) for p in paths]
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    fc = []
    # Normalize every input first: segments arrive with mixed timebases and
    # xfade refuses to link mismatched timebases -- one bad join kills the
    # whole chain.
    vlabels, alabels = [], []
    scale_dim = "640:360" if DRAFT_MODE else "1920:1080"
    for k in range(len(paths)):
        if normalize:
            # 1. Reset PTS to 0.
            # 2. Add tpad to clone the final frame if it runs short by a few milliseconds.
            fc.append(
                f"[{k}:v]scale={scale_dim},setsar=1,fps=24,setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5},format=yuv420p[v{k}n]"
            )
            # Trim audio to exact video stream duration to prevent audio/video length drift
            fc.append(
                f"[{k}:a]aresample=48000,aformat=channel_layouts=stereo,"
                f"asetpts=PTS-STARTPTS,atrim=0:{durs[k]:.3f}[a{k}n]"
            )
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")
        else:
            fc.append(
                f"[{k}:v]setpts=PTS-STARTPTS,settb=1/24,"
                f"tpad=stop_mode=clone:stop_duration={f + 0.5}[v{k}n]"
            )
            fc.append(f"[{k}:a]asetpts=PTS-STARTPTS[a{k}n]")
            vlabels.append(f"v{k}n")
            alabels.append(f"a{k}n")

    vprev = vlabels[0]
    for k in range(1, len(paths)):
        off = sum(durs[:k]) - f * k
        vout = f"vx{k}"
        fc.append(f"[{vprev}][{vlabels[k]}]xfade=transition=fade:duration={f}:offset={off:.3f}[{vout}]")
        vprev = vout

    aprev = alabels[0]
    for k in range(1, len(paths)):
        aout = f"ax{k}"
        fc.append(f"[{aprev}][{alabels[k]}]acrossfade=d={f}[{aout}]")
        aprev = aout

    cmd += ["-filter_complex", ";".join(fc),
            "-map", f"[{vprev}]", "-map", f"[{aprev}]",
            "-c:v", "libx264", "-crf", "18", "-preset", preset,
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=timeout, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (crossfaded joins)")


def stitch(filenames, out="mirror_of_spheres_json.mp4"):
    """Join segments in order. Falls back to plain concat if xfade fails."""
    paths = [os.path.abspath(os.path.join(OUT_DIR, n)) for n in filenames]
    lst = os.path.join(OUT_DIR, "concat.txt")
    with open(lst, "w") as f:
        for p in paths:
            f.write(f"file '{p}'\n")
    ffmpeg = FFMPEG or shutil.which("ffmpeg")
    if ffmpeg is None:
        print("ffmpeg not found -- skipping auto-stitch. Run:")
        print(f"  ffmpeg -f concat -safe 0 -i {lst} -c copy {out}")
        return
    if XFADE_S > 0 and len(paths) > 1:
        try:
            # Two-level crossfade: one giant 84-input filtergraph exhausts
            # memory (84 simultaneous decoders + a 160+ filter chain), so
            # stitch small chunks first and then crossfade the chunks.
            CHUNK = 7
            if len(paths) > CHUNK:
                chunkdir = os.path.join(OUT_DIR, "chunks")
                os.makedirs(chunkdir, exist_ok=True)
                chunk_paths = []
                for ci in range(0, len(paths), CHUNK):
                    cpaths = paths[ci:ci + CHUNK]
                    if len(cpaths) > 1:
                        cfile = os.path.abspath(os.path.join(
                            chunkdir, f"chunk{ci // CHUNK:02d}.mp4"))
                        print(f"  stitching chunk {ci // CHUNK + 1} "
                              f"({len(cpaths)} segments)...")
                        stitch_xfade(ffmpeg, cpaths, cfile, normalize=True)
                        chunk_paths.append(cfile)
                    else:
                        chunk_paths.append(cpaths[0])
                print(f"  stitching {len(chunk_paths)} chunks into final...")
                stitch_xfade(ffmpeg, chunk_paths, out, timeout=3600,
                             normalize=True)
                shutil.rmtree(chunkdir, ignore_errors=True)
            else:
                stitch_xfade(ffmpeg, paths, out, normalize=True)
            return
        except Exception as exc:
            import traceback
            print(f"xfade stitch failed ({exc.__class__.__name__}: {exc}); falling back to concat filter")
            traceback.print_exc()
    # Fallback: concat filter with re-encode. Immune to the timestamp/DTS
    # quirks of stream-copy across mixed-mux segments.
    segs = "".join(f"[{k}:v][{k}:a]" for k in range(len(paths)))
    cmd = [ffmpeg, "-hide_banner", "-loglevel", "warning", "-y"]
    for p in paths:
        cmd += ["-i", p]
    cmd += ["-filter_complex", f"{segs}concat=n={len(paths)}:v=1:a=1[vcat][acat]",
            "-map", "[vcat]", "-map", "[acat]",
            "-c:v", "libx264", "-crf", "18", "-preset", "fast",
            "-c:a", "aac", "-b:a", "160k", out]
    subprocess.run(cmd, check=True, timeout=3600, stdin=subprocess.DEVNULL)
    print(f"Done -> {out} (concat fallback)")


# ------------------------------------------------------------------ run

def cold_open_path():
    return os.path.join(OUT_DIR, "cold_open.mp4")


def run_cold_open():
    """Pre-step: generate the black-void loop-point frame, return its last
    frame as PNG bytes (the seed for segment 0). Reuses the cached file."""
    path = cold_open_path()
    if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
        print("[cold_open] already rendered -- resuming, skipping")
    else:
        print("[cold_open] generating the loop-point frame (black void, him walking)...")
        ok = generate_with_retry({"key": "cold_open", "segment": cold_open_segment()},
                                 None, path)
        if not ok:
            print("[cold_open] failed -- segment 0 will generate fresh.")
            return None
    raw = last_frame_of(path)
    if raw:
        print("[cold_open] last frame extracted -- seeding segment 0.")
    return raw


def dry_run():
    """Print every segment's full JSON request. No API calls, no key."""
    req0 = build_request(cold_open_segment(), {})
    print(f"# Mirror of Spheres JSON prompts -- {len(SEGMENTS)} segments + cold open (dry run)\n")
    print("## cold_open\n")
    print(json.dumps(req0, indent=2) + "\n")
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            break
        req = build_request(seg["segment"], {"bridger_ref": "Image1",
                                             "first_frame": "Image2"})
        print(f"## {i:02d}_{seg['key']} ({len(json.dumps(req))} chars)\n")
        print(json.dumps(req, indent=2) + "\n")


def seg_path(i):
    return os.path.join(OUT_DIR, f"{i:02d}_{SEGMENTS[i]['key']}.mp4")


def main():
    if DRY_RUN:
        dry_run()
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(FRAMES_DIR, exist_ok=True)
    rlog_init({
        "schema": SCHEMA_VERSION,
        "model": MODEL_ID,
        "resolution": RESOLUTION,
        "draft_mode": DRAFT_MODE,
        "gate_seconds": GATE_SECONDS,
        "xfade_s": XFADE_S,
        "max_segments": MAX_SEGMENTS,
        "cold_open": COLD_OPEN,
        "bridger_ref": bool(BRIDGER_REF_B64),
    })
    print(f"ffmpeg: {FFMPEG if FFMPEG else 'NOT FOUND -- chaining disabled, all segments will generate fresh!'}")
    print(f"segments: {len(SEGMENTS)} (12 act-transitions + 72 gates) | {RESOLUTION} | "
          f"xfade: {XFADE_S}s {'(hard cuts)' if XFADE_S == 0 else ''} | "
          f"{'DRAFT' if DRAFT_MODE else 'PRODUCTION'}")
    print("prompting: all-JSON (meta state variables + per-segment variables) | "
          "chaining: last-frame stills only, no video references")

    # Cold open first: its last frame seeds segment 0.
    cold_seed = run_cold_open() if COLD_OPEN else None

    made, failed = [], []
    for i, seg in enumerate(SEGMENTS):
        if MAX_SEGMENTS and i >= MAX_SEGMENTS:
            print(f"(stopping at MAX_SEGMENTS={MAX_SEGMENTS})")
            break
        path = seg_path(i)
        if RESUME_EXISTING and os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[{seg['key']}] already rendered -- resuming, skipping")
        else:
            first_frame = None
            if i == 0 and cold_seed:
                print("    seeding from cold-open last frame")
                first_frame = base64.b64encode(cold_seed).decode("ascii")
            elif i > 0 and os.path.exists(seg_path(i - 1)):
                print(f"    chaining from last frame of {os.path.basename(seg_path(i - 1))}")
                raw = last_frame_of(seg_path(i - 1))
                first_frame = base64.b64encode(raw).decode("ascii") if raw else None
                if first_frame is None:
                    print(f"    WARNING: last-frame extraction failed -- [{seg['key']}] will generate FRESH (expect a jump cut)")
            elif i > 0:
                print("    (no previous segment file; starting fresh)")
            # Gap-fill: if the NEXT segment already exists, constrain this
            # one's tail to the next segment's first frame.
            if i + 1 < len(SEGMENTS) and os.path.exists(seg_path(i + 1)):
                print(f"    (gap-fill tail note: {os.path.basename(seg_path(i + 1))} already exists)")
            t0 = time.time()
            try:
                ok = generate_with_retry(seg, first_frame, path)
            except KeyboardInterrupt:
                print("\nInterrupted. Re-run anytime -- finished segments are kept.")
                break
            rlog({"event": "segment_done", "key": seg["key"], "ok": ok,
                  "elapsed_s": round(time.time() - t0, 1),
                  "chained": first_frame is not None})
            if not ok:
                failed.append(seg["key"])
                continue
        if os.path.exists(path):
            made.append(os.path.basename(path))
    if failed:
        print(f"\nFailed segments: {failed}")
        print("Tweak / check quota, then re-run -- finished segments are skipped.")
    if made:
        stitch(made)


if __name__ == "__main__":
    main()